In [ ]:
%matplotlib widget

## Notebook Overview

This notebook investigates the dynamics of the **double spherical pendulum (DSP)** through analytical mechanics, numerical simulation, and computational modeling. The system consists of two coupled pendulums free to move in three dimensions, providing a setting in which to explore the relationship between relatively simple mechanical constraints and complex nonlinear motion.

The investigation is organized into five sections:

1. **Physical Problem:** Define the DSP, establish its generalized coordinates and constraints, derive its Lagrangian equations of motion, and identify the assumptions and symmetries of the system.

2. **Visualizations:** Numerically integrate the equations of motion, visualize the resulting trajectories, and investigate special solutions, including aligned and anti-aligned conical motion and their small-oscillation dynamics.

3. **Symmetries and Conserved Quantities:** Examine energy and angular-momentum conservation, investigate sensitivity to initial conditions, and use phase-space methods, including Poincaré sections, to explore nonlinear behavior.

4. **Advanced Numerical Approaches:** Compare numerical integration techniques and investigate machine-learning-based dynamics models, particularly constrained Lagrangian neural networks (CLNNs).

5. **Hamiltonian vs. Lagrangian:** Reformulate the DSP in canonical coordinates, compare numerical predictions from the two formulations, and illustrate Liouville's theorem using a reduced normal-mode system.

Together, these sections demonstrate how analytical and computational approaches complement one another in the study of nonlinear mechanical systems.

## List of Abbreviations

| Abbreviation | Definition |
|---|---|
| CLNN | Constrained Lagrangian Neural Network |
| DSP | Double Spherical Pendulum |
| E-L | Euler–Lagrange |
| EOMs | Equations of Motion |
| ICs | Initial Conditions |
| IVP | Initial Value Problem |
| LNN | Lagrangian Neural Network |
| ODE | Ordinary Differential Equation |

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# 1. Physical Problem

## 1.1 Investigating the Double Spherical Pendulum

**Problem 7.40** in Classical Mechanics (John R. Taylor)

The "spherical pendulum" is just a simple pendulum that is free to move in any sideways direction. (By contrast, a "simple pendulum"--unqualified--is confined to a single vertical plane.) The bob of a spherical penulum moves on a sphere, centered on the point of support with radius $r = R$, the length of the pendulum. A convenient choice of coordinates is the spherical polars, $r$, $\theta$, $\phi$, with the origin at the point of support and the polar axis pointing straight down. The two variables $\theta$ and $\phi$ make a good choice of generalized coordinates. **(a)** Find the Lagrangian and the two Lagrange equations. **(b)** Explain what the $\phi$ equation tells us about the $z$ component of angular momentum $\ell_z$ **(c)** For the special case that $\phi = \text{const}$, describe what the $\theta$ equation tells us. **(d)** Use the $\phi$ equation to replace the $\dot{\phi}$ by $\ell_z$ in the $\theta$ equation and discuss the existence of an angle $\theta_o$ at which $\theta$ can remain constant. Why is this motion called a conical pendulum? **(e)** Show that if $\theta = \theta_0 + \epsilon$, with $\epsilon$ small, then $\theta$ oscillates about $\theta_o$ in harmonic motion. Describe the motion of the pendulum's bob.

Taylor's problem considers a single spherical pendulum: one bob of mass $m$ on a rigid, massless rod of length $R$, swinging freely in any direction about a fixed pivot, described by generalized coordinates $(\theta, \phi)$ measured from the downward vertical.

Here, that setup is extended by hanging a **second** spherical pendulum from the first bob: mass $m_2$ on a rod of length $R_2$, attached not to a fixed pivot but to the moving position of the first bob $(m_1, R_1)$. Each rod is still free to swing in any azimuthal direction (nothing is confined to a plane) so the system now has four generalized coordinates, $(\theta_1, \phi_1, \theta_2, \phi_2)$, each pair measured from the vertical exactly as in the original problem, just with the second pendulum's "pivot" co-moving with the first bob instead of being fixed in space. Equivalently, the resulting system admits two complementary interpretations: it is the fully three-dimensional generalization of the planar double pendulum, and at the same time the two-body generalization of the single spherical pendulum treated in Taylor's problem.

The reason this extension is interesting: Taylor's single spherical pendulum is *integrable*. According to the Liouville-Arnold theorem which states that in a Hamiltonian system with $n$ degrees of freedom, if there are also $n$ independent, Poisson commuting first integrals (conserved quantities where pairwise Poisson brackets vanish, $\{H_i, H_j\} = 0$), then the equations of motion (EOM) for the system can be solved in quadratures (a problem is reduced to quadrature when solving the dynamics has been reduced to evaluating an integral). Its two symmetries, no explicit time dependence and rotational symmetry about $z$, give two independent conserved quantities, energy $E$ and angular momentum $\ell_z$, which for a 2-degree-of-freedom system is enough to reduce the motion to quadrature. Eliminating $\dot\phi$ via $\ell_z$ turns the problem into an effective 1D equation in $\theta$ alone,

$$R\ddot{\theta} = \frac{\ell_z^2}{m^2 R^3}\frac{\cos\theta}{\sin^3\theta} - g\sin\theta.$$

This is a 2nd-order ordinary differential equation (ODE). Using energy

$$E = \dfrac{1}{2}mR^2\dot{\theta}^2 + \dfrac{1}{2}mR^2\dot{\phi}^2\sin^2\theta - mgR\cos\theta$$

and conservation of (the $z$-component of) the angular momentum 

$$\ell_z = mR^2\dot{\phi}\sin^2\theta \Longrightarrow \dot{\phi} = \dfrac{\ell_z}{mR^2\sin^2\theta},$$

the dynamics become 1st-order and reduce to quadrature

\begin{align*}
E &= \dfrac{1}{2}mR^2\dot{\theta}^2 + V_{eff}(\theta), \qquad \text{where } V_{eff}(\theta) = \frac{\ell_z^2}{2mR^2\sin^2\theta} - mgR\cos\theta \\
\dot{\theta} &= \pm \sqrt{\dfrac{2}{mR^2}[E - V_{eff}(\theta)]} \\
\int_{t_0}^{t} dt &= \pm \sqrt{\frac{mR^2}{2}} \int_{\theta(t_0)}^{\theta(t)} \frac{d\theta'}{\sqrt{E - V_{eff}(\theta')}}.
\end{align*}

After a suitable change of variables, the above quadrature for $t(\theta)$ can be expressed in terms of elliptic integrals, and $\theta(t)$ can be obtained by inversion. Elementary closed-form solutions appear in the special case the original problem asks about: steady conical rotation at a fixed $\theta_o$ (parts (c)-(d)), while the small harmonic oscillations about $\theta_o$ are an approximation (part (e)).

Coupling a second spherical pendulum onto the first keeps both symmetries intact, the system is still time-translation invariant, and still has the same combined rotational symmetry about $z$ (see below), but now there are 4 degrees of freedom and still those same 2 conserved quantities. That is no longer sufficient to guarantee Liouville integrability.

## 1.2 Describing the system using diagrams and mathematical expressions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from mpl_toolkits.mplot3d import proj3d

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection="3d")

R1, R2 = 1.0, 1.0
theta1 = np.radians(35)
phi1 = np.radians(20)
theta2 = np.radians(65)
phi2 = np.radians(-70)

pivot = np.array([0.0, 0.0, 0.0])

def geometry(theta1, phi1, theta2, phi2):
    r1 = R1 * np.array([
        np.sin(theta1) * np.cos(phi1),
        np.sin(theta1) * np.sin(phi1),
        np.cos(theta1)
    ])

    r2 = r1 + R2 * np.array([
        np.sin(theta2) * np.cos(phi2),
        np.sin(theta2) * np.sin(phi2),
        np.cos(theta2)
    ])

    return {
        "r1": r1,
        "r2": r2,
        "r1_proj": np.array([r1[0], r1[1], 0.0]),
        "r2_proj": np.array([r2[0], r2[1], r1[2]]),
        "r1_hat": r1 / R1,
        "r2_hat": (r2 - r1) / R2,
        "r1_proj_hat": np.array([np.cos(phi1), np.sin(phi1), 0.0]),
        "r2_proj_hat": np.array([np.cos(phi2), np.sin(phi2), 0.0])
    }

def make_arc(a, b, center, radius=0.4, n=50):
    a = a / np.linalg.norm(a)
    b = b / np.linalg.norm(b)

    angle = np.arccos(np.clip(np.dot(a, b), -1.0, 1.0))

    # No arc is needed if the directions coincide
    if angle < 1e-9:
        return np.tile(center + radius * a, (n, 1))

    # Points from 0 to 1 along the arc
    s_values = np.linspace(0, 1, n)

    points = []
    for s in s_values:
        v = (np.sin((1 - s) * angle) * a + np.sin(s * angle) * b) / np.sin(angle)
        points.append(center + radius * v)

    return np.array(points)

def make_label_2d(point, text, fontsize=13):
    x2, y2, _ = proj3d.proj_transform(point[0], point[1], point[2], ax.get_proj())
    label = ax.text2D(x2, y2, text, fontsize=fontsize)

    return label

def move_label_2d(label, point):
    x2, y2, _ = proj3d.proj_transform(point[0], point[1], point[2], ax.get_proj())
    label.set_position((x2, y2))

def update_label_projections():
    move_label_2d(m1_label, label_points["m1"])
    move_label_2d(m2_label, label_points["m2"])
    move_label_2d(R1_label, label_points["R1"])
    move_label_2d(R2_label, label_points["R2"])
    move_label_2d(theta1_label, label_points["theta1"])
    move_label_2d(phi1_label, label_points["phi1"])
    move_label_2d(theta2_label, label_points["theta2"])
    move_label_2d(phi2_label, label_points["phi2"])

def on_mouse_move(event):
    if event.inaxes == ax and event.button is not None:
        update_label_projections()
        fig.canvas.draw_idle()

def on_mouse_release(event):
    update_label_projections()
    fig.canvas.draw_idle()

g = geometry(theta1, phi1, theta2, phi2)

# Rod 1
rod1, = ax.plot(
    [pivot[0], g["r1"][0]],
    [pivot[1], g["r1"][1]],
    [pivot[2], g["r1"][2]],
    linewidth=3
)

# Rod 2
rod2, = ax.plot(
    [g["r1"][0], g["r2"][0]],
    [g["r1"][1], g["r2"][1]],
    [g["r1"][2], g["r2"][2]],
    linewidth=3
)

# Masses
mass1 = ax.scatter(*g["r1"], s=100)
mass2 = ax.scatter(*g["r2"], s=100)

# Plot dimensions
reach = R1 + R2

ax.set_xlim(-reach, reach)
ax.set_ylim(-reach, reach)
ax.set_zlim(reach, -reach)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z (down)")
ax.set_box_aspect((1, 1, 1))
ax.set_title("Double Spherical Pendulum")

# Coordinate axes
axis_length = 1.5
ax.quiver(0, 0, 0, axis_length, 0, 0, arrow_length_ratio=0.08)
ax.quiver(0, 0, 0, 0, axis_length, 0, arrow_length_ratio=0.08)
ax.quiver(0, 0, 0, 0, 0, axis_length, arrow_length_ratio=0.08)
ax.text(axis_length + 0.1, 0, 0, "x")
ax.text(0, axis_length + 0.1, 0, "y")
ax.text(0, 0, axis_length + 0.1, "z")

# Projection of r1 onto the x-y plane
proj1_horizontal, = ax.plot(
    [0, g["r1_proj"][0]],
    [0, g["r1_proj"][1]],
    [0, g["r1_proj"][2]],
    linestyle=":"
)
proj1_vertical, = ax.plot(
    [g["r1_proj"][0], g["r1"][0]],
    [g["r1_proj"][1], g["r1"][1]],
    [g["r1_proj"][2], g["r1"][2]],
    linestyle=":"
)
proj2_horizontal, = ax.plot(
    [g["r1"][0], g["r2_proj"][0]],
    [g["r1"][1], g["r2_proj"][1]],
    [g["r1"][2], g["r2_proj"][2]],
    linestyle=":"
)
proj2_vertical, = ax.plot(
    [g["r2_proj"][0], g["r2"][0]],
    [g["r2_proj"][1], g["r2"][1]],
    [g["r2_proj"][2], g["r2"][2]],
    linestyle=":"
)

local_axis_length = 0.8
x_prime, = ax.plot(
    [g["r1"][0], g["r1"][0] + local_axis_length],
    [g["r1"][1], g["r1"][1]],
    [g["r1"][2], g["r1"][2]],
    color="red",
    alpha=0.8
)
y_prime, = ax.plot(
    [g["r1"][0], g["r1"][0]],
    [g["r1"][1], g["r1"][1] + local_axis_length],
    [g["r1"][2], g["r1"][2]],
    color="red",
    alpha=0.8
)
z_prime, = ax.plot(
    [g["r1"][0], g["r1"][0]],
    [g["r1"][1], g["r1"][1]],
    [g["r1"][2], g["r1"][2] + local_axis_length],
    color="red",
    alpha=0.8
)

plane_size = 0.8
x0_plane = np.linspace(0 - plane_size, 0 + plane_size, 2)
y0_plane = np.linspace(0 - plane_size, 0 + plane_size, 2)
X0, Y0 = np.meshgrid(x0_plane, y0_plane)
Z0 = np.full_like(X0, 0)
ax.plot_surface(X0, Y0, Z0, alpha=0.2)

# Unit vectors we'll need for the angle arcs
x_hat = np.array([1.0, 0.0, 0.0])
z_hat = np.array([0.0, 0.0, 1.0])

theta1_arc = make_arc(z_hat, g["r1_hat"], pivot, radius=0.4)
theta1_arc_line, = ax.plot(
    theta1_arc[:, 0],
    theta1_arc[:, 1],
    theta1_arc[:, 2],
    linewidth=2
)

phi1_arc = make_arc(x_hat, g["r1_proj_hat"], pivot, radius=0.6)
phi1_arc_line, = ax.plot(
    phi1_arc[:, 0],
    phi1_arc[:, 1],
    phi1_arc[:, 2],
    linewidth=2
)

theta2_arc = make_arc(z_hat, g["r2_hat"], g["r1"], radius=0.4)
theta2_arc_line, = ax.plot(
    theta2_arc[:, 0],
    theta2_arc[:, 1],
    theta2_arc[:, 2],
    linewidth=2
)

phi2_arc = make_arc(x_hat, g["r2_proj_hat"], g["r1"], radius=0.6)
phi2_arc_line, = ax.plot(
    phi2_arc[:, 0],
    phi2_arc[:, 1],
    phi2_arc[:, 2],
    linewidth=2
)

m1_pos = g["r1"] + np.array([0.08, 0.08, 0.0])
m2_pos = g["r2"] + np.array([0.08, 0.08, 0.0])
m1_label = make_label_2d(m1_pos, r"$m_1$")
m2_label = make_label_2d(m2_pos, r"$m_2$")

R1_pos = (pivot + g["r1"]) / 2
R2_pos = (g["r1"] + g["r2"]) / 2
R1_label = make_label_2d(R1_pos, r"$R_1$")
R2_label = make_label_2d(R2_pos, r"$R_2$")

theta1_mid = theta1_arc[len(theta1_arc) // 2]
phi1_mid = phi1_arc[len(phi1_arc) // 2]
theta2_mid = theta2_arc[len(theta2_arc) // 2]
phi2_mid = phi2_arc[len(phi2_arc) // 2]

theta1_label = make_label_2d(theta1_mid, r"$\theta_1$")
phi1_label = make_label_2d(phi1_mid, r"$\phi_1$")
theta2_label = make_label_2d(theta2_mid, r"$\theta_2$")
phi2_label = make_label_2d(phi2_mid, r"$\phi_2$")

label_points = {
    "m1": m1_pos,
    "m2": m2_pos,
    "R1": R1_pos, 
    "R2": R2_pos, 
    "theta1": theta1_mid, 
    "phi1": phi1_mid,
    "theta2": theta2_mid,
    "phi2": phi2_mid
}

def update(change=None):
    theta1_new = np.radians(theta1_slider.value)
    phi1_new = np.radians(phi1_slider.value)
    theta2_new = np.radians(theta2_slider.value)
    phi2_new = np.radians(phi2_slider.value)

    g = geometry(theta1_new, phi1_new, theta2_new, phi2_new)

    # Update rods
    rod1.set_data_3d(
        [pivot[0], g["r1"][0]],
        [pivot[1], g["r1"][1]],
        [pivot[2], g["r1"][2]]
    )
    rod2.set_data_3d(
        [g["r1"][0], g["r2"][0]],
        [g["r1"][1], g["r2"][1]],
        [g["r1"][2], g["r2"][2]]
    )

    # Update masses
    mass1._offsets3d = (
        [g["r1"][0]],
        [g["r1"][1]],
        [g["r1"][2]]
    )
    mass2._offsets3d = (
        [g["r2"][0]],
        [g["r2"][1]],
        [g["r2"][2]]
    )

    # Update projections
    proj1_horizontal.set_data_3d(
        [0, g["r1_proj"][0]],
        [0, g["r1_proj"][1]],
        [0, g["r1_proj"][2]],
    )
    proj1_vertical.set_data_3d(
        [g["r1_proj"][0], g["r1"][0]],
        [g["r1_proj"][1], g["r1"][1]],
        [g["r1_proj"][2], g["r1"][2]] 
    )
    proj2_horizontal.set_data_3d(
        [g["r1"][0], g["r2_proj"][0]],
        [g["r1"][1], g["r2_proj"][1]],
        [g["r1"][2], g["r2_proj"][2]]
    )
    proj2_vertical.set_data_3d(
        [g["r2_proj"][0], g["r2"][0]],
        [g["r2_proj"][1], g["r2"][1]],
        [g["r2_proj"][2], g["r2"][2]]
    )

    # Recalculate angle arcs
    theta1_arc_new = make_arc(z_hat, g["r1_hat"], pivot, radius=0.4)
    phi1_arc_new = make_arc(x_hat, g["r1_proj_hat"], pivot, radius=0.6)
    theta2_arc_new = make_arc(z_hat, g["r2_hat"], g["r1"], radius=0.4)
    phi2_arc_new = make_arc(x_hat, g["r2_proj_hat"], g["r1"], radius=0.6)

    # Update angle-arc lines
    theta1_arc_line.set_data_3d(
        theta1_arc_new[:, 0],
        theta1_arc_new[:, 1],
        theta1_arc_new[:, 2]
    )
    phi1_arc_line.set_data_3d(
        phi1_arc_new[:, 0],
        phi1_arc_new[:, 1],
        phi1_arc_new[:, 2]
    )
    theta2_arc_line.set_data_3d(
        theta2_arc_new[:, 0],
        theta2_arc_new[:, 1],
        theta2_arc_new[:, 2]
    )
    phi2_arc_line.set_data_3d(
        phi2_arc_new[:, 0],
        phi2_arc_new[:, 1],
        phi2_arc_new[:, 2]
    )

    # Update primed axes
    x_prime.set_data_3d(
        [g["r1"][0], g["r1"][0] + local_axis_length],
        [g["r1"][1], g["r1"][1]],
        [g["r1"][2], g["r1"][2]]
    )
    y_prime.set_data_3d(
        [g["r1"][0], g["r1"][0]],
        [g["r1"][1], g["r1"][1] + local_axis_length],
        [g["r1"][2], g["r1"][2]]
    )
    z_prime.set_data_3d(
        [g["r1"][0], g["r1"][0]],
        [g["r1"][1], g["r1"][1]],
        [g["r1"][2], g["r1"][2] + local_axis_length]
    )

    # Update labels 
    m1_pos_new = g["r1"] + np.array([0.08, 0.08, 0.0])
    m2_pos_new = g["r2"] + np.array([0.08, 0.08, 0.0])
    R1_pos_new = (pivot + g["r1"]) / 2
    R2_pos_new = (g["r1"] + g["r2"]) / 2
    theta1_mid_new = theta1_arc_new[len(theta1_arc_new) // 2]
    phi1_mid_new = phi1_arc_new[len(phi1_arc_new) // 2]
    theta2_mid_new = theta2_arc_new[len(theta2_arc_new) // 2]
    phi2_mid_new = phi2_arc_new[len(phi2_arc_new) // 2]

    label_points["m1"] = m1_pos_new
    label_points["m2"] = m2_pos_new
    label_points["R1"] = R1_pos_new
    label_points["R2"] = R2_pos_new
    label_points["theta1"] = theta1_mid_new
    label_points["phi1"] = phi1_mid_new
    label_points["theta2"] = theta2_mid_new
    label_points["phi2"] = phi2_mid_new

    update_label_projections()

    # Redraw
    fig.canvas.draw_idle()

fig.canvas.mpl_connect("motion_notify_event", on_mouse_move)
fig.canvas.mpl_connect("button_release_event", on_mouse_release)

theta1_slider = widgets.FloatSlider(value=35, min=0, max=180, step=1, description=r"$\theta_1$")
phi1_slider = widgets.FloatSlider(value=20, min=-180, max=180, step=1, description=r"$\phi_1$")
theta2_slider = widgets.FloatSlider(value=65, min=0, max=180, step=1, description=r"$\theta_2$")
phi2_slider = widgets.FloatSlider(value=-70, min=-180, max=180, step=1, description=r"$\phi_2$")

theta1_slider.observe(update, names="value")
phi1_slider.observe(update, names="value")
theta2_slider.observe(update, names="value")
phi2_slider.observe(update, names="value")

display(
    widgets.VBox([
        theta1_slider, 
        phi1_slider, 
        theta2_slider,
        phi2_slider
    ])
)

The system has four generalized coordinates, $(\theta_1, \phi_1, \theta_2, \phi_2)$, together with the fixed rod lengths $R_1, R_2$ and bob masses $m_1, m_2$ shown in the diagram above. Using the spherical-polar convention above (the $z$-axis pointing straight down, $\theta_i$ measured from it, $\phi_i$ the azimuth about it), the position of bob 1 relative to the fixed pivot is

$$\vec r_1 = \big(R_1\sin\theta_1\cos\phi_1,\ \ R_1\sin\theta_1\sin\phi_1,\ \ R_1\cos\theta_1\big),$$

and bob 2's position is bob 1's position plus a second spherical displacement of length $R_2$, measured from a vertical axis through bob 1:

$$\vec r_2 = \vec r_1 + \big(R_2\sin\theta_2\cos\phi_2,\ \ R_2\sin\theta_2\sin\phi_2,\ \ R_2\cos\theta_2\big).$$

In components,

$$x_1 = R_1\sin\theta_1\cos\phi_1, \qquad y_1 = R_1\sin\theta_1\sin\phi_1, \qquad z_1 = R_1\cos\theta_1,$$

$$x_2 = x_1 + R_2\sin\theta_2\cos\phi_2, \qquad y_2 = y_1 + R_2\sin\theta_2\sin\phi_2, \qquad z_2 = z_1 + R_2\cos\theta_2.$$

Because $z$ points down, $z_1$ and $z_2$ are literally how far each bob hangs below the pivot, so each bob's gravitational potential energy is $-m_i g z_i$ (it decreases as the bob swings lower). These are exactly the Cartesian expressions differentiated below to build the kinetic energy $T$ and the Lagrangian $L = T - V$.

## 1.3 Describing the system in terms of the Lagrangian

**Kinetic energy.** Bob 1 traces a sphere of radius $R_1$ about the fixed pivot, so its speed-squared has the standard spherical-pendulum form (obtained by differentiating $x_1,y_1,z_1$ above and using $\cos^2+\sin^2=1$):

$$|\dot{\vec r}_1|^2 = R_1^2\left(\dot\theta_1^2 + \sin^2\theta_1\,\dot\phi_1^2\right), \qquad T_1 = \tfrac12 m_1 |\dot{\vec r}_1|^2.$$

Bob 2's position is $\vec r_2 = \vec r_1 + \vec s_2$, where $\vec s_2 = R_2(\sin\theta_2\cos\phi_2,\ \sin\theta_2\sin\phi_2,\ \cos\theta_2)$ is a second spherical displacement of the same functional form, so by the identical calculation $|\dot{\vec s}_2|^2 = R_2^2(\dot\theta_2^2+\sin^2\theta_2\,\dot\phi_2^2)$. Since $\dot{\vec r}_2 = \dot{\vec r}_1+\dot{\vec s}_2$,

$$T_2 = \tfrac12 m_2|\dot{\vec r}_2|^2 = \tfrac12 m_2\Big[R_1^2(\dot\theta_1^2+\sin^2\theta_1\dot\phi_1^2) + R_2^2(\dot\theta_2^2+\sin^2\theta_2\dot\phi_2^2) + 2\,\dot{\vec r}_1\cdot\dot{\vec s}_2\Big].$$

The cross term $\dot{\vec r}_1\cdot\dot{\vec s}_2$ is what physically **couples** the two pendulums, it vanishes only in the decoupled limit $R_2\to0$ or $m_2\to0$. It expands into a long, non-illuminating sum of trigonometric terms in $(\theta_1,\phi_1,\theta_2,\phi_2,\dot\theta_1,\dot\phi_1,\dot\theta_2,\dot\phi_2)$, so rather than expand it by hand we let `sympy` differentiate $x_2,y_2,z_2$ symbolically (as done above). That is exactly why the position vectors were built symbolically in the first place. Total kinetic energy: $T = T_1+T_2$.

**Potential energy.** Taking the pivot as the reference height and remembering $z$ points *down*, each bob's height *below* the pivot is exactly its $z$-coordinate, so

$$U = -m_1 g z_1 - m_2 g z_2 = -m_1 g R_1\cos\theta_1 - m_2 g\left(R_1\cos\theta_1+R_2\cos\theta_2\right).$$

**Lagrangian.**

$$L = T - U.$$

## 1.4 Symmetries, Conserved Quantities, and Assumptions

Both conservation laws below are instances of **Noether's theorem**: every continuous symmetry of the Lagrangian corresponds to a conserved quantity.

| Symmetry | Transformation | Conserved quantity |
|---|---|---|
| Time translation | $t \to t+\epsilon$ (no explicit $t$ in $L$) | Energy $H = T+U$ |
| Rotation about $z$ | $\phi_1\to\phi_1+\epsilon,\ \phi_2\to\phi_2+\epsilon$ | $p_{\phi_1}+p_{\phi_2}$ |

**Modeling assumptions:**
1. **Rigid, massless rods** ($R_1, R_2$ fixed): a holonomic constraint that lets us describe the system with 4 generalized coordinates $(\theta_1,\theta_2,\phi_1,\phi_2)$ instead of 6 Cartesian coordinates plus 2 constraint (Lagrange-multiplier) equations.
2. **Conservative forces only**: no air resistance or pivot friction, so every generalized force derives from a potential, $F_i = -\partial U/\partial q_i$, as required for the Euler-Lagrange equations to apply directly.
3. **No small-angle approximation**: the full nonlinear equations of motion are solved numerically rather than linearized, which is what makes chaotic behavior possible in the first place.

# 2. Visualizations

## 2.1 Numerical integration to solve the equations of motion - plots of trajectories in state/coordinate space

**From implicit to explicit equations of motion.** The four Euler-Lagrange equations `EL1`–`EL4` are *implicit* in the generalized accelerations $(\ddot\theta_1,\ddot\theta_2,\ddot\phi_1,\ddot\phi_2)$ where each one is a linear combination of all four, since $T$ is quadratic in the generalized velocities. That means they can be written in mass-matrix form

$$M(\theta_1,\theta_2,\phi_1,\phi_2)\,\ddot{\mathbf q} = F(\theta_1,\theta_2,\phi_1,\phi_2,\dot\theta_1,\dot\theta_2,\dot\phi_1,\dot\phi_2), \qquad \mathbf q = (\theta_1,\theta_2,\phi_1,\phi_2),$$

which is exactly what `sm.linear_eq_to_matrix` extracts and `M.LUsolve(F)` solves the resulting symbolic linear system via LU decomposition to get closed-form expressions for each acceleration. Solving this system symbolically once allows the resulting acceleration expressions to be lambdified and evaluated directly during numerical integration, rather than reconstructing and solving the Euler-Lagrange linear system at every RHS evaluation. 

**Reduction to a first-order system.** `scipy.integrate.solve_ivp` only accepts first-order systems, so each second-order equation is split using the auxiliary velocity variables introduced later, $u_i = \dot\theta_i$, $v_i = \dot\phi_i$, giving the 8-dimensional state vector

$$\mathbf U(t) = (\theta_1, u_1, \theta_2, u_2, \phi_1, v_1, \phi_2, v_2).$$

The `Pendulum.dy_dt(t, y)` method returns $\dot{\mathbf U}$: the "position" half is trivial ($\dot\theta_i = u_i$, $\dot\phi_i = v_i$), and the "velocity" half evaluates the lambdified closed-form accelerations at the current state.

**The solver.** `Pendulum.solve_ode` hands `dy_dt` to `solve_ivp` over the requested time span, sampled at the requested output times (`t_eval=t_pts`), with tolerances `atol=1e-8, rtol=1e-6` — tighter than SciPy's defaults, to keep numerical error well below the scale of the interesting dynamics. By default, `Pendulum.solve_ode` uses `method='RK45'`, **explicit Runge-Kutta 4(5)** (`RK45`, Dormand-Prince): it advances each step with a 5th-order estimate, compares it against an embedded 4th-order estimate to gauge local error, and shrinks/grows the step size to keep the estimated local error within the tolerance determined by  `atol` and `rtol`. This is examined more systematically in the work-precision study in Section 4.1

**Note.** RK45 is *not* symplectic, so it doesn't exactly conserve the energy $H$ identified in the Symmetries section. Over very long integrations, energy can drift slowly even though the tolerances keep local error tiny at each step. This is not expected to affect the qualitative trajectory and phase-space visualizations shown here, but it's worth keeping in mind if the chaos exploration (long-time sensitivity to initial conditions) later needs a sanity check that drift isn't being mistaken for genuine chaotic divergence.

In [ ]:
import numpy as np
from scipy.integrate import odeint, solve_ivp

import matplotlib.pyplot as plt
from matplotlib import animation, rc
from IPython.display import HTML

In [ ]:
# The dpi (dots-per-inch) setting will affect the resolution and how large
# the plots appear on screen and printed. So you may want/need to adjust 
# the figsize when creating the figure.
plt.rcParams['figure.dpi'] = 100.    # this is the default for notebook

# Change the common font size (smaller when higher dpi)
font_size = 10
plt.rcParams.update({'font.size': font_size})

In [ ]:
import sympy as sp

# Defining symbols
t, g = sp.symbols('t g')
m1, m2 = sp.symbols('m1 m2')
R1, R2 = sp.symbols('R1 R2')

# Defining functions
theta1, theta2, phi1, phi2 = sp.symbols(r'\theta_1, \theta_2, \phi_1, \phi_2', cls=sp.Function)

theta1 = theta1(t)
theta2 = theta2(t)
phi1 = phi1(t)
phi2 = phi2(t)

theta1_dot = sp.diff(theta1, t)
theta2_dot = sp.diff(theta2, t)
phi1_dot = sp.diff(phi1, t)
phi2_dot = sp.diff(phi2, t)
theta1_ddot = sp.diff(theta1_dot, t)
theta2_ddot = sp.diff(theta2_dot, t)
phi1_ddot = sp.diff(phi1_dot, t)
phi2_ddot = sp.diff(phi2_dot, t)

x1 = R1 * sp.sin(theta1) * sp.cos(phi1)
y1 = R1 * sp.sin(theta1) * sp.sin(phi1)
z1 = R1 * sp.cos(theta1)
x2 = x1 + R2 * sp.sin(theta2) * sp.cos(phi2)
y2 = y1 + R2 * sp.sin(theta2) * sp.sin(phi2)
z2 = z1 + R2 * sp.cos(theta2)

x1_num = sp.lambdify((theta1, theta2, phi1, phi2, R1, R2), x1)
y1_num = sp.lambdify((theta1, theta2, phi1, phi2, R1, R2), y1)
z1_num = sp.lambdify((theta1, theta2, phi1, phi2, R1, R2), z1)
x2_num = sp.lambdify((theta1, theta2, phi1, phi2, R1, R2), x2)
y2_num = sp.lambdify((theta1, theta2, phi1, phi2, R1, R2), y2)
z2_num = sp.lambdify((theta1, theta2, phi1, phi2, R1, R2), z2)

The Lagrangian

In [ ]:
T1 = sp.Rational(1, 2) * m1 * (sp.diff(x1, t)**2 + sp.diff(y1, t)**2 + sp.diff(z1, t)**2)
T2 = sp.Rational(1, 2) * m2 * (sp.diff(x2, t)**2 + sp.diff(y2, t)**2 + sp.diff(z2, t)**2)
T = T1 + T2

U1 = -m1 * g * z1
U2 = -m2 * g * z2
U = U1 + U2

L = T - U

and the Lagrange equations...

\begin{align*}
\frac{\partial L}{\partial \theta_1} - \frac{d}{dt} \frac{\partial L}{\partial \dot{\theta}_1} = 0 \\
\frac{\partial L}{\partial \theta_2} - \frac{d}{dt} \frac{\partial L}{\partial \dot{\theta}_2} = 0 \\
\frac{\partial L}{\partial \phi_1} - \frac{d}{dt} \frac{\partial L}{\partial \dot{\phi}_1} = 0 \\
\frac{\partial L}{\partial \phi_2} - \frac{d}{dt} \frac{\partial L}{\partial \dot{\phi}_2} = 0
\end{align*}

In [ ]:
EL1 = (sp.diff(sp.diff(L, theta1_dot), t) - sp.diff(L, theta1)).simplify()
EL2 = (sp.diff(sp.diff(L, theta2_dot), t) - sp.diff(L, theta2)).simplify()
EL3 = (sp.diff(sp.diff(L, phi1_dot), t) - sp.diff(L, phi1)).simplify()
EL4 = (sp.diff(sp.diff(L, phi2_dot), t) - sp.diff(L, phi2)).simplify()

In [ ]:
# sols = sp.solve([EL1, EL2, EL3, EL4], [theta1_ddot, theta2_ddot, phi1_ddot, phi2_ddot], simplify=False, rational=False)

M, F = sp.linear_eq_to_matrix([EL1, EL2, EL3, EL4], [theta1_ddot, theta2_ddot, phi1_ddot, phi2_ddot])
accel = M.LUsolve(F)

sols = {
    theta1_ddot: accel[0],
    theta2_ddot: accel[1],
    phi1_ddot: accel[2],
    phi2_ddot: accel[3],
}

We have four second-order ODEs. 

\begin{align*}
\frac{d^2\theta_1}{dt^2} = ... \\
\frac{d^2\theta_2}{dt^2} = ... \\
\frac{d^2\phi_1}{dt^2} = ... \\
\frac{d^2\phi_2}{dt^2} = ...
\end{align*}

We can turn this into eight first-order ODEs by defining new dependent variables. For example,

\begin{align*}
u_1 = \frac{d\theta_1}{dt} \\
u_2 = \frac{d\theta_2}{dt} \\
v_1 = \frac{d\phi_1}{dt} \\
v_2 = \frac{d\phi_2}{dt}
\end{align*}

Now introduce the vector 

\begin{align*}
\textbf{U}(t) = 
\begin{pmatrix}
\theta_1 \\
u_1 \\
\theta_2 \\ 
u_2 \\
\phi_1 \\
v_1 \\
\phi_2 \\
v_2
\end{pmatrix}
\Longrightarrow
\frac{d\textbf{U}}{dt} = 
\begin{pmatrix}
u_1 \\
\dfrac{du_1}{dt} \\
u_2 \\
\dfrac{du_2}{dt} \\
v_1 \\
\dfrac{dv_1}{dt} \\
v_2 \\
\dfrac{dv_2}{dt}
\end{pmatrix}
\end{align*}

In [ ]:
accel_args = (t, g, m1, m2, R1, R2, theta1, theta2, theta1_dot, theta2_dot, phi1, phi2, phi1_dot, phi2_dot)

dtheta1_dt_num = sp.lambdify(theta1_dot, theta1_dot)
du1_dt_num = sp.lambdify(accel_args, sols[theta1_ddot])
dtheta2_dt_num = sp.lambdify(theta2_dot, theta2_dot)
du2_dt_num = sp.lambdify(accel_args, sols[theta2_ddot])

dphi1_dt_num = sp.lambdify(phi1_dot, phi1_dot) 
dv1_dt_num = sp.lambdify(accel_args, sols[phi1_ddot])
dphi2_dt_num = sp.lambdify(phi2_dot, phi2_dot)
dv2_dt_num = sp.lambdify(accel_args, sols[phi2_ddot])

In [ ]:
accel_num = sp.lambdify(accel_args, [sols[theta1_ddot], sols[theta2_ddot], sols[phi1_ddot], sols[phi2_ddot]], cse=True)

### Pendulum class and utility functions

In [ ]:
class Pendulum():
    """
    Pendulum class implements the parameters and differential equation for 
     a pendulum using the notation from Taylor.
     
    Parameters
    ----------
    m1 : float
        mass of the first pendulum bob
    m2 : float
        mass of the second pendulum bob
    R1 : float
        length of the first pendulum rod
    R2 : float
        length of the second pendulum rod
    g : float
        acceleration due to gravity

    Methods
    -------
    dy_dt(y, t)
        Returns the right side of the differential equation in vector y, 
        given time t and the corresponding value of y.
    """

    def __init__(self, m1=1., m2=1., R1=1., R2=1., g=9.8):
        self.m1 = m1
        self.m2 = m2
        self.R1 = R1
        self.R2 = R2
        self.g = g 

    def dy_dt(self, t, y):
        """
        This function returns the right-hand side of the diffeq:
        [dtheta_1/dt, du_1/dt, dtheta_2/dt, du_2/dt, dphi_1/dt, dv_1/dt, dphi_2/dt, dv_2/dt]

        Parameters
        ----------
        y : array-like
            A vector containing the current values of the system variables.
        t : float
            time 

        Returns
        -------

        """
        theta1, u1, theta2, u2, phi1, v1, phi2, v2 = y
        theta1_ddot, theta2_ddot, phi1_ddot, phi2_ddot = accel_num(t, self.g, self.m1, self.m2, self.R1, self.R2, 
                                                                   theta1, theta2, u1, u2, phi1, phi2, v1, v2)
        return np.array([u1, theta1_ddot, u2, theta2_ddot, v1, phi1_ddot, v2, phi2_ddot])

    def solve_ode(self, t_pts, theta1_0, u1_0, theta2_0, u2_0, phi1_0, v1_0, phi2_0, v2_0,
                  abserr=1.0e-8, relerr=1.0e-6, method='RK45'):
        """
        Solve the ODE given the array of time points and initial conditions.
        Use solve_ivp.
        Specify smaller abserr and relerr to get more precision.

        `method` selects the solve_ivp integrator.  The default 'RK45' is what
        Secs. 2-4 were produced with, so it is kept as the default to leave those
        results unchanged.  The work-precision study of Sec. 4.1 found 'DOP853'
        more efficient over the tested tolerance range, making it preferable for 
        the long or high-accuracy runs considered there. 
        """
        y = [theta1_0, u1_0, theta2_0, u2_0, phi1_0, v1_0, phi2_0, v2_0]
        theta1, u1, theta2, u2, phi1, v1, phi2, v2 = solve_ivp(self.dy_dt, (t_pts[0], t_pts[-1]), y, t_eval=t_pts, 
                                                               atol=abserr, rtol=relerr,
                                                               method=method).y
        return theta1, u1, theta2, u2, phi1, v1, phi2, v2

    def solve_ode_benchmark(self, t_pts, theta1_0, u1_0, theta2_0, u2_0, phi1_0, v1_0, phi2_0, v2_0,
                  abserr=1.0e-8, relerr=1.0e-6, method='RK45'):
        """
        Solve the DSP equations and return the full solve_ivp result
        for numerical integration benchmark. 
        Used in Section 4.1
        """
        y = [theta1_0, u1_0, theta2_0, u2_0, phi1_0, v1_0, phi2_0, v2_0]
        sol = solve_ivp(self.dy_dt, (t_pts[0], t_pts[-1]), y, t_eval=t_pts, 
                                                               atol=abserr, rtol=relerr,
                                                               method=method)
        return sol

In [ ]:
def plot_y_vs_x(x, y, axis_labels=None, label=None, title=None, 
                color=None, linestyle=None, semilogy=False, loglog=False,
                points=False, ax=None):
    """
    Generic plotting function: return a figure axis with a plot of y vs. x,
    with line color and style, title, axis labels, and line label
    """
    if ax is None:        # if the axis object doesn't exist, make one
        ax = plt.gca()

    if (semilogy):
        line, = ax.semilogy(x, y, label=label, 
                            color=color, linestyle=linestyle)
    elif (loglog):
        line, = ax.loglog(x, y, label=label, 
                          color=color, linestyle=linestyle)
    else:
        if not points:
            line, = ax.plot(x, y, label=label, 
                        color=color, linestyle=linestyle)
        else:
            line = ax.scatter(x, y, label=label, color=color, marker='^')

    if label is not None:    # if a label is passed, show the legend
        ax.legend()
    if title is not None:    # set a title if one is passed
        ax.set_title(title)
    if axis_labels is not None:  # set x-axis and y-axis labels if passed  
        ax.set_xlabel(axis_labels[0])
        ax.set_ylabel(axis_labels[1])

    return ax, line

In [ ]:
def start_stop_indices(t_pts, plot_start, plot_stop):
    """Given an array (e.g., of times) and desired starting and stop values,
        return the array indices that are closest to those values.
    """
    start_index = (np.fabs(t_pts-plot_start)).argmin()  # index in t_pts array 
    stop_index = (np.fabs(t_pts-plot_stop)).argmin()  # index in t_pts array 
    return start_index, stop_index

### AnimationPendulumPlot class

In [ ]:
class AnimationPendulumPlot():
    """
    AnimationPlot class uses matplotlib.animation.FuncAnimation to animate
     the dynamics of an oscillator.  This includes a simple time dependence
     graph, a state space graph with Poincare map, and a physical model.
     
     We'll start with a pendulum and then generalize later.
     
    Parameters
    ----------
    phi_vs_t : boolean
        If True, plot phi(t) vs. t
    
    phi_dot_vs_t : boolean
        If True, plot phi_dot(t) vs. t
    
    state_space : boolean
        If True, plot phi_dot(t) s. phi(t)
    
    physics_pend : boolean
        If True, draw the pendulum at phi(t) vs. t


    Methods
    -------
    plot_setup
    
    t_pts_init
    
    add_pendulum
    
    animate_pendulum
    
    plot_setup
    
    start_animation
    """
    def __init__(self, phi_vs_t=True, phi_dot_vs_t=False,
                 state_space=True, physical_pend=True):
        self.theta1_list = []
        self.theta1_dot_list = []
        self.theta2_list = []
        self.theta2_dot_list = []
        self.phi1_list = []
        self.phi1_dot_list = []
        self.phi2_list = []
        self.phi2_dot_list = []
        self.length = 0.8
        self.line_colors = ['blue', 'red']
        self.pt_colors = ['black', 'brown']
        self.q_align = ['left', 'right']
                  
    def t_pts_init(self, t_start=0., t_end=100., delta_t=0.01): 
        """Create the array of time points for the full iteration"""
        self.t_start = t_start
        self.t_end = t_end
        self.delta_t = delta_t
        self.t_pts = np.arange(t_start, t_end+delta_t, delta_t)  

    def add_pendulum(self, pend, theta1_0=0., u1_0=0., theta2_0=0., u2_0=0., phi1_0=0., v1_0=0., phi2_0=0., v2_0=0.):
        """Add a pendulum to be plotted as a class instance of Pendulum
            along with initial conditions.  So it knows all of the parameters
            as well through the Pendulum class.
        """
        self.pend = pend
        theta1, u1, theta2, u2, phi1, v1, phi2, v2 = pend.solve_ode(self.t_pts, theta1_0, u1_0, theta2_0, u2_0, phi1_0, v1_0, phi2_0, v2_0)
        self.theta1_list.append(theta1)
        self.theta1_dot_list.append(u1)
        self.theta2_list.append(theta2)
        self.theta2_dot_list.append(u2)
        self.phi1_list.append(phi1)
        self.phi1_dot_list.append(v1)
        self.phi2_list.append(phi2)
        self.phi2_dot_list.append(v2)
    
    def plot_setup(self, plot_start, plot_end):
        """Set up the plots to be displayed. """

            # start the plot!
#         overall_title = 'Parameters:  ' + \
#                         rf' $\omega = {omega_ext:.2f},$' + \
#                         rf' $\gamma = {gamma_ext:.3f},$' + \
#                         rf' $\omega_0 = {omega_0:.2f},$' + \
#                         rf' $\beta = {beta:.2f},$' + \
#                         rf'  $\phi_0 = {phi_0:.2f},$' + \
#                         rf' $\dot\phi_0 = {phi_dot_0:.2f}$' + \
#                         '\n'     # \n means a new line (adds some space here)
#         self.fig = plt.figure(figsize=(10,3.3), num='Pendulum Plots')
#         self.fig.suptitle(overall_title, va='top')

        # Labels for individual plot axes
        theta1_vs_time_labels = (r'$t$', r'$\theta_1(t)$')
        theta1_dot_vs_time_labels = (r'$t$', r'$d\theta_1/dt(t)$')
        theta2_vs_time_labels = (r'$t$', r'$\theta_2(t)$')
        theta2_dot_vs_time_labels = (r'$t$', r'$d\theta_2/dt(t)$')
        phi1_vs_time_labels = (r'$t$', r'$\phi_1(t)$')
        phi1_dot_vs_time_labels = (r'$t$', r'$d\phi_1/dt(t)$')
        phi2_vs_time_labels = (r'$t$', r'$\phi_2(t)$')
        phi2_dot_vs_time_labels = (r'$t$', r'$d\phi_2/dt(t)$')
        state_space_labels_theta1 = (r'$\theta_1$', r'$d\theta_1/dt$')
        state_space_labels_theta2 = (r'$\theta_2$', r'$d\theta_2/dt$')
        state_space_labels_phi1 = (r'$\phi_1$', r'$d\phi_1/dt$')
        state_space_labels_phi2 = (r'$\phi_2$', r'$d\phi_2/dt$')
            
        self.fig = plt.figure(figsize=(15, 5), num='Pendulum animation')

        self.ax_1 = self.fig.add_subplot(1,3,1)    
        self.ax_1.set_xlabel(r'$t$')
        self.ax_1.set_ylabel(r'$q_i(t)$')
        self.line_1 = []
        self.pt_theta1 = []
        self.pt_theta2 = []
        self.pt_phi1 = []
        self.pt_phi2 = []
        
        self.ax_2 = self.fig.add_subplot(1,3,2, projection='3d')
        self.ax_2.set_box_aspect((1, 1, 1))   # aspect ratio 1 subplot
        # self.ax_2.set_rorigin(0.)   # origin in the middle
        # self.ax_2.set_theta_zero_location('S')  # phi=0 at the bottom
        L = self.pend.R1 + self.pend.R2
        self.ax_2.set_xlim(-L, L)
        self.ax_2.set_ylim(-L, L)  # r goes from 0 to L
        self.ax_2.set_zlim(L, -L)
        self.ax_2.grid(False)   # no longitude/lattitude lines
        self.ax_2.set_xticklabels([])   # turn off angle labels
        self.ax_2.set_yticklabels([])    # turn off radial labels
        self.ax_2.set_zticklabels([])
        # self.ax_2.spines['polar'].set_visible(False) # no circular border
        self.line_2_1 = []
        self.line_2_2 = []
        self.pt_2_1 = []
        self.pt_2_2 = []
        self.q_text = []       
        
        self.ax_3 = self.fig.add_subplot(1,3,3)
        self.ax_3.set_xlabel(r'$q_i$')
        self.ax_3.set_ylabel(r'$\dot{q}_i$')
        self.line_3 = []
        self.pt_3_1 = []
        self.pt_3_2 = []
        self.pt_3_3 = []
        self.pt_3_4 = []


        # plot new arrays from start to stop
        self.start, self.stop = start_stop_indices(self.t_pts, plot_start, 
                                                   plot_end)
        self.t_pts_plt = self.t_pts[self.start : self.stop]
        self.theta1_plt_list = []
        self.theta1_dot_plt_list = []
        self.theta2_plt_list = []
        self.theta2_dot_plt_list = []
        self.phi1_plt_list = []
        self.phi1_dot_plt_list = []
        self.phi2_plt_list = []
        self.phi2_dot_plt_list = []
        for i, (theta1, theta1_dot, theta2, theta2_dot, phi1, phi1_dot, phi2, phi2_dot) in enumerate(zip(self.theta1_list, self.theta1_dot_list, self.theta2_list, self.theta2_dot_list,
                                                                 self.phi1_list, self.phi1_dot_list, self.phi2_list, self.phi2_dot_list)):
            theta1_plt = theta1[self.start : self.stop]
            self.theta1_plt_list.append(theta1_plt)

            theta1_dot_plt = theta1_dot[self.start : self.stop]
            self.theta1_dot_plt_list.append(theta1_dot_plt)

            theta2_plt = theta2[self.start : self.stop]
            self.theta2_plt_list.append(theta2_plt)

            theta2_dot_plt = theta2_dot[self.start : self.stop]
            self.theta2_dot_plt_list.append(theta2_dot_plt)

            phi1_plt = phi1[self.start : self.stop]
            self.phi1_plt_list.append(phi1_plt)

            phi1_dot_plt = phi1_dot[self.start : self.stop]
            self.phi1_dot_plt_list.append(phi1_dot_plt)

            phi2_plt = phi2[self.start : self.stop]
            self.phi2_plt_list.append(phi2_plt)

            phi2_dot_plt = phi2_dot[self.start : self.stop]
            self.phi2_dot_plt_list.append(phi2_dot_plt)

            line_1_theta1, = self.ax_1.plot(self.t_pts_plt, theta1_plt, 
                                            color=self.line_colors[i])
            self.line_1.append(line_1_theta1)
            pt_1_theta1, = self.ax_1.plot(self.t_pts_plt[0], theta1_plt[0], 
                                          'o', color=self.pt_colors[i])
            self.pt_theta1.append(pt_1_theta1)

            line_1_theta2, = self.ax_1.plot(self.t_pts_plt, theta2_plt,
                                            color=self.line_colors[i])
            self.line_1.append(line_1_theta2)
            pt_1_theta2, = self.ax_1.plot(self.t_pts_plt[0], theta2_plt[0],
                                          'o', color=self.pt_colors[i])
            self.pt_theta2.append(pt_1_theta2)

            line_1_phi1, = self.ax_1.plot(self.t_pts_plt, phi1_plt,
                                          color=self.line_colors[i])
            self.line_1.append(line_1_phi1)
            pt_1_phi1, = self.ax_1.plot(self.t_pts_plt[0], phi1_plt[0],
                                      'o', color=self.pt_colors[i])
            self.pt_phi1.append(pt_1_phi1)

            line_1_phi2, = self.ax_1.plot(self.t_pts_plt, phi2_plt,
                                          color=self.line_colors[i])
            self.line_1.append(line_1_phi2)
            pt_1_phi2, = self.ax_1.plot(self.t_pts_plt[0], phi2_plt[0],
                                        'o', color=self.pt_colors[i])
            self.pt_phi2.append(pt_1_phi2)

            x1_0 = x1_num(theta1_plt[0], theta2_plt[0], phi1_plt[0], phi2_plt[0], self.pend.R1, self.pend.R2)
            y1_0 = y1_num(theta1_plt[0], theta2_plt[0], phi1_plt[0], phi2_plt[0], self.pend.R1, self.pend.R2)
            z1_0 = z1_num(theta1_plt[0], theta2_plt[0], phi1_plt[0], phi2_plt[0], self.pend.R1, self.pend.R2)
            x2_0 = x2_num(theta1_plt[0], theta2_plt[0], phi1_plt[0], phi2_plt[0], self.pend.R1, self.pend.R2)
            y2_0 = y2_num(theta1_plt[0], theta2_plt[0], phi1_plt[0], phi2_plt[0], self.pend.R1, self.pend.R2)
            z2_0 = z2_num(theta1_plt[0], theta2_plt[0], phi1_plt[0], phi2_plt[0], self.pend.R1, self.pend.R2)

            self.ax_2.plot([0.], [0.], [0.], color='black', marker='o', markersize=5)
            line_2_1, = self.ax_2.plot([0., x1_0],
                                       [0., y1_0],
                                       [0., z1_0],  
                                       color=self.line_colors[i], lw=3)
            self.line_2_1.append(line_2_1)
            pt_2_1, = self.ax_2.plot([x1_0], [y1_0], [z1_0],  
                                     marker='o', markersize=15,
                                     color=self.pt_colors[i])
            self.pt_2_1.append(pt_2_1)
            line_2_2, = self.ax_2.plot([x1_0, x2_0],
                                       [y1_0, y2_0],
                                       [z1_0, z2_0],
                                       color=self.line_colors[i], lw=3)
            self.line_2_2.append(line_2_2)
            pt_2_2, = self.ax_2.plot([x2_0], [y2_0], [z2_0], 
                                     marker='o', markersize=15, 
                                     color=self.pt_colors[i])
            self.pt_2_2.append(pt_2_2)
            q_string = (
                rf'$\theta_1={theta1_plt[0]:.2f}$' + '\n' + 
                rf'$\theta_2={theta2_plt[0]:.2f}$' + '\n' + 
                rf'$\phi_1={phi1_plt[0]:.2f}$' + '\n' + 
                rf'$\phi_2={phi2_plt[0]:.2f}$'
            )
            q_text = self.ax_2.text2D(0.05, 0.95 - 0.25*i, q_string,
                                      transform=self.ax_2.transAxes, 
                                      horizontalalignment=self.q_align[i])
            self.q_text.append(q_text)

            line_3_1, = self.ax_3.plot(theta1_plt, theta1_dot_plt,
                                       color=self.line_colors[i])
            self.line_3.append(line_3_1)
            pt_3_1, = self.ax_3.plot(theta1_plt[0], theta1_dot_plt[0],
                                     'o', color=self.pt_colors[i])
            self.pt_3_1.append(pt_3_1)

            line_3_2, = self.ax_3.plot(theta2_plt, theta2_dot_plt,
                                       color=self.line_colors[i])
            self.line_3.append(line_3_2)
            pt_3_2, = self.ax_3.plot(theta2_plt[0], theta2_dot_plt[0],
                                     'o', color=self.pt_colors[i])
            self.pt_3_2.append(pt_3_2)

            line_3_3, = self.ax_3.plot(phi1_plt, phi1_dot_plt,
                                       color=self.line_colors[i])
            self.line_3.append(line_3_3)
            pt_3_3, = self.ax_3.plot(phi1_plt[0], phi1_dot_plt[0],
                                     'o', color=self.pt_colors[i])
            self.pt_3_3.append(pt_3_3)

            line_3_4, = self.ax_3.plot(phi2_plt, phi2_dot_plt,
                                       color=self.line_colors[i])
            self.line_3.append(line_3_4)
            pt_3_4, = self.ax_3.plot(phi2_plt[0], phi2_dot_plt[0],
                                     'o', color=self.pt_colors[i])
            self.pt_3_4.append(pt_3_4)

        self.fig.tight_layout()

    def animate_pendulum(self, i, t_pts_skip, theta1_skip_list, theta1_dot_skip_list, theta2_skip_list, theta2_dot_skip_list, 
                         phi1_skip_list, phi1_dot_skip_list, phi2_skip_list, phi2_dot_skip_list):
        for index, (theta1_skip, theta1_dot_skip, theta2_skip, theta2_dot_skip, phi1_skip, phi1_dot_skip, phi2_skip, phi2_dot_skip) in \
        enumerate(zip(theta1_skip_list, theta1_dot_skip_list, theta2_skip_list, theta2_dot_skip_list, phi1_skip_list, phi1_dot_skip_list, phi2_skip_list, phi2_dot_skip_list)):
            self.pt_theta1[index].set_data([t_pts_skip[i]], [theta1_skip[i]])
            self.pt_theta2[index].set_data([t_pts_skip[i]], [theta2_skip[i]])
            self.pt_phi1[index].set_data([t_pts_skip[i]], [phi1_skip[i]])
            self.pt_phi2[index].set_data([t_pts_skip[i]], [phi2_skip[i]])

            x1 = x1_num(theta1_skip[i], theta2_skip[i], phi1_skip[i], phi2_skip[i], self.pend.R1, self.pend.R2)
            y1 = y1_num(theta1_skip[i], theta2_skip[i], phi1_skip[i], phi2_skip[i], self.pend.R1, self.pend.R2)
            z1 = z1_num(theta1_skip[i], theta2_skip[i], phi1_skip[i], phi2_skip[i], self.pend.R1, self.pend.R2)
            x2 = x2_num(theta1_skip[i], theta2_skip[i], phi1_skip[i], phi2_skip[i], self.pend.R1, self.pend.R2)
            y2 = y2_num(theta1_skip[i], theta2_skip[i], phi1_skip[i], phi2_skip[i], self.pend.R1, self.pend.R2)
            z2 = z2_num(theta1_skip[i], theta2_skip[i], phi1_skip[i], phi2_skip[i], self.pend.R1, self.pend.R2)

            self.line_2_1[index].set_data_3d([0., x1], [0., y1], [0., z1])
            self.pt_2_1[index].set_data_3d([x1], [y1], [z1])
            self.line_2_2[index].set_data_3d([x1, x2], [y1, y2], [z1, z2])
            self.pt_2_2[index].set_data_3d([x2], [y2], [z2])
            q_string = (
                rf'$\theta_1={theta1_skip[i]:.2f}$' + '\n' + 
                rf'$\theta_2={theta2_skip[i]:.2f}$' + '\n' + 
                rf'$\phi_1={phi1_skip[i]:.2f}$' + '\n' + 
                rf'$\phi_2={phi2_skip[i]:.2f}$' 
            )
            self.q_text[index].set_text(q_string)

            self.pt_3_1[index].set_data([theta1_skip[i]], [theta1_dot_skip[i]])
            self.pt_3_2[index].set_data([theta2_skip[i]], [theta2_dot_skip[i]])
            self.pt_3_3[index].set_data([phi1_skip[i]], [phi1_dot_skip[i]])
            self.pt_3_4[index].set_data([phi2_skip[i]], [phi2_dot_skip[i]])
        
        #return self.pt_1_theta1, self.pt_1_theta2, self.pt_1_phi1, self.pt_1_phi2, self.line_2_1, self.pt_2_1, self.line_2_2, self.pt_2_2, self.q_text, self.pt_3_1, self.pt_3_2, self.pt_3_3, self.pt_3_4
        
    def start_animation(self, skip=2, interval=25.):
        self.skip = skip          # skip between points in t_pts array
        self.interval = interval  # time between frames in milliseconds
        
        theta1_skip_list = []
        theta1_dot_skip_list = []
        theta2_skip_list = []
        theta2_dot_skip_list = []
        phi1_skip_list = []
        phi1_dot_skip_list = []
        phi2_skip_list = []
        phi2_dot_skip_list = []
        for i, (theta1_plt, theta1_dot_plt, theta2_plt, theta2_dot_plt, phi1_plt, phi1_dot_plt, phi2_plt, phi2_dot_plt) in enumerate(zip(self.theta1_plt_list, 
                                                                                                                                         self.theta1_dot_plt_list,
                                                                                                                                         self.theta2_plt_list,
                                                                                                                                         self.theta2_dot_plt_list,
                                                                                                                                         self.phi1_plt_list,
                                                                                                                                         self.phi1_dot_plt_list,
                                                                                                                                         self.phi2_plt_list,
                                                                                                                                         self.phi2_dot_plt_list)):
            theta1_skip_list.append(theta1_plt[::self.skip])
            theta1_dot_skip_list.append(theta1_dot_plt[::self.skip])
            theta2_skip_list.append(theta2_plt[::self.skip])
            theta2_dot_skip_list.append(theta2_dot_plt[::self.skip])
            phi1_skip_list.append(phi1_plt[::self.skip])
            phi1_dot_skip_list.append(phi1_dot_plt[::self.skip])
            phi2_skip_list.append(phi2_plt[::self.skip])
            phi2_dot_skip_list.append(phi2_dot_plt[::self.skip])

        t_pts_skip = self.t_pts_plt[::self.skip]
        self.anim = animation.FuncAnimation(self.fig, self.animate_pendulum, 
                           fargs=(t_pts_skip, 
                                  theta1_skip_list, theta1_dot_skip_list,
                                  theta2_skip_list, theta2_dot_skip_list,
                                  phi1_skip_list, phi1_dot_skip_list,
                                  phi2_skip_list, phi2_dot_skip_list), 
                           init_func=None,
                           frames=len(t_pts_skip), 
                           interval=self.interval, 
                           blit=False, repeat=False, 
                           save_count=0)

        return HTML(self.anim.to_jshtml())
        # self.fig.show() 

### Plots to animate

The animation cells may require several minutes to execute, depending on the simulation duration, number of frames, and available hardware. 
This is expected behavior and does not necessarily indicate an issue with the numerical solver. 
Subsequent animations may have different execution times depending on their configurations.

In [ ]:
# Create a pendulum animation instance. 
pendulum_anim = AnimationPendulumPlot(phi_vs_t=True, 
                                      state_space=True,
                                      physical_pend=True)

# Common plotting time (generate the full time here then use slices below)
t_start = 0.
t_end = 200.
delta_t = 0.01
pendulum_anim.t_pts_init(t_start, t_end, delta_t)

# Pendulum parameters
m1_param = 1.
m2_param = 1.
R1_param = 1.
R2_param = 1.
g_param = 9.8

# Instantiate a pendulum 
p1 = Pendulum(m1=m1_param, m2=m2_param, R1=R1_param, R2=R2_param, g=g_param)

# Initial conditions specified
theta1_0 = np.pi / 3
u1_0 = 3.0
theta2_0 = np.pi / 3
u2_0 = 4.0
phi1_0 = 0.  # -np.pi / 2.
v1_0 = -3.0
phi2_0 = -np.pi
v2_0 = -4.0

# Add a pendulum to the animation plots; this solves the differential
#  equation for the full t_pts array, generating phi and phi_dot internally. 
pendulum_anim.add_pendulum(p1, theta1_0, u1_0, theta2_0, u2_0, phi1_0, v1_0, phi2_0, v2_0)

In [ ]:
import matplotlib as mpl

mpl.rcParams['animation.embed_limit'] = 300  # MB

In [ ]:
plot_start = 0.   # time to begin plotting
plot_end = 30.    # time to end plotting
pendulum_anim.plot_setup(plot_start, plot_end)

# Start the animation (adjust skip and interval for a smooth plot at a 
#  useful speed)
skip = 2        # skip between time points (in units of delta_t) 
interval = 15.  # time between frames in milliseconds
# pendulum_anim.start_animation(skip, interval)
display(pendulum_anim.start_animation(skip, interval))

### Interpreting the state/coordinate space plots

**Coordinate- and state-space visualization.** The animation displays the numerically integrated double spherical pendulum trajectory over $0 \leq t \leq 30$ from three complementary viewpoints. The left panel shows the four generalized coordinates $q_i = (\theta_1, \theta_2, \phi_1, \phi_2)$ as functions of time, with moving markers indicating the instantaneous state of the system. The center panel reconstructs the corresponding physical motion in three-dimensional Cartesian space using the spherical coordinate relaations for the two masses, providing a direct geometri visualization of the coupled pendulum dynamics. The right panel shows four two-dimensional state-space projections, $(\theta_1, \dot{\theta}_1)$, $(\theta_2, \dot{\theta}_2)$, $(\phi_1, \dot{\phi}_1)$, and $(\phi_2, \dot{\phi}_2)$. These projections are slices of the full eight-dimensional state space rather than a complete state-space representation. Together, they provide a more intuitive view of how each generalized coordinate evolves with its conjugate velocity. The irregular, non-repeating trajectories and continually changing loops reflect the strongly coupled nonlinear motion of the system, in contrast with the closed or nearly closed state-space curves expected for simple periodic motion. 

**Parameters.** The trajectory shown uses $m_1 = m_2 = 1$, $R_1 = R_2 = 1$, and $g = 9.8$, with initial conditions $\theta_1(0) = \theta_2(0) = \pi/3$, $\dot{\theta}_1(0) = 3$, $\dot{\theta}_2 = 4$, $\phi_1(0) = 0$, $\dot{\phi}_1(0) = -3$, $\phi_2(0) = -\pi$, and $\dot{\phi}_2(0) = -4$. The equations of motion are integrated using RK45 with an output spacing of $\Delta t = 0.01$. The animation displays the first $30$ time units of the resulting trajectory. 

## 2.2 Vector field plots for the differential equations across conditions

### Special Cases and Reduced Dynamics
The general double spherical pendulum is described by four generalized coordinates, $(\theta_1, \theta_2, \phi_1, \phi_2)$, and  their corresponding velocities, giving an eight-dimensional state space. Although the equations of motion define a vector field throughout this space, an eight-dimensional vector field cannot be visualized directly in the same way as the two-dimensional phase portrait of a simple pendulum. To obtain a more intuitive picture of the dynamics, special motions for which the effective dynamics can be reduced to lower-dimensional systems were considered. 

One such special case is **conical motion**. For a single spherical pendulum, conical motion occurs when the bob moves in a horizontal circle while maintaining a constant polar angle. The string therefore sweeps out the surface of a cone as the pendulum rotates about the vertical axis. Mathematically, the constant polar angle requires 

$$\theta(t) = \theta_0, \qquad \dot{\theta} = \ddot{\theta} = 0,$$

while conservation of the vertical component of angular momentum implies uniform azimuthal motion,

$$\phi(t) = \omega t + \phi_0, \qquad \dot{\phi} = \omega, \qquad \ddot{\phi} = 0.$$

The analogous motion for the double spherical pendulum is a **double-conical relative equilibrium**. Physically, both links maintain fixed polar angles and a fixed orientation relative to one another while the entire configuration rotates about the vertical axis. We begin by requiring

$$\theta_1 = \theta_{1, 0}, \qquad \theta_2 = \theta_{2, 0}, \qquad \phi_1 - \phi_2 = \text{const},$$

so that the shape of the pendulum remains fixed. Consequently, 

$$\dot{\theta}_1 = \dot{\theta}_2 = \ddot{\theta}_1 = \ddot{\theta}_2 = 0, \qquad \dot{\phi}_1 = \dot{\phi}_2 = \omega(t).$$

Substitution of these conditions into the azimuthal EOMs further requires, for nondegenerate rotating solutions, 

$$\dot{\omega} = 0, \qquad \sin(\phi_1 - \phi_2) = 0.$$

Thus the fixed-shape solutions rotate uniformly and separate naturally into two configurations,

$$\phi_1 - \phi_2 = 0$$

for the **aligned** case and

$$\phi_1 - \phi_2 = \pi$$

for the **anti-aligned** case. The remaining equations determine which combinations of $\theta_{1, 0}$, $\theta_{2, 0}$, and $\omega$ can produce these exact double-conical motions. 

These relative equilibria provide convenient reference states for examining the local dynamics of the full system. After constructing examples of both the aligned and anti-aligned conical soutions, we perturb an aligned solution and linearize th equations of motion about it. The resulting normal modes reduce the local motion to independent two-dimensional modal phase planes, making it possible to visualize their vector fields.

In [ ]:
omega = sp.symbols(r'\omega', positive=True)

conical_subs = {
    theta1_dot: 0, 
    theta2_dot: 0,
    theta1_ddot: 0,
    theta2_ddot: 0,
    phi1_dot: omega,
    phi2_dot: omega,
    phi1_ddot: 0,
    phi2_ddot: 0,
}

EL1_conical = sp.trigsimp(EL1.subs(conical_subs))
EL2_conical = sp.trigsimp(EL2.subs(conical_subs))
EL3_conical = sp.trigsimp(EL3.subs(conical_subs))
EL4_conical = sp.trigsimp(EL4.subs(conical_subs))

### **Double-Conical Equilibria Conditions**

After imposing the conditions for fixed-shape, uniform conical motion, the four E-L equations reduce to two independent equations governing the polar angles. These equations contain three unknown quantities,

$$\theta_{1, 0}, \qquad \theta_{2, 0}, \qquad \omega,$$

where $\theta_{1, 0}$ and $\theta_{2, 0}$ specify the fixed configuration of the two links and $\omega$ is their common angular velocity about the vertical axis. The azimuthal separation has already been restricted to either the aligned or anti-aligned configuration. 

Since there are two equilibrium equations but three unknowns, the conical solution is not represented by a single isolated configuration. Instead, one of the equations can first be solved analytically for the required angular velocity, 

$$\omega^2 = \omega^2(\theta_{1, 0}, \theta_{2, 0}).$$

Substituting this expression into the remaining equation eliminates $\omega$, leaving a single relation between the two polar angles,

$$F(\theta_{1, 0}, \theta_{2, 0}) = 0.$$

This equation defines a **family of allowable double-conical configurations** in the $(\theta_{1, 0}, \theta_{2, 0})$ plane. Each point satisfying the relation represents a pair of constant polar angles for which a double-conical relative equilibrium is possible, provided the corresponding value of $\omega^2$ is physically admissible. Once a particular point on this solution curve is selected, its required angular velocity is recovered from the analytic expression for $\omega^2$.

Thus, rather than searching numerically for all three quantities $(\theta_{1, 0}, \theta_{2, 0}, \omega)$ simultaneously, the problem is reduced to finding the zero contour

$$F(\theta_{1, 0}, \theta_{2, 0}) = 0$$

and then calculating $\omega$ for any chosen configuration along that contour. The aligned and anti-aligned cases produce different forms of $F$, and therefore different families of allowable conical configurations. 

In [ ]:
aligned_subs = {
    phi2: phi1
}

EL1_aligned = sp.trigsimp(EL1_conical.subs(aligned_subs))
EL2_aligned = sp.trigsimp(EL2_conical.subs(aligned_subs))
EL3_aligned = sp.trigsimp(EL3_conical.subs(aligned_subs))
EL4_aligned = sp.trigsimp(EL4_conical.subs(aligned_subs))

omega_squared = sp.solve(EL2_aligned, omega**2)[0]

angle_relation = sp.trigsimp(EL1_aligned.subs(omega**2, omega_squared))
angle_relation = sp.factor(angle_relation)

def F_aligned(theta1, theta2, m1=1., m2=1., R1=1., R2=1.):
    return (m1 + m2) * np.sin(theta1) - (((m1 + m2) * R1 * np.sin(theta1) + m2 * R2 * np.sin(theta2)) * np.cos(theta1) * np.tan(theta2)) / (R1 * np.sin(theta1) + R2 * np.sin(theta2))

theta1_vals = np.linspace(0.01, np.pi/2 - 0.01, 200)
theta2_vals = np.linspace(0.01, np.pi/2 - 0.01, 200)

THETA1, THETA2 = np.meshgrid(theta1_vals, theta2_vals)

F_aligned_vals = F_aligned(THETA1, THETA2)

fig, ax = plt.subplots(figsize=(7, 6))

ax.contour(THETA1, THETA2, F_aligned_vals, levels=[0])
ax.set_xlabel(r'$\theta_1$')
ax.set_ylabel(r'$\theta_2$')
ax.set_title('Aligned Double-Conical Equilibria')
ax.grid()
plt.show()

In [ ]:
from scipy.optimize import root_scalar

theta1_o_aligned = 0.8
root = root_scalar(lambda theta2: F_aligned(theta1_o_aligned, theta2), bracket=[0.01, np.pi/2 - 0.01])
theta2_o_aligned = root.root

print(rf"$\theta_{{1,o}}$ = {theta1_o_aligned}")
print(rf"$\theta_{{2,o}}$ = {theta2_o_aligned}")

g_val = 9.8
R1_val = 1.0
R2_val = 1.0

omega_o_aligned = np.sqrt((g_val * np.tan(theta2_o_aligned)) / (R1_val * np.sin(theta1_o_aligned) + R2_val * np.sin(theta2_o_aligned)))
print(rf"$\omega_0$ = {omega_o_aligned}")

In [ ]:
# Create a pendulum animation instance. 
pendulum_anim = AnimationPendulumPlot(phi_vs_t=True, 
                                      state_space=True,
                                      physical_pend=True)

# Common plotting time (generate the full time here then use slices below)
t_start = 0.
t_end = 200.
delta_t = 0.01
pendulum_anim.t_pts_init(t_start, t_end, delta_t)

# Pendulum parameters
m1_param = 1.
m2_param = 1.
R1_param = 1.
R2_param = 1.
g_param = 9.8

# Instantiate a pendulum 
p2 = Pendulum(m1=m1_param, m2=m2_param, R1=R1_param, R2=R2_param, g=g_param)

# Initial conditions specified
theta1_0 = theta1_o_aligned
u1_0 = 0.0
theta2_0 = theta2_o_aligned
u2_0 = 0.0
phi1_0 = 0.0  # -np.pi / 2.
v1_0 = omega_o_aligned
phi2_0 = 0.0
v2_0 = omega_o_aligned

# Add a pendulum to the animation plots; this solves the differential
#  equation for the full t_pts array, generating phi and phi_dot internally. 
pendulum_anim.add_pendulum(p2, theta1_0, u1_0, theta2_0, u2_0, phi1_0, v1_0, phi2_0, v2_0)

In [ ]:
plot_start = 0.   # time to begin plotting
plot_end = 30.    # time to end plotting
pendulum_anim.plot_setup(plot_start, plot_end)

# Start the animation (adjust skip and interval for a smooth plot at a 
#  useful speed)
skip = 2        # skip between time points (in units of delta_t) 
interval = 15.  # time between frames in milliseconds
# pendulum_anim.start_animation(skip, interval)
display(pendulum_anim.start_animation(skip, interval))

### **Aligned Conical Motion**

A representative configuration was selected from the family of aligned double-conical equilibria. For $m_1 = m_2 = 1$, $R_1 = R_2 = 1$, and $g = 9.8$, we chose

$$\theta_{1, 0} = 0.8, \qquad \theta_{2, 0} \approx 0.95136, \qquad \omega_0 \approx 2.99560.$$

The system was initialized with zero polar velocities and equal azimuthal positions and velocities,

$$\dot{\theta}_1(0) = \dot{\theta}_2(0) = 0, \qquad \phi_1(0) = \phi_2(0) = 0, \qquad \dot{\phi}_1(0) = \dot{\phi}_2(0) = \omega_0.$$

This initial condition (IC) places the system directly on the aligned conical relative equilibrium.

**Generalized Coordinates**
The coordinate plot confirms the expected behavior of the conical solution. Both polar angles remain constant at their equilibrium values, 

$$\theta_1(t) = \theta_{1, 0}, \qquad \theta_2(t) = \theta_{2, 0},$$

while the two azimuthal coordinates increase linearly and remain coincident,

$$\phi_1(t) = \phi_2(t) = \omega_0 t.$$

The absense of oscillations or secular drift in either polar angle indicates that the numerically integrated trajectory remains on the derived conical solution. Meanwhile, the linear evolution of the azimuthal coordinates confirms uniform rotation at the predicted angular velocity.

**Physical Motion**
The three-dimensional animation provides the physical interpretation of the same result. Rather than swinging back and forth like a planar pendulu, the two-link configuration retains a fixed shape while rotating continuously about the vertical axis., Since 

$$\theta_1, \theta_2 = \text{constant}, \qquad \phi_1 - \phi_2 = 0,$$

the two links remain in azimuth throughout the motion. Each mass therefore traces a horizontal circular path, and the complete double-pendulum configuration rotates rigidly around the vertical axis. In this sense, the motion is a relative equilbrium: the configuration is not stationary in the laboratory frame, but its shape is stationary in a frame rotating with angular velocity $\omega_0$.

**Coordinate-Velocity State-Space Projections**
The coordinate-velocity projections provide another representation of the same solution. In the polar projections, 

$$(\theta_1, \dot{\theta}_1), \qquad (\theta_2, \dot{\theta}_2),$$

the trajectories remains at a single point,

$$(\theta_{1, 0}, 0), \qquad (\theta_{2, 0}, 0),$$

because neither polar coordinate changes during exact conical motion.

In the azimuthal projections,

$$(\phi_1, \dot{\phi}_1), \qquad (\phi_2, \dot{\phi}_2),$$

the angular velocities remain fixed at $\omega_0$ while the azimuthal coordinates increase continuously. The trajectories therfore move horizontally through these projections at 

$$\dot{\phi}_1 = \dot{\phi}_2 = \omega_0.$$

These plots are projections of the full eight-dimensional state rather than complete phase portraits. Taken together with the coordinate plot and three-dimensional animation, however, they confirm the defining features of the aligned conical solution: constant polar angles, constant and equal azimuthal velocities, fixed relative orientation, and uniform rotation of the entire configuration. 

### **Small Oscillations About the Relative Equilibrium**

The exact conical solution describes a relative equilibrium of the double spherical pendulum: its shape is constant in a frame rotating with angular velocity $\omega_0$. To investigate the dynamics in the neighborhood of this solution, small perturbations are introduced into each of the four generalized coordinates,

$$\theta_1 = \theta_{1, 0} + \epsilon_1(t), \qquad \theta_2 = \theta_{2, 0} + \epsilon_2(t), \\
\phi_1 = \omega_0 t + \eta_1(t), \qquad \phi_2 = \omega_0 t + \eta_2(t),$$ 

where $\epsilon_i$ and $\eta_i$ are assumed to be small. The $\epsilon_i$ describe deviations of the polar angles from their equilibrium values, while the $\eta_i$ describe deviations from the uniform conical rotation. In particualr, $\eta_1 - \eta_2$ allows the perturbed system to depart from exact azimuthal alignment rather than artifically constraining the motion to remain aligned. 

Substituting these expressions into the full nonlinear equations of motion and retaining only terms that are first order in the perturbations gives a linear system of the form 

$$M\ddot{\textbf{q}} + B\dot{\textbf{q}} + K\textbf{q} = 0,$$

where

$$\textbf{q} = \begin{pmatrix} \epsilon_1 \\ \epsilon_2 \\ \eta_1 \\ \eta_2 \end{pmatrix}.$$

Here, $M$, $B$, and $K$ are the coefficient matrices of the perturbation accelerations, velocities, and displacements, respectively. The $B\dot{\textbf{q}}$ term is a velocity-coupling term arising from linearization about the rotating relative equilibrium. It should not be interpreted as dissipative damping.

In [ ]:
O = sp.symbols('O')

epsilon1 = sp.Function(r'epsilon_1')(t)
epsilon2 = sp.Function(r'epsilon_2')(t)
eta1 = sp.Function(r'eta_1')(t)
eta2 = sp.Function(r'eta_2')(t)

perturb_subs = {
    # coordinates
    theta1: theta1_o_aligned + O*epsilon1,
    theta2: theta2_o_aligned + O*epsilon2,
    phi1: omega_o_aligned*t + O*eta1,
    phi2: omega_o_aligned*t + O*eta2,

    # velocities
    theta1_dot: O*sp.diff(epsilon1, t),
    theta2_dot: O*sp.diff(epsilon2, t),
    phi1_dot: omega_o_aligned + O*sp.diff(eta1, t),
    phi2_dot: omega_o_aligned + O*sp.diff(eta2, t),

    # accelerations
    theta1_ddot: O*sp.diff(epsilon1, t, 2),
    theta2_ddot: O*sp.diff(epsilon2, t, 2),
    phi1_ddot: O*sp.diff(eta1, t, 2),
    phi2_ddot: O*sp.diff(eta2, t, 2)
}

EL1_pert = EL1.subs(perturb_subs)
EL1_linear = sp.diff(EL1_pert, O).subs(O, 0)
EL1_linear = sp.trigsimp(sp.simplify(EL1_linear))

EL2_pert = EL2.subs(perturb_subs)
EL2_linear = sp.diff(EL2_pert, O).subs(O, 0)
EL2_linear = sp.trigsimp(sp.simplify(EL2_linear))

EL3_pert = EL3.subs(perturb_subs)
EL3_linear = sp.diff(EL3_pert, O).subs(O, 0)
EL3_linear = sp.trigsimp(sp.simplify(EL3_linear))

EL4_pert = EL4.subs(perturb_subs)
EL4_linear = sp.diff(EL4_pert, O).subs(O, 0)
EL4_linear = sp.trigsimp(sp.simplify(EL4_linear))

**Normal Modes and the Eigenvalue Equation**  

Although the perturbation equations are linear, the four coordinates remain coupled. We therefore seek normal modes for which all perturbation coordinates evolve with the same characteristic time dependence. using the exponential ansatz

$$\mathbf{q}(t) = \mathbf{a}e^{i\omega t},$$

gives 

$$(-\omega^2M + i\omega B + K)\mathbf{a} = 0.$$

A nonzero mode vector $\mathbf{a}$ can exist only for particular values of $i\omega$. In principle, these values could be obtained from the characteristic equation

$$\det(-\omega^2M + i\omega B + K) = 0.$$

Rather than explicitly expanding and solving this determinant, the second-order system was rewritten as a first-order system. Defining the eight-dimensional perturbation state

$$\mathbf{x} = \begin{pmatrix} \mathbf{q} \\ \dot{\mathbf{q}} \end{pmatrix},$$

the equations become

$$\dot{\mathbf{x}} = A\mathbf{x},$$

with

$$A = \begin{pmatrix} 0 & I \\ -M^{-1}K & -M^{-1}B \end{pmatrix}.$$

The normal-mode problem is therefore converted into the standard matrix eigenvalue equation

$$A\mathbf{v} = \lambda\mathbf{v}.$$

Numerically computing the eigenvalues and eigenvectors of $A$ gives both pieces of information needed for the small-oscillation analysis. The eigenvalues determine the characteristic time dependence of each mode, while the eigenvectors determine the relative amplitudes and phases of the original perturbation coordinates within that mode. For a purely oscillatory conjugate pair, 

$$\lambda = \pm i\omega_i,$$

the magnitude of the imaginary part gives the normal-mode frequency $\omega_i$.

In [ ]:
parameter_subs = {
    m1: 1.0,
    m2: 1.0,
    R1: 1.0,
    R2: 1.0,
    g: 9.8
}

E_matrix = sp.Matrix([
    EL1_linear.subs(parameter_subs),
    EL2_linear.subs(parameter_subs),
    EL3_linear.subs(parameter_subs),
    EL4_linear.subs(parameter_subs)
])

q = sp.Matrix([
    epsilon1,
    epsilon2,
    eta1,
    eta2
])

q_dot = sp.Matrix([
    sp.diff(epsilon1, t),
    sp.diff(epsilon2, t),
    sp.diff(eta1, t),
    sp.diff(eta2, t)
])

q_ddot = sp.Matrix([
    sp.diff(epsilon1, t, 2),
    sp.diff(epsilon2, t, 2),
    sp.diff(eta1, t, 2),
    sp.diff(eta2, t, 2)
])

M_matrix = E_matrix.jacobian(q_ddot)
B_matrix = E_matrix.jacobian(q_dot)
K_matrix = E_matrix.jacobian(q)

M_np = np.array(M_matrix.evalf(), dtype=float)
B_np = np.array(B_matrix.evalf(), dtype=float)
K_np = np.array(K_matrix.evalf(), dtype=float)

Z_np = np.zeros((4, 4))
I_np = np.eye(4)

A_np = np.block([
    [Z_np, I_np],
    [-np.linalg.solve(M_np, K_np),
     -np.linalg.solve(M_np, B_np)]
])

eigenvalues_aligned, eigenvectors_aligned = np.linalg.eig(A_np)

for i, eigenvalue in enumerate(eigenvalues_aligned):
    eigenvector = eigenvectors_aligned[:, i]
    print(rf"$\lambda = {eigenvalue}$")
    print(rf"$\omega = {abs(eigenvalue.imag)}$")
    print(f"eigenvector = {eigenvector}\n")

**Reduction from the 8D State Space to 2D Modal Dynamics**  

The linearized DSP still has an eight-dimensional state,

$$(\mathbf{q}, \dot{\mathbf{q}}) = (\epsilon_1, \epsilon_2, \eta_1, \eta_2, \dot{\epsilon}_1, \dot{\epsilon}_2, \dot{\eta}_1, \dot{\eta}_2),$$

so its complete vector field remains impossible to display directly. The normal modes provide a natural reduction. Each oscillatory eigenmode defines its own invariant two-dimensional plane within this eight-dimensional linearized state space. Motion intialized entirely in one mode remains within that modal plane. 

Introducing a normal coordinate $\xi_i$ for a particular oscillatory mode, its dynamics take the simple harmonic

$$\ddot{\xi}_i + \omega_i^2\xi_i = 0.$$

Writing

$$\nu_i = \dot{\xi}_i$$

then gives the two-dimensional first-order system

$$\dot{\xi}_i = \nu_i, \qquad \dot{\nu}_i = -\omega_i^2\xi_i.$$

The corresponding vector field is therefore 

$$\mathbf{F}_i(\xi_i, \nu_i) = \begin{pmatrix} \nu_i \\ -\omega_i^2\xi_i \end{pmatrix}. $$

This is the key visualization step. This is not a projection of the entire nonlinear eight-dimensional DSP onto an arbitrary two-dimensional plane. Instead, linearization idenntifies genuine normal-mode subspaces of the dynamics, and each oscillatory mode can be represented completely by two variables $(\xi_i, \dot{\xi}_i)$. The otherwise unvisualizable local eight-dimensional dynamics can therefore be examined through a collection of two-dimensional modal vector fields.

For the aligned conical configuration, the eigenvalue analysis produces three nonzero oscillatory normal-mode frequencies. The remaining near-zero secctor is associated with continuous rotational symmetry of the system. The three oscillatory modes are consequently visualized separately in their respective $(\xi_i, \dot{\xi}_i)$ phase planes. 

In [ ]:
class VectorFieldViewer:

    def __init__(self, X_lims, Y_lims, omega, colormap = mpl.cm.viridis):
        self.X, self.Y = np.mgrid[
            X_lims[0]:X_lims[1]:X_lims[2],
            Y_lims[0]:Y_lims[1]:Y_lims[2]
        ]
        self.omega = omega
        self.cm = colormap
        # pass

    def setup_fig(self):
        fig = plt.figure()
        ax = fig.add_subplot(111)
        
    def u(self):
        return self.Y
    
    def v(self):
        # return np.ones(self.X.shape)
        return -self.omega**2 * self.X
    
    def grid(self):
        plt.plot(self.X, self.Y, marker=".", color="k", linestyle="none")

    def colors(self):
        l = np.power(self.u(), 2) + np.power(self.v(), 2)
        l = l / np.max(l)
        return self.cm(l).reshape(self.X.size, 4)
    
    def quiver(self, **kwargs):
        plt.quiver(self.X, self.Y, self.u(), self.v(), color=self.colors(), **kwargs)
        plt.xlabel("X")
        plt.ylabel("Y")
        # plt.show()

    def normed_quiver(self, **kwargs):
        U = self.u() / (np.abs(self.u()) + 0.00001)
        V = self.v() / (np.abs(self.v()) + 0.00001)
        plt.quiver(self.X, self.Y, U, V, color=self.colors(), **kwargs)
        plt.xlabel("X")
        plt.ylabel("Y")
        # plt.show()

    def streamplot(self, **kwargs):
        l = np.power(self.u(), 2) + np.power(self.v(), 2)
        l = l / np.max(l)
        colors = self.cm(l.T)
        plt.streamplot( self.X.T, self.Y.T, self.u().T, self.v().T, color=l.T, cmap=self.cm, **kwargs)
        plt.grid()
        plt.xlabel("X")
        plt.ylabel("Y")
        # plt.show()

    def trajectory(self, q_0, q_dot_0, t_pts):
        q, q_dot = self.solve_ode(q_0, q_dot_0, t_pts)
        plt.plot(q, q_dot, color='b')

    def dy_dt(self, t, y):
        q, q_dot = y
        return [q_dot, -self.omega**2 * q]

    def solve_ode(self, phi_0, phi_dot_0, t_pts, abserr=1.0e-8, relerr=1.0e-6):
        y = [phi_0, phi_dot_0]
        phi, phi_dot = solve_ivp(self.dy_dt, (t_pts[0], t_pts[-1]), y, t_eval=t_pts, atol=abserr, rtol=relerr).y
        return phi, phi_dot

In [ ]:
omega1_aligned_pert = abs(eigenvalues_aligned[0].imag)
omega2_aligned_pert = abs(eigenvalues_aligned[4].imag)
omega3_aligned_pert = abs(eigenvalues_aligned[6].imag)

vf1 = VectorFieldViewer(
    X_lims=(-0.1, 0.1, 21j),
    Y_lims=(-1.0, 1.0, 21j),
    omega=omega1_aligned_pert
)
vf2 = VectorFieldViewer(
    X_lims=(-0.1, 0.1, 21j),
    Y_lims=(-1.0, 1.0, 21j),
    omega=omega2_aligned_pert
)
vf3 = VectorFieldViewer(
    X_lims=(-0.1, 0.1, 21j),
    Y_lims=(-1.0, 1.0, 21j),
    omega=omega3_aligned_pert
)

plt.figure()
vf1.streamplot()
plt.title(rf"Mode 1: $\omega_1 = {omega1_aligned_pert:.3f}$")
plt.show()

plt.figure()
vf2.streamplot()
plt.title(rf"Mode 2: $\omega_2 = {omega2_aligned_pert:.3f}$")
plt.show()

plt.figure()
vf3.streamplot()
plt.title(rf"Mode 3: $\omega_3 = {omega3_aligned_pert:.3f}$")
plt.show()

### **Modal Vector Fields**

Although the full linearized double spherical pendulum evolves in an eight-dimensional state space,

$$\mathbf{U} = \begin{pmatrix} \mathbf{q} \\ \dot{\mathbf{q}} \end{pmatrix} \in \mathbb{R}^8,$$

each oscillatory normal mode reduces the linearized dynamics to a single modal coordinate $\xi_i$,

$$\ddot{\xi}_i + \omega_i^2\xi_i = 0.$$

Introducing the modal velocity $\nu_i = \dot{\xi}_i$ gives the two-dimensional system

$$\dot{\xi}_i = \nu_i, \qquad \dot{\nu}_i = -\omega_i^2\xi_i,$$

and therefore the vector field

$$\mathbf{F}_i(\xi_i, \nu_i) = \begin{pmatrix} \nu_i \\ -\omega_i^2\xi_i \end{pmatrix}.$$

The resulting trajectories form closed ellipses centered on the equilibrium $(\xi_i, \nu_i) = (0, 0)$, representing the periodic exchange between modal displacement and modal velocity. Their shape is controlled by the normal frequency through

$$\ddot{\xi}_i = -\omega_i^2\xi_i.$$

Physically, $\omega_i^2$ characterizes the effective restoring strength of the mode relative to its inertia. A larger $\omega_i$ produces a stronger restoring acceleration for a given modal displacement, causing the system to reverse direction more rapidly and attain larger modal velocities. Thus the normal frequency determines the relative scales of modal displacement and modal velocity along each elliptical trajectory. The apparent shape of these ellipses in the vector-field plots also depends on the chosen axis scales, so their visual stretching should not by itself be interpreted as a physical property of the mode. The three modes considered here have frequencies 

$$\omega_1 = 8.929, \qquad \omega_2 = 5.185, \qquad \omega_3 = 4.482 \text{ rad/s}.$$

Most importantly, these plots are not arbitrary two-dimensional projections of the eight-dimensional DSP state space. Each oscillatory eigenmode defines a two-dimensional invariant subspace of the linearized dynamics. Once the motion is restricted to a single mode, its evolution is completely described by the pair $(\xi_i, \dot{\xi}_i)$. The reduction therefore describes the full linearized dynamics within that special modal subspace, rather than discarding six dimensions merely for visualization. The vector field can consequently be interpreted in the same way as the phase portrait of a one-degree-of-freedom harmonic oscillator. 

------

### **Real Form of the Normal Modes**

Although the eigenvalue problem is solved using complex exponentials, the physical perturbations must be real. For an oscillatory eigenvalue $\lambda = i\omega$, the consider the complex normal-mode solution

$$ \mathbf{z}(t) = \mathbf{a}e^{i\omega t}.$$

Unlike the usual small-oscillation problem, the velocity-coupling term $B\dot{\mathbf{q}}$ gives the DSP the complex eigenvalue equation

$$(K - \omega^2M + i\omega B)\mathbf{a} = 0,$$

so the mode vector $\mathbf{a}$ is generally complex. Following Taylor's notation, each component may be written 

$$a_i = \alpha_i e^{i\delta_i}, \qquad \alpha_i = |a_i|,$$

where the different $\delta_i$ encode the phases of the components of the mode. Because the eigenvalue equation in linear and homogeneous, an eigenvector is only defined up to multiplication by a nonzero scalar. In particular, multiplying the entire mode by an arbitrary phase $e^{-i\delta_0}$ gives 

$$a_i = \alpha_i e^{-(\delta_i + \delta_0)}.$$

The corresponding complex motion is therefore

$$z_i(t) = \alpha_i e^{i[\omega t - (\delta_i + \delta_0)]}.$$

Taking the real part gives the physical perturbation,

$$q_i(t) = \alpha_i\cos[\omega t - (\delta_i + \delta_0)].$$

Thus, all coordinates in a normal mode oscillate with the frequency $\omega$. The amplitudes $\alpha_i$ and phase differences $\delta_i - \delta_j$ are determined by the complex eigenvector and describe the shape of the mode, while $\delta_0$ is an arbitrary overall phase that shifts the entire oscillation in time. 

In [ ]:
v1 = eigenvectors_aligned[:, 0]
a1 = v1[:4]
a1_re = np.real(a1)
a1_im = np.imag(a1)

print("omega1 = ", omega1_aligned_pert)
print("a1 = ", a1)
print("Re(a1) = ", a1_re)
print("Im(a1) = ", a1_im)

alpha1 = np.abs(a1)
delta1 = -np.angle(a1)

print(r"$\alpha_1$ = ", alpha1)
print(r"$\delta_1$ = ", delta1)

In [ ]:
delta = 0.0
t_pts = np.linspace(0., 5., 1000)

q_mode1 = np.array([alpha1[j] * np.cos(omega1_aligned_pert * t_pts - delta + delta1[j]) for j in range(4)]).T

plt.figure()
plt.plot(t_pts, q_mode1[:, 0], label=r'$\epsilon_1$')
plt.plot(t_pts, q_mode1[:, 1], label=r'$\epsilon_2$')
plt.plot(t_pts, q_mode1[:, 2], label=r'$\eta_1$')
plt.plot(t_pts, q_mode1[:, 3], label=r'$\eta_2$')
plt.xlabel(r'$t$')
plt.ylabel("Perturbation")
plt.legend()
plt.grid()
plt.show()

The double-conical solutions provide relative equilibria about which the DSP can undergo small oscillations. Writing

$$\theta_i(t) = \theta_{i, 0} + \epsilon_i(t), \qquad \phi_i(t) = \omega_o t + \phi_{i, 0} + \eta_i(t),$$

and linearizing about the equilibrium reduces the perturbative dynamics to a set of normal modes. Within each oscillatory mode, the normal coordinate satisfies

$$\ddot{\xi}_n + \omega_i^2\xi_n = 0,$$

so the perturbation coordinates $\epsilon_i$ and $\eta_i$ inherit harmonic motion with the same modal frequency but generally different amplitudes and relative phases. Consequently, 

$$\theta_i(t) = \theta_{i, 0} + \alpha_{\epsilon_i}\cos(\omega_n t - \delta_{\epsilon_i}),$$

showing that the polar angles oscillate harmonically about their conical equilibrium values. The azimuthal coordinates retain their underlying uniform rotation, 

$$\phi_i(t) = \omega_0 t + \phi_{i, 0} + \alpha_{\eta_i}\cos(\omega_n t - \delta_{\eta_i}),$$

with a small harmonic deviation superimposed upon it.

Physically, the bobs therefore continue to circulate about the vertical axis while periodically wobbling about their unperturbed conical trajectories. Their heights and radial distances from the vertical axis oscillate as the polar angles vary, while the azimuthal perturbations cause small periodic advances and delays relative to uniform rotation. For a general small perturbation, several normal modes may be excited simultaneously, so the resulting motion is a superposition of harmonic normal-mode oscillations about the conical relative equilibrium rather than a single harmonic oscillation.

------

### **Anti-Aligned Conical Motion**

The second family of double-conical solutions corresponds to the anti-aligned configuration. the same fixed-shape conditions used for the aligned case are imposed, 

$$\theta_1 = \theta_{1, 0}, \qquad \theta_2 = \theta_{2, 0}, \qquad \dot{\phi}_1 = \dot{\phi}_2 = \omega,$$

but the two links are now separated azimuthally by 

$$\phi_1 - \phi_2 = \pi$$

rather than $\phi_1 - \phi_2 = 0$. Thus, the two links remain on opposite sides of the vertical axis while the entire configuration rotates uniformly. 

This change in relative orientation changes the coupling between the two links and therefore produces a different equilibrium condition. Eliminating $\omega$ from the anti-aligned equations gives

$$F_{\text{anti}}(\theta_{1, 0}, \theta_{2, 0}) = 0,$$

whose zero contour defines the family of allowable anti-aligned double-conical configurations. As shown below, this family differs substantially from the aligned branch. In the physical region shown, the anti-aligned solutions lie above the line

$$\theta_{2, 0} = \theta_{1, 0},$$

so the second link must have the larger polar angle. This is also consistent with the anti-aligned expression

$$\omega^2 = \frac{g\tan\theta_{2, 0}}{R_2\sin\theta_{2, 0} - R_1\sin\theta_{1, 0}},$$

which requires a positive denominator for the range $0 < \theta_i < \pi/2$. For equal link lengths, this gives $\theta_{2, 0} > \theta_{1, 0}$. 

Unlike the aligned equilibrium curve, the anti-aligned branch also bends back as $\theta_{2, 0}$ increases. Consequently, the relationship between the two equilibrium angles is not necessarily one-to-one. Over part of the branch, a single value of $\theta_{1, 0}$ can correspond to more than one anti-aligned configuration. The relative orientation of the links therefore changes not merely the numerical values of the equilibrium angles, but the overall structure of the family of conical equilibria.

In [ ]:
anti_aligned_subs = {
    phi2: phi1 - sp.pi,
}

EL1_anti_aligned = sp.trigsimp(EL1_conical.subs(anti_aligned_subs))
EL2_anti_aligned = sp.trigsimp(EL2_conical.subs(anti_aligned_subs))
EL3_anti_aligned = sp.trigsimp(EL3_conical.subs(anti_aligned_subs))
EL4_anti_aligned = sp.trigsimp(EL4_conical.subs(anti_aligned_subs))

omega_squared = sp.solve(EL2_anti_aligned, omega**2)[0]

angle_relation = sp.trigsimp(EL1_anti_aligned.subs(omega**2, omega_squared))
angle_relation = sp.factor(angle_relation)

def F_anti_aligned(theta1, theta2, m1=1., m2=1., R1=1., R2=1.):
    return (m1 + m2) * np.sin(theta1) - (((m1 + m2) * R1 * np.sin(theta1) - m2 * R2 * np.sin(theta2)) * np.cos(theta1) * np.tan(theta2)) / (R2 * np.sin(theta2) - R1 * np.sin(theta1))

theta1_vals = np.linspace(0.01, np.pi/2 - 0.01, 200)
theta2_vals = np.linspace(0.01, np.pi/2 - 0.01, 200)

THETA1, THETA2 = np.meshgrid(theta1_vals, theta2_vals)

denominator = (R2_val * np.sin(THETA2) - R1_val * np.sin(THETA1))
with np.errstate(divide='ignore', invalid='ignore'):
    F_anti_aligned_vals = F_anti_aligned(THETA1, THETA2)
F_anti_aligned_vals = np.where(denominator > 0, F_anti_aligned_vals, np.nan)

fig, ax = plt.subplots(figsize=(7, 6))

ax.contour(THETA1, THETA2, F_anti_aligned_vals, levels=[0])
ax.plot(theta1_vals, theta1_vals, '--', label=r'$\theta_2 = \theta_1$')
ax.set_xlabel(r'$\theta_1$')
ax.set_ylabel(r'$\theta_2$')
ax.set_title('Anti-Aligned Double-Conical Equilibria')
ax.grid()
plt.show()

In [ ]:
from scipy.optimize import root_scalar

theta1_o_anti = 0.4
root = root_scalar(lambda theta2: F_anti_aligned(theta1_o_anti, theta2), bracket=[theta1_o_anti + 0.01, np.pi/2 - 0.01])
theta2_o_anti = root.root

print(rf"$\theta_{{1,o}}$ = {theta1_o_anti}")
print(rf"$\theta_{{2,o}}$ = {theta2_o_anti}")

g_val = 9.8
R1_val = 1.0
R2_val = 1.0

omega_o_anti = np.sqrt((g_val * np.tan(theta2_o_anti)) / (R2_val * np.sin(theta2_o_anti) - R1_val * np.sin(theta1_o_anti)))
print(rf"$\omega_0$ = {omega_o_anti}")

In [ ]:
# Create a pendulum animation instance. 
pendulum_anim = AnimationPendulumPlot(phi_vs_t=True, 
                                      state_space=True,
                                      physical_pend=True)

# Common plotting time (generate the full time here then use slices below)
t_start = 0.
t_end = 200.
delta_t = 0.01
pendulum_anim.t_pts_init(t_start, t_end, delta_t)

# Pendulum parameters
m1_param = 1.
m2_param = 1.
R1_param = 1.
R2_param = 1.
g_param = 9.8

# Instantiate a pendulum 
p3 = Pendulum(m1=m1_param, m2=m2_param, R1=R1_param, R2=R2_param, g=g_param)

# Initial conditions specified
theta1_0 = theta1_o_anti
u1_0 = 0.0
theta2_0 = theta2_o_anti
u2_0 = 0.0
phi1_0 = np.pi  # -np.pi / 2.
v1_0 = omega_o_anti
phi2_0 = 0.0
v2_0 = omega_o_anti

# Add a pendulum to the animation plots; this solves the differential
#  equation for the full t_pts array, generating phi and phi_dot internally. 
pendulum_anim.add_pendulum(p3, theta1_0, u1_0, theta2_0, u2_0, phi1_0, v1_0, phi2_0, v2_0)

In [ ]:
plot_start = 0.   # time to begin plotting
plot_end = 30.    # time to end plotting
pendulum_anim.plot_setup(plot_start, plot_end)

# Start the animation (adjust skip and interval for a smooth plot at a 
#  useful speed)
skip = 2        # skip between time points (in units of delta_t) 
interval = 15.  # time between frames in milliseconds
# pendulum_anim.start_animation(skip, interval)
display(pendulum_anim.start_animation(skip, interval))

### **Anti-Aligned Conical Solution**

The numerical solution displays the same characteristic behavior as the aligned relative equilibrium, but with the links maintained at an azimuthal separation of $\pi$. The coordinate plot, $\theta_1$ and $\theta_2$ remain constant while $\phi_1$ and $\phi_2$ increase linearly with the same slope, remaining separated by $\pi$. The animation therefore shows a rigid double-conical configuration rotating uniformly about the vertical axis, with the two links extending in opposite azimuthal directions rather than remaining aligned.

The coordinate-velocity projections provide the corresponding state-space picture. The polar states $(\theta_i, \dot{\theta}_i)$ remain fixed at $(\theta_{i, 0}, 0)$, while the azimuthal states $(\phi_i, \dot{\phi}_i)$ advance at constant $\dot{\phi}_i = \omega_0$. Thus, as in the aligned case, the configuration is stationary in a frame rotating with $\omega_0$. The principal difference is its fixed anti-aligned geometry, 

$$\phi_1 - \phi_2 = \pi$$

rather than $\phi_1 - \phi_2 = 0$.

In [ ]:
O = sp.symbols('O')

epsilon1 = sp.Function(r'epsilon_1')(t)
epsilon2 = sp.Function(r'epsilon_2')(t)
eta1 = sp.Function(r'eta_1')(t)
eta2 = sp.Function(r'eta_2')(t)

perturb_subs_anti = {
    # coordinates
    theta1: theta1_o_anti + O*epsilon1,
    theta2: theta2_o_anti + O*epsilon2,
    phi1: omega_o_anti*t + sp.pi + O*eta1,
    phi2: omega_o_anti*t + O*eta2,

    # velocities
    theta1_dot: O*sp.diff(epsilon1, t),
    theta2_dot: O*sp.diff(epsilon2, t),
    phi1_dot: omega_o_anti + O*sp.diff(eta1, t),
    phi2_dot: omega_o_anti + O*sp.diff(eta2, t),

    # accelerations
    theta1_ddot: O*sp.diff(epsilon1, t, 2),
    theta2_ddot: O*sp.diff(epsilon2, t, 2),
    phi1_ddot: O*sp.diff(eta1, t, 2),
    phi2_ddot: O*sp.diff(eta2, t, 2)
}

EL1_pert_anti = EL1.subs(perturb_subs_anti)
EL1_linear_anti = sp.diff(EL1_pert_anti, O).subs(O, 0)
EL1_linear_anti = sp.trigsimp(sp.simplify(EL1_linear_anti))

EL2_pert_anti = EL2.subs(perturb_subs_anti)
EL2_linear_anti = sp.diff(EL2_pert_anti, O).subs(O, 0)
EL2_linear_anti = sp.trigsimp(sp.simplify(EL2_linear_anti))

EL3_pert_anti = EL3.subs(perturb_subs_anti)
EL3_linear_anti = sp.diff(EL3_pert_anti, O).subs(O, 0)
EL3_linear_anti = sp.trigsimp(sp.simplify(EL3_linear_anti))

EL4_pert_anti = EL4.subs(perturb_subs_anti)
EL4_linear_anti = sp.diff(EL4_pert_anti, O).subs(O, 0)
EL4_linear_anti = sp.trigsimp(sp.simplify(EL4_linear_anti))

In [ ]:
parameter_subs = {
    m1: 1.0,
    m2: 1.0,
    R1: 1.0,
    R2: 1.0,
    g: 9.8
}

E_matrix_anti = sp.Matrix([
    EL1_linear_anti.subs(parameter_subs),
    EL2_linear_anti.subs(parameter_subs),
    EL3_linear_anti.subs(parameter_subs),
    EL4_linear_anti.subs(parameter_subs)
])

q = sp.Matrix([
    epsilon1,
    epsilon2,
    eta1,
    eta2
])

q_dot = sp.Matrix([
    sp.diff(epsilon1, t),
    sp.diff(epsilon2, t),
    sp.diff(eta1, t),
    sp.diff(eta2, t)
])

q_ddot = sp.Matrix([
    sp.diff(epsilon1, t, 2),
    sp.diff(epsilon2, t, 2),
    sp.diff(eta1, t, 2),
    sp.diff(eta2, t, 2)
])

M_matrix_anti = E_matrix_anti.jacobian(q_ddot)
B_matrix_anti = E_matrix_anti.jacobian(q_dot)
K_matrix_anti = E_matrix_anti.jacobian(q)

M_np_anti = np.array(M_matrix_anti.evalf(), dtype=float)
B_np_anti = np.array(B_matrix_anti.evalf(), dtype=float)
K_np_anti = np.array(K_matrix_anti.evalf(), dtype=float)

Z_np = np.zeros((4, 4))
I_np = np.eye(4)

A_np_anti = np.block([
    [Z_np, I_np],
    [-np.linalg.solve(M_np_anti, K_np_anti),
     -np.linalg.solve(M_np_anti, B_np_anti)]
])

eigenvalues_anti, eigenvectors_anti = np.linalg.eig(A_np_anti)

for i, eigenvalue in enumerate(eigenvalues_anti):
    eigenvector = eigenvectors_anti[:, i]
    print(rf"$\lambda = {eigenvalue}$")
    print(rf"$\omega = {abs(eigenvalue.imag)}$")
    print(f"eigenvector = {eigenvector}\n")

In [ ]:
omega1_anti = abs(eigenvalues_anti[0].imag)
omega2_anti = abs(eigenvalues_anti[2].imag)
omega3_anti = abs(eigenvalues_anti[4].imag)

vf1_anti = VectorFieldViewer(
    X_lims=(-0.1, 0.1, 21j),
    Y_lims=(-1.0, 1.0, 21j),
    omega=omega1_anti
)
vf2_anti = VectorFieldViewer(
    X_lims=(-0.1, 0.1, 21j),
    Y_lims=(-1.0, 1.0, 21j),
    omega=omega2_anti
)
vf3_anti = VectorFieldViewer(
    X_lims=(-0.1, 0.1, 21j),
    Y_lims=(-1.0, 1.0, 21j),
    omega=omega3_anti
)

plt.figure()
vf1_anti.streamplot()
plt.title(rf"Anti-Aligned Mode 1: $\omega_1 = {omega1_anti:.3f}$")
plt.show()

plt.figure()
vf2_anti.streamplot()
plt.title(rf"Anti-Aligned Mode 2: $\omega_2 = {omega2_anti:.3f}$")
plt.show()

plt.figure()
vf3_anti.streamplot()
plt.title(rf"Anti-Aligned Mode 3: $\omega_3 = {omega3_anti:.3f}$")
plt.show()

### **Small Perturbations About the Anti-Aligned Equilibrium**

Linearizing about the selected anti-aligned relative equilibrium produces three oscillatory normal modes,

$$\omega_1 = 8.712, \qquad \omega_2 = 7.581, \qquad \omega_3 = 3.511 \text{ rad/s}.$$

As in the aligned analysis, each mode reduces to a two-dimensional harmonic system,

$$\ddot{\xi}_n = -\omega_n^2\xi_n,$$

whose vector field consists of closed elliptical trajectories about the equilibrium. The different values of $\omega_n$ correspond to different effective restoring responses within this particular anti-aligned configuration. 

Thus, small perturbations of the anti-aligned relative equilibrium exhibit the same general normal-mode behavior established previously: the polar angles undergo small harmonic oscillations about their equilibrium values, while small periodic deviations are superimposed on the underlying uniform azimuthal rotation. The bobs consequently continue rotating about the vertical axis while executing small harmonic wobbles about the anti-aligned conical configuration. 

# 3. Exploration of Symmetries and Conserved Quantities

## 3.1 Visualizations of the conserved quantities of the system

The DSP is a conservative system with two continuous symmetries that are immediately visible in its Lagrangian:

1. **Time-translation symmetry:** the Lagrangian of the DSP has no explicit dependence on time,

$$\frac{\partial\mathcal{L}}{\partial t} = 0.$$

The quantity associated with time-translation symmetry is the Hamiltonian,  

$$\mathcal{H} = \sum_i p_i \dot{q}_i - \mathcal{L}, \qquad p_i = \frac{\partial\mathcal{L}}{\partial\dot{q}_i}.$$

Provided the relation between the generalized coordinates and Cartesians is time-independent,

$$\mathbf{r}_\alpha = \mathbf{r}_\alpha(q_1, \cdots, q_n),$$

the Hamiltonian $\mathcal{H}$ is just the total energy, 

$$\mathcal{H} = T + U.$$

A proof of this can be seen in Taylor's Classical Mechanics Section 7.8.

2. **Rotational symmetry about the vertical $z$-axis:** the absolute azimuthal orientation of the system does not affect its dynamics. The Lagrangian may depend on the relative azimuth $\phi_1 - \phi_2$, but it is invariant under the simultaneous rotation

$$\phi_1 \rightarrow \phi_1 + \epsilon, \qquad \phi_2 \rightarrow \phi_2 + \epsilon.$$

By Noether's theorem, this continuous rotational symmetry implies conservation of the total $z$-component of angular momentum,

$$L_z = p_{\phi_1} + p_{\phi_2} = \frac{\partial\mathcal{L}}{\partial\dot{\phi}_1} + \frac{\partial\mathcal{L}}{\partial\dot{\phi}_2}.$$

We first verify these symmetry statements symbolically and then evaluate the resulting conserved quantities along a numerical trajectory.

In [ ]:
q_dot = [theta1_dot, theta2_dot, phi1_dot, phi2_dot]

H = sp.simplify(sum(qd * sp.diff(L, qd) for qd in q_dot) - L)
E = sp.simplify(T + U)

p_phi1 = sp.simplify(sp.diff(L, phi1_dot))
p_phi2 = sp.simplify(sp.diff(L, phi2_dot))

Lz = sp.simplify(p_phi1 + p_phi2)

In [ ]:
conserved_args = (
    theta1, theta1_dot, 
    theta2, theta2_dot, 
    phi1, phi1_dot,
    phi2, phi2_dot,
    m1, m2, R1, R2, g
)

H_num = sp.lambdify(conserved_args, H, "numpy")
Lz_num = sp.lambdify(conserved_args, Lz, "numpy")

In [ ]:
t_vals = pendulum_anim.t_pts

theta1_vals, u1_vals, theta2_vals, u2_vals, phi1_vals, v1_vals, phi2_vals, v2_vals = p1.solve_ode(t_vals, theta1_0, u1_0, theta2_0, u2_0, phi1_0, v1_0, phi2_0, v2_0)

trajectory = (
    theta1_vals, u1_vals,
    theta2_vals, u2_vals,
    phi1_vals, v1_vals,
    phi2_vals, v2_vals
)

parameters = (p1.m1, p1.m2, p1.R1, p1.R2, p1.g)

H_vals = H_num(*trajectory, *parameters)
Lz_vals = Lz_num(*trajectory, *parameters)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

ax[0].plot(t_vals, H_vals)
ax[0].set_ylabel(r"$\mathcal{H}$")
ax[0].set_title(r"Hamiltonian $\mathcal{H}$")
ax[0].grid()

ax[1].plot(t_vals, Lz_vals)
ax[1].set_xlabel(r"$t$")
ax[1].set_ylabel(r"$L_z$")
ax[1].set_title(r"$z$-Angular Momentum $L_z$")
ax[1].grid()

fig.suptitle("Conserved Quantities")
plt.tight_layout()
plt.show()

In [ ]:
H_abs_err = np.abs(H_vals - H_vals[0])
Lz_abs_err = np.abs(Lz_vals - Lz_vals[0])

fig, ax = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

ax[0].plot(t_vals[1:], H_abs_err[1:])
ax[0].set_yscale('log')
ax[0].set_ylabel(r'$|\Delta \mathcal{H}|$')
ax[0].set_title(r'Absolute Error in $\mathcal{H}$')
ax[0].grid()

ax[1].plot(t_vals[1:], Lz_abs_err[1:])
ax[1].set_yscale('log')
ax[1].set_ylabel(r'$|\Delta L_z|$')
ax[1].set_title(r'Absolute Error in $L_z$')
ax[1].grid()

fig.suptitle('Absolute Conservation Error')
plt.tight_layout()
plt.show() 

In [ ]:
H_rel_err = np.abs((H_vals - H_vals[0]) / H_vals[0])
Lz_rel_err = np.abs((Lz_vals - Lz_vals[0]) / Lz_vals[0])

fig, ax = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

ax[0].semilogy(t_vals, H_rel_err)
ax[0].set_ylabel(r"$|\Delta\mathcal{H}/\mathcal{H}_0|$")
ax[0].set_title(r"Relative Error in $\mathcal{H}$")
ax[0].grid()

ax[1].semilogy(t_vals, Lz_rel_err)
ax[1].set_xlabel(r"$t$")
ax[1].set_ylabel(r"$|\Delta L_z/L_{z,0}|$")
ax[1].set_title(r"Relative Error in $L_z$")
ax[1].grid()

fig.suptitle("Conservation Error")
plt.tight_layout()
plt.show()

### **Numerical Verification of Conserved Quantities**

The time-translation and axial rotational symmetries of the double spherical pendulum imply, through Noether's theorem, conservation of the Hamiltonian $\mathcal{H}$ and total $z$-component of angular momentum $L_z$, respectively. Therefore, along an exact solution to EOMs,

$$\mathcal{H}(t) = \mathcal{H}(0), \qquad L_z(t) = L_z(0).$$

The figures above numerically verify these conservation laws along a nonlinear trajectory of the system. Although the plots of $\mathcal{H}$ and $L_z$ appear to exhibit a gradual increase, the axes are displayed using offsets around their nearly constant values. The actual variations are extremely small: the absolute deviations

$$|\Delta\mathcal{H}(t)| = |\mathcal{H}(t) - \mathcal{H}(0)|, \qquad |\Delta L_z(t)| = |L_z(t) - L_z(0)|$$

remain approximately on the order of $10^{-10}$ and $10^{-11}$, respectively, over the integration interval. 

To account for the different magnitudes of the conserved quantities, the relative deviations are also computed as 

$$\epsilon_{\mathcal{H}}(t) = \frac{|\mathcal{H}(t) - \mathcal{H}(0)|}{|\mathcal{H}(0)|}, \qquad \epsilon_{L_z}(t) = \frac{|L_z(t) - L_z(0)|}{|L_z(0)|}.$$

Both remain on the order of $10^{-11}$ or smaller throughout the simulation. The small long-term drift therefore represents accumulated numerical integration error rather than physical evolution of $\mathcal{H}$ or $L_z$. Thus the numerical results verify to high precision the conservation laws predicted by the time-translation and axial rotational symmetries of the Lagrangian through Noether's theorem.

## 3.2 Whether or not the system exhibits (could exhibit) chaos?

Although the DSP conserves both its total energy and $z$-ccomponent of angular momentum, these conservation laws doe not imply that its motion must be regular. For sufficiently nonlinear motion, deterministic systems may exhibit chaos, characterized by complex phase-space dynamics and strong sensitivity to initial conditions. 

The small-oscillation analysis considered previously provides a natural starting point for exploring this behavior. Near a stable equilibrium, sufficiently small perturbations are well described by the linearlized equations of motion and their corresponding normal modes. As the amplitude of the perturbation is increased, however, nonlinear terms become increasingly important and the small-oscillation approximation eventually breaks down. 

To investigate this behavior, consider a one-parameter family of ICs obtained by perturbing the relative equilibrium along one of the oscillatory normal-mode directions found from the linearized system. Using the full state-space vector $\mathbf{U}(t)$ introduced previously, the initial state is written as

$$\mathbf{U}(0; A) = \mathbf{U}_{\text{eq}} + A\Delta\mathbf{U},$$

where $\mathbf{U}_{\text{eq}}$ represents the relative-equilibrium state, $\Delta\mathbf{U}$ specifies a fixed normal-mode perturbation direction, and $A$ controls the magnitude of the perturbation. By gradually increasing $A$ while evolving each IC according to the full nonlinear EOMs, the resulting trajectories can be compared through their coordinate evolution and projections of state space. This allows the dynamics to be examined as the system moves away from the small-amplitude regime and increasingly nonlinear effects become important. 

In [ ]:
# Relative equilibrium state
U_eq = np.array([theta1_o_aligned, 0., theta2_o_aligned, 0., 0., omega_o_aligned, 0., omega_o_aligned])

lambda1 = eigenvalues_aligned[0]
mode1 = eigenvectors_aligned[:, 0]

def mode_initial_conditions(A):
    Delta_U = np.array([
        np.real(mode1[0]), # epsilon_1
        np.real(mode1[4]), # epsilon_1_dot
        np.real(mode1[1]), # epsilon_2
        np.real(mode1[5]), # epsilon_2_dot
        np.real(mode1[2]), # eta_1
        np.real(mode1[6]), # eta_1_dot
        np.real(mode1[3]), # eta_2
        np.real(mode1[7])  # eta_2_dot
    ]) 
    
    return U_eq + A * Delta_U

# Small perturbation amplitude
A = 0.01
U0 = mode_initial_conditions(A)
theta1_0, u1_0, theta2_0, u2_0, phi1_0, v1_0, phi2_0, v2_0 = U0

# Use a relatively short interval for the linear/nonlinear comparison
t_test = np.linspace(0., 5., 2001)
theta1_nl, u1_nl, theta2_nl, u2_nl, phi1_nl, v1_nl, phi2_nl, v2_nl = p1.solve_ode(t_test, theta1_0, u1_0, theta2_0, u2_0, phi1_0, v1_0, phi2_0, v2_0)

epsilon1_nl = theta1_nl - theta1_o_aligned
epsilon2_nl = theta2_nl - theta2_o_aligned
eta1_nl = phi1_nl - omega_o_aligned * t_test
eta2_nl = phi2_nl - omega_o_aligned * t_test

delta_linear = A * np.real(mode1[:, np.newaxis] * np.exp(lambda1 * t_test)[np.newaxis, :])

epsilon1_linear = delta_linear[0]
epsilon2_linear = delta_linear[1]
eta1_linear = delta_linear[2]
eta2_linear = delta_linear[3]

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)

axes[0, 0].plot(t_test, epsilon1_nl, label='Nonlinear')
axes[0, 0].plot(t_test, epsilon1_linear, '--', label='Linear')
axes[0, 0].set_ylabel(r'$\epsilon_1$')
axes[0, 0].legend()
axes[0, 0].grid()

axes[0, 1].plot(t_test, epsilon2_nl, label='Nonlinear')
axes[0, 1].plot(t_test, epsilon2_linear, '--', label='Linear')
axes[0, 1].set_ylabel(r'$\epsilon_2$')
axes[0, 1].legend()
axes[0, 1].grid()

axes[1, 0].plot(t_test, eta1_nl, label='Nonlinear')
axes[1, 0].plot(t_test, eta1_linear, '--', label='Linear')
axes[1, 0].set_ylabel(r'$\eta_1$')
axes[1, 0].legend()
axes[1, 0].grid()

axes[1, 1].plot(t_test, eta2_nl, label='Nonlinear')
axes[1, 1].plot(t_test, eta2_linear, '--', label='Linear')
axes[1, 1].set_ylabel(r'$\eta_2$')
axes[1, 1].legend()
axes[1, 1].grid()

fig.suptitle(rf'Linear vs. Nonlinear Dynamics Near Relative Equilibrium ($A={A}$)')
plt.tight_layout()
plt.show()

For a small perturbation amplitude, $A = 0.01$, the solution obtained from the linearized equations closely approximates the solution of the full nonlinear EOMs. The near-perfect agreement across the perturbation coordinates $\epsilon_1$, $\epsilon_2$, $\eta_1$, and $\eta_2$ confirms that the normal-mode description accurately captures the dynamics sufficiently close to the relative equilibrium. As $A$ is increased, higher-order nonlinear terms that are neglected in the linearization are expected to become increasingly significant, causing the linear approximation to gradually break down.

In [ ]:
# Labels for individual plot axes
theta1_vs_time_labels = (r'$t$', r'$\theta_1(t)$')
theta1_dot_vs_time_labels = (r'$t$', r'$d\theta_1/dt(t)$')
theta1_state_space_labels = (r'$\theta_1$', r'$d\theta_1/dt$')
theta2_vs_time_labels = (r'$t$', r'$\theta_2(t)$')
theta2_dot_vs_time_labels = (r'$t$', r'$d\theta_2/dt(t)$')
theta2_state_space_labels = (r'$\theta_2$', r'$d\theta_2/dt$')

# Common plotting time (generate the full time then use slices)
t_start = 0.
t_end = 1000.
delta_t = 0.01

t_pts = np.arange(t_start, t_end + delta_t, delta_t)  

In [ ]:
# initial conditions specified
A = 0.01
U0 = mode_initial_conditions(A)
theta1_0, u1_0, theta2_0, u2_0, phi1_0, v1_0, phi2_0, v2_0 = U0
theta1_vals, u1_vals, theta2_vals, u2_vals, phi1_vals, v1_vals, phi2_vals, v2_vals = p1.solve_ode(t_pts, *U0)

# Change the common font size
font_size = 14
plt.rcParams.update({'font.size': font_size})

# start the plot!
fig = plt.figure(figsize=(15,5))
overall_title = (rf'Perturbation Amplitude $A={A}$') 
fig.suptitle(overall_title, va='baseline')
    
# first plot: plot from t=0 to t=15   
ax_a = fig.add_subplot(1,3,1)                  

start, stop = start_stop_indices(t_pts, 0., 30.)    
plot_y_vs_x(t_pts[start : stop], theta1_vals[start : stop], 
            axis_labels=theta1_vs_time_labels, 
            color='blue',
            label=None, 
            title=rf'$\theta_1(t)$', 
            ax=ax_a)    
                              
# second plot: state space plot from t=0 to t=15   
ax_b = fig.add_subplot(1,3,2)                  

start, stop = start_stop_indices(t_pts, 30., 60.)    
plot_y_vs_x(theta1_vals[start : stop], u1_vals[start : stop], 
            axis_labels=theta1_state_space_labels, 
            color='blue',
            label=None, 
            title=rf'$30 \leq t \leq 60$', 
            ax=ax_b)    

# third plot: state space plot from t= to t=12   
ax_c = fig.add_subplot(1,3,3)                  

start, stop = start_stop_indices(t_pts, 10., 1000.)    
plot_y_vs_x(theta1_vals[start : stop], u1_vals[start : stop], 
            axis_labels=theta1_state_space_labels, 
            color='blue',
            label=None, 
            title=rf'$10 \leq t \leq 1000$', 
            ax=ax_c)    

fig.tight_layout()
plt.show()

In [ ]:
# Labels for individual plot axes
theta1_vs_time_labels = (r'$t$', r'$\theta_1(t)$')
Delta_theta1_vs_time_labels = (r'$t$', r'$\Delta\theta_1(t)$')
theta1_dot_vs_time_labels = (r'$t$', r'$d\theta_1/dt(t)$')
theta1_state_space_labels = (r'$\theta_1$', r'$d\theta_1/dt$')

# Common plotting time (generate the full time then use slices)
t_start = 0.
t_end = 100.
delta_t = 0.01

t_pts = np.arange(t_start, t_end + delta_t, delta_t)

In [ ]:
# make a plot of Delta phi for same pendulum but two different initial conds
A = 0.01
U0_1 = mode_initial_conditions(A)
theta1_1, u1_1, theta2_1, u2_1, phi1_1, v1_1, phi2_1, v2_1 = p1.solve_ode(t_pts, *U0_1)

U0_2 = U0_1.copy()
delta_0 = -1.e-6
U0_2[0] += delta_0   # .000000001 radian lower
phi_dot_0 = 0.0
theta1_2, u1_2, theta2_2, u2_2, phi1_2, v1_2, phi2_2, v2_2 = p1.solve_ode(t_pts, *U0_2)

U_1 = np.array([theta1_1, u1_1, theta2_1, u2_1, phi1_1, v1_1, phi2_1, v2_1])
U_2 = np.array([theta1_2, u1_2, theta2_2, u2_2, phi1_2, v1_2, phi2_2, v2_2])

# Calculate the absolute value of \phi_2 - \phi_1
Delta_U = U_2 - U_1
Delta_U[4] = (Delta_U[4] + np.pi) % (2.*np.pi) - np.pi
Delta_U[6] = (Delta_U[6] + np.pi) % (2.*np.pi) - np.pi
Delta_U = np.linalg.norm(Delta_U, axis=0)

# Change the common font size
font_size = 14
plt.rcParams.update({'font.size': font_size})

# start the plot!
fig = plt.figure(figsize=(10,10))
overall_title = rf'Sensitivity to Initial Conditions: $A={A}$, ' # + \
                # rf'$\delta_0={delta_0:.0e}$' 
fig.suptitle(overall_title, va='baseline')
    
# two plot: plot from t=0 to t=8 and another from t=0 to t=100 
ax_a = fig.add_subplot(2,1,1)                  

start, stop = start_stop_indices(t_pts, 0., 16.)
ax_a.semilogy(t_pts[start : stop], Delta_U[start : stop], 
            color='blue', label=None)    
ax_a.set_xlabel(r'$t$')
ax_a.set_ylabel(r'$|\Delta\mathbf{U}|$')

ax_b = fig.add_subplot(2,1,2)                  

start, stop = start_stop_indices(t_pts, 0., 40.)
plot_y_vs_x(t_pts[start : stop], Delta_U[start : stop], 
            color='blue', label=None, semilogy=True)    
ax_b.set_xlabel(r'$t$')
ax_b.set_ylabel(r'$|\Delta\mathbf{U}|$')
                              
fig.tight_layout()
plt.show()

In [ ]:
# initial conditions specified
A = 1.0
U0 = mode_initial_conditions(A)
theta1_0, u1_0, theta2_0, u2_0, phi1_0, v1_0, phi2_0, v2_0 = U0
theta1_vals, u1_vals, theta2_vals, u2_vals, phi1_vals, v1_vals, phi2_vals, v2_vals = p1.solve_ode(t_pts, *U0)

# Change the common font size
font_size = 14
plt.rcParams.update({'font.size': font_size})

# start the plot!
fig = plt.figure(figsize=(15,5))
overall_title = (rf'Perturbation Amplitude $A={A}$') 
fig.suptitle(overall_title, va='baseline')
    
# first plot: plot from t=0 to t=15   
ax_a = fig.add_subplot(1,3,1)                  

start, stop = start_stop_indices(t_pts, 0., 30.)    
plot_y_vs_x(t_pts[start : stop], theta1_vals[start : stop], 
            axis_labels=theta1_vs_time_labels, 
            color='blue',
            label=None, 
            title=rf'$\theta_1(t)$', 
            ax=ax_a)    
                              
# second plot: state space plot from t=0 to t=15   
ax_b = fig.add_subplot(1,3,2)                  

start, stop = start_stop_indices(t_pts, 30., 60.)    
plot_y_vs_x(theta1_vals[start : stop], u1_vals[start : stop], 
            axis_labels=theta1_state_space_labels, 
            color='blue',
            label=None, 
            title=rf'$30 \leq t \leq 60$', 
            ax=ax_b)    

# third plot: state space plot from t= to t=12   
ax_c = fig.add_subplot(1,3,3)                  

start, stop = start_stop_indices(t_pts, 10., 1000.)    
plot_y_vs_x(theta1_vals[start : stop], u1_vals[start : stop], 
            axis_labels=theta1_state_space_labels, 
            color='blue',
            label=None, 
            title=rf'$10 \leq t \leq 1000$', 
            ax=ax_c)    

fig.tight_layout()
plt.show()

In [ ]:
# make a plot of Delta phi for same pendulum but two different initial conds
A = 1.0
U0_1 = mode_initial_conditions(A)
theta1_1, u1_1, theta2_1, u2_1, phi1_1, v1_1, phi2_1, v2_1 = p1.solve_ode(t_pts, *U0_1)

U0_2 = U0_1.copy()
delta_0 = -1.e-6
U0_2[0] += delta_0   # .000000001 radian lower
phi_dot_0 = 0.0
theta1_2, u1_2, theta2_2, u2_2, phi1_2, v1_2, phi2_2, v2_2 = p1.solve_ode(t_pts, *U0_2)

U_1 = np.array([theta1_1, u1_1, theta2_1, u2_1, phi1_1, v1_1, phi2_1, v2_1])
U_2 = np.array([theta1_2, u1_2, theta2_2, u2_2, phi1_2, v1_2, phi2_2, v2_2])

# Calculate the absolute value of \phi_2 - \phi_1
Delta_U = U_2 - U_1
Delta_U[4] = (Delta_U[4] + np.pi) % (2.*np.pi) - np.pi
Delta_U[6] = (Delta_U[6] + np.pi) % (2.*np.pi) - np.pi
Delta_U = np.linalg.norm(Delta_U, axis=0)

# Change the common font size
font_size = 14
plt.rcParams.update({'font.size': font_size})

# start the plot!
fig = plt.figure(figsize=(10,10))
overall_title = rf'Sensitivity to Initial Conditions: $A={A}$, ' # + \
                # rf'$\delta_0={delta_0:.0e}$' 
fig.suptitle(overall_title, va='baseline')
    
# two plot: plot from t=0 to t=8 and another from t=0 to t=100 
ax_a = fig.add_subplot(2,1,1)                  

start, stop = start_stop_indices(t_pts, 0., 10.)
ax_a.semilogy(t_pts[start : stop], Delta_U[start : stop], 
            color='blue', label=None)    
ax_a.set_xlabel(r'$t$')
ax_a.set_ylabel(r'$|\Delta\mathbf{U}|$')

ax_b = fig.add_subplot(2,1,2)                  

start, stop = start_stop_indices(t_pts, 0., 100.)
plot_y_vs_x(t_pts[start : stop], Delta_U[start : stop], 
            color='blue', label=None, semilogy=True)    
ax_b.set_xlabel(r'$t$')
ax_b.set_ylabel(r'$|\Delta\mathbf{U}|$')
                              
fig.tight_layout()
plt.show()

In [ ]:
# initial conditions specified
A = 2.0
U0 = mode_initial_conditions(A)
theta1_0, u1_0, theta2_0, u2_0, phi1_0, v1_0, phi2_0, v2_0 = U0
theta1_vals, u1_vals, theta2_vals, u2_vals, phi1_vals, v1_vals, phi2_vals, v2_vals = p1.solve_ode(t_pts, *U0)

# Change the common font size
font_size = 14
plt.rcParams.update({'font.size': font_size})

# start the plot!
fig = plt.figure(figsize=(15,5))
overall_title = (rf'Perturbation Amplitude $A={A}$') 
fig.suptitle(overall_title, va='baseline')
    
# first plot: plot from t=0 to t=15   
ax_a = fig.add_subplot(1,3,1)                  

start, stop = start_stop_indices(t_pts, 0., 30.)    
plot_y_vs_x(t_pts[start : stop], theta1_vals[start : stop], 
            axis_labels=theta1_vs_time_labels, 
            color='blue',
            label=None, 
            title=rf'$\theta_1(t)$', 
            ax=ax_a)    
                              
# second plot: state space plot from t=0 to t=15   
ax_b = fig.add_subplot(1,3,2)                  

start, stop = start_stop_indices(t_pts, 30., 60.)    
plot_y_vs_x(theta1_vals[start : stop], u1_vals[start : stop], 
            axis_labels=theta1_state_space_labels, 
            color='blue',
            label=None, 
            title=rf'$30 \leq t \leq 60$', 
            ax=ax_b)    

# third plot: state space plot from t= to t=12   
ax_c = fig.add_subplot(1,3,3)                  

start, stop = start_stop_indices(t_pts, 10., 1000.)    
plot_y_vs_x(theta1_vals[start : stop], u1_vals[start : stop], 
            axis_labels=theta1_state_space_labels, 
            color='blue',
            label=None, 
            title=rf'$10 \leq t \leq 1000$', 
            ax=ax_c)    

fig.tight_layout()
plt.show()

In [ ]:
# make a plot of Delta phi for same pendulum but two different initial conds
A = 2.0
U0_1 = mode_initial_conditions(A)
theta1_1, u1_1, theta2_1, u2_1, phi1_1, v1_1, phi2_1, v2_1 = p1.solve_ode(t_pts, *U0_1)

U0_2 = U0_1.copy()
delta_0 = -1.e-6
U0_2[0] += delta_0   # .000000001 radian lower
phi_dot_0 = 0.0
theta1_2, u1_2, theta2_2, u2_2, phi1_2, v1_2, phi2_2, v2_2 = p1.solve_ode(t_pts, *U0_2)

U_1 = np.array([theta1_1, u1_1, theta2_1, u2_1, phi1_1, v1_1, phi2_1, v2_1])
U_2 = np.array([theta1_2, u1_2, theta2_2, u2_2, phi1_2, v1_2, phi2_2, v2_2])

# Calculate the absolute value of \phi_2 - \phi_1
Delta_U = U_2 - U_1
Delta_U[4] = (Delta_U[4] + np.pi) % (2.*np.pi) - np.pi
Delta_U[6] = (Delta_U[6] + np.pi) % (2.*np.pi) - np.pi
Delta_U = np.linalg.norm(Delta_U, axis=0)

# Change the common font size
font_size = 14
plt.rcParams.update({'font.size': font_size})

# start the plot!
fig = plt.figure(figsize=(10,10))
overall_title = rf'Sensitivity to Initial Conditions: $A={A}$, ' + \
                rf'$\delta_0={delta_0:.0e}$' 
fig.suptitle(overall_title, va='baseline')
    
# two plot: plot from t=0 to t=8 and another from t=0 to t=100 
ax_a = fig.add_subplot(2,1,1)                  

start, stop = start_stop_indices(t_pts, 0., 10.)
ax_a.semilogy(t_pts[start : stop], Delta_U[start : stop], 
            color='blue', label=None)    
ax_a.set_xlabel(r'$t$')
ax_a.set_ylabel(r'$|\Delta\mathbf{U}|$')

ax_b = fig.add_subplot(2,1,2)                  

start, stop = start_stop_indices(t_pts, 0., 100.)
plot_y_vs_x(t_pts[start : stop], Delta_U[start : stop], 
            color='blue', label=None, semilogy=True)    
ax_b.set_xlabel(r'$t$')
ax_b.set_ylabel(r'$|\Delta\mathbf{U}|$')
                              
fig.tight_layout()
plt.show()

In [ ]:
# initial conditions specified
A = 4.0
U0 = mode_initial_conditions(A)
theta1_0, u1_0, theta2_0, u2_0, phi1_0, v1_0, phi2_0, v2_0 = U0
theta1_vals, u1_vals, theta2_vals, u2_vals, phi1_vals, v1_vals, phi2_vals, v2_vals = p1.solve_ode(t_pts, *U0)

# Change the common font size
font_size = 14
plt.rcParams.update({'font.size': font_size})

# start the plot!
fig = plt.figure(figsize=(15,5))
overall_title = (rf'Perturbation Amplitude $A={A}$') 
fig.suptitle(overall_title, va='baseline')
    
# first plot: plot from t=0 to t=15   
ax_a = fig.add_subplot(1,3,1)                  

start, stop = start_stop_indices(t_pts, 0., 30.)    
plot_y_vs_x(t_pts[start : stop], theta1_vals[start : stop], 
            axis_labels=theta1_vs_time_labels, 
            color='blue',
            label=None, 
            title=rf'$\theta_1(t)$', 
            ax=ax_a)    
                              
# second plot: state space plot from t=0 to t=15   
ax_b = fig.add_subplot(1,3,2)                  

start, stop = start_stop_indices(t_pts, 30., 60.)    
plot_y_vs_x(theta1_vals[start : stop], u1_vals[start : stop], 
            axis_labels=theta1_state_space_labels, 
            color='blue',
            label=None, 
            title=rf'$30 \leq t \leq 60$', 
            ax=ax_b)    

# third plot: state space plot from t= to t=12   
ax_c = fig.add_subplot(1,3,3)                  

start, stop = start_stop_indices(t_pts, 10., 1000.)    
plot_y_vs_x(theta1_vals[start : stop], u1_vals[start : stop], 
            axis_labels=theta1_state_space_labels, 
            color='blue',
            label=None, 
            title=rf'$10 \leq t \leq 1000$', 
            ax=ax_c)    

fig.tight_layout()
plt.show()

In [ ]:
# make a plot of Delta phi for same pendulum but two different initial conds
A = 4.0
U0_1 = mode_initial_conditions(A)
theta1_1, u1_1, theta2_1, u2_1, phi1_1, v1_1, phi2_1, v2_1 = p1.solve_ode(t_pts, *U0_1)

U0_2 = U0_1.copy()
delta_0 = -1.e-6
U0_2[0] += delta_0   # .000000001 radian lower
phi_dot_0 = 0.0
theta1_2, u1_2, theta2_2, u2_2, phi1_2, v1_2, phi2_2, v2_2 = p1.solve_ode(t_pts, *U0_2)

U_1 = np.array([theta1_1, u1_1, theta2_1, u2_1, phi1_1, v1_1, phi2_1, v2_1])
U_2 = np.array([theta1_2, u1_2, theta2_2, u2_2, phi1_2, v1_2, phi2_2, v2_2])

# Calculate the absolute value of \phi_2 - \phi_1
Delta_U = U_2 - U_1
Delta_U[4] = (Delta_U[4] + np.pi) % (2.*np.pi) - np.pi
Delta_U[6] = (Delta_U[6] + np.pi) % (2.*np.pi) - np.pi
Delta_U = np.linalg.norm(Delta_U, axis=0)

# Change the common font size
font_size = 14
plt.rcParams.update({'font.size': font_size})

# start the plot!
fig = plt.figure(figsize=(10,10))
overall_title = rf'Sensitivity to Initial Conditions: $A={A}$, ' # + \
                # rf'$\delta_0={delta_0:.0e}$' 
fig.suptitle(overall_title, va='baseline')
    
# two plot: plot from t=0 to t=8 and another from t=0 to t=100 
ax_a = fig.add_subplot(2,1,1)                  

start, stop = start_stop_indices(t_pts, 0., 10.)
ax_a.semilogy(t_pts[start : stop], Delta_U[start : stop], 
            color='blue', label=None)    
ax_a.set_xlabel(r'$t$')
ax_a.set_ylabel(r'$|\Delta\mathbf{U}|$')

ax_b = fig.add_subplot(2,1,2)                  

start, stop = start_stop_indices(t_pts, 0., 100.)
plot_y_vs_x(t_pts[start : stop], Delta_U[start : stop], 
            color='blue', label=None, semilogy=True)    
ax_b.set_xlabel(r'$t$')
ax_b.set_ylabel(r'$|\Delta\mathbf{U}|$')
                              
fig.tight_layout()
plt.show()

In [ ]:
# initial conditions specified
A = 8.0
U0 = mode_initial_conditions(A)
theta1_0, u1_0, theta2_0, u2_0, phi1_0, v1_0, phi2_0, v2_0 = U0
theta1_vals, u1_vals, theta2_vals, u2_vals, phi1_vals, v1_vals, phi2_vals, v2_vals = p1.solve_ode(t_pts, *U0)

# Change the common font size
font_size = 14
plt.rcParams.update({'font.size': font_size})

# start the plot!
fig = plt.figure(figsize=(15,5))
overall_title = (rf'Perturbation Amplitude $A={A}$') 
fig.suptitle(overall_title, va='baseline')
    
# first plot: plot from t=0 to t=15   
ax_a = fig.add_subplot(1,3,1)                  

start, stop = start_stop_indices(t_pts, 0., 30.)    
plot_y_vs_x(t_pts[start : stop], theta1_vals[start : stop], 
            axis_labels=theta1_vs_time_labels, 
            color='blue',
            label=None, 
            title=rf'$\theta_1(t)$', 
            ax=ax_a)    
                              
# second plot: state space plot from t=0 to t=15   
ax_b = fig.add_subplot(1,3,2)                  

start, stop = start_stop_indices(t_pts, 30., 60.)    
plot_y_vs_x(theta1_vals[start : stop], u1_vals[start : stop], 
            axis_labels=theta1_state_space_labels, 
            color='blue',
            label=None, 
            title=rf'$30 \leq t \leq 60$', 
            ax=ax_b)    

# third plot: state space plot from t= to t=12   
ax_c = fig.add_subplot(1,3,3)                  

start, stop = start_stop_indices(t_pts, 10., 1000.)    
plot_y_vs_x(theta1_vals[start : stop], u1_vals[start : stop], 
            axis_labels=theta1_state_space_labels, 
            color='blue',
            label=None, 
            title=rf'$10 \leq t \leq 1000$', 
            ax=ax_c)    

fig.tight_layout()
plt.show()

In [ ]:
# make a plot of Delta phi for same pendulum but two different initial conds
A = 8.0
U0_1 = mode_initial_conditions(A)
theta1_1, u1_1, theta2_1, u2_1, phi1_1, v1_1, phi2_1, v2_1 = p1.solve_ode(t_pts, *U0_1)

U0_2 = U0_1.copy()
delta_0 = -1.e-6
U0_2[0] += delta_0   # .000000001 radian lower
phi_dot_0 = 0.0
theta1_2, u1_2, theta2_2, u2_2, phi1_2, v1_2, phi2_2, v2_2 = p1.solve_ode(t_pts, *U0_2)

U_1 = np.array([theta1_1, u1_1, theta2_1, u2_1, phi1_1, v1_1, phi2_1, v2_1])
U_2 = np.array([theta1_2, u1_2, theta2_2, u2_2, phi1_2, v1_2, phi2_2, v2_2])

# Calculate the absolute value of \phi_2 - \phi_1
Delta_U = U_2 - U_1
Delta_U[4] = (Delta_U[4] + np.pi) % (2.*np.pi) - np.pi
Delta_U[6] = (Delta_U[6] + np.pi) % (2.*np.pi) - np.pi
Delta_U = np.linalg.norm(Delta_U, axis=0)

# Change the common font size
font_size = 14
plt.rcParams.update({'font.size': font_size})

# start the plot!
fig = plt.figure(figsize=(10,10))
overall_title = rf'Sensitivity to Initial Conditions: $A={A}$, ' # + \
                # rf'$\delta_0={delta_0:.0e}$' 
fig.suptitle(overall_title, va='baseline')
    
# two plot: plot from t=0 to t=8 and another from t=0 to t=100 
ax_a = fig.add_subplot(2,1,1)                  

start, stop = start_stop_indices(t_pts, 0., 10.)
ax_a.semilogy(t_pts[start : stop], Delta_U[start : stop], 
            color='blue', label=None)    
ax_a.set_xlabel(r'$t$')
ax_a.set_ylabel(r'$|\Delta\mathbf{U}|$')

ax_b = fig.add_subplot(2,1,2)                  

start, stop = start_stop_indices(t_pts, 0., 100.)
plot_y_vs_x(t_pts[start : stop], Delta_U[start : stop], 
            color='blue', label=None, semilogy=True)    
ax_b.set_xlabel(r'$t$')
ax_b.set_ylabel(r'$|\Delta\mathbf{U}|$')
                              
fig.tight_layout()
plt.show()

In [ ]:
# initial conditions specified
A = 16.0
U0 = mode_initial_conditions(A)
theta1_0, u1_0, theta2_0, u2_0, phi1_0, v1_0, phi2_0, v2_0 = U0
theta1_vals, u1_vals, theta2_vals, u2_vals, phi1_vals, v1_vals, phi2_vals, v2_vals = p1.solve_ode(t_pts, *U0)

# Change the common font size
font_size = 14
plt.rcParams.update({'font.size': font_size})

# start the plot!
fig = plt.figure(figsize=(15,5))
overall_title = (rf'Perturbation Amplitude $A={A}$') 
fig.suptitle(overall_title, va='baseline')
    
# first plot: plot from t=0 to t=15   
ax_a = fig.add_subplot(1,3,1)                  

start, stop = start_stop_indices(t_pts, 0., 30.)    
plot_y_vs_x(t_pts[start : stop], theta1_vals[start : stop], 
            axis_labels=theta1_vs_time_labels, 
            color='blue',
            label=None, 
            title=rf'$\theta_1(t)$', 
            ax=ax_a)    
                              
# second plot: state space plot from t=0 to t=15   
ax_b = fig.add_subplot(1,3,2)                  

start, stop = start_stop_indices(t_pts, 30., 60.)    
plot_y_vs_x(theta1_vals[start : stop], u1_vals[start : stop], 
            axis_labels=theta1_state_space_labels, 
            color='blue',
            label=None, 
            title=rf'$30 \leq t \leq 60$', 
            ax=ax_b)    

# third plot: state space plot from t= to t=12   
ax_c = fig.add_subplot(1,3,3)                  

start, stop = start_stop_indices(t_pts, 10., 1000.)    
plot_y_vs_x(theta1_vals[start : stop], u1_vals[start : stop], 
            axis_labels=theta1_state_space_labels, 
            color='blue',
            label=None, 
            title=rf'$10 \leq t \leq 1000$', 
            ax=ax_c)    

fig.tight_layout()
plt.show()

In [ ]:
# make a plot of Delta phi for same pendulum but two different initial conds
A = 16.0
U0_1 = mode_initial_conditions(A)
theta1_1, u1_1, theta2_1, u2_1, phi1_1, v1_1, phi2_1, v2_1 = p1.solve_ode(t_pts, *U0_1)

U0_2 = U0_1.copy()
delta_0 = -1.e-6
U0_2[0] += delta_0   # .000000001 radian lower
phi_dot_0 = 0.0
theta1_2, u1_2, theta2_2, u2_2, phi1_2, v1_2, phi2_2, v2_2 = p1.solve_ode(t_pts, *U0_2)

U_1 = np.array([theta1_1, u1_1, theta2_1, u2_1, phi1_1, v1_1, phi2_1, v2_1])
U_2 = np.array([theta1_2, u1_2, theta2_2, u2_2, phi1_2, v1_2, phi2_2, v2_2])

# Calculate the absolute value of \phi_2 - \phi_1
Delta_U = U_2 - U_1
Delta_U[4] = (Delta_U[4] + np.pi) % (2.*np.pi) - np.pi
Delta_U[6] = (Delta_U[6] + np.pi) % (2.*np.pi) - np.pi
Delta_U = np.linalg.norm(Delta_U, axis=0)

# Change the common font size
font_size = 14
plt.rcParams.update({'font.size': font_size})

# start the plot!
fig = plt.figure(figsize=(10,10))
overall_title = rf'Sensitivity to Initial Conditions: $A={A}$, ' # + \
                # rf'$\delta_0={delta_0:.0e}$' 
fig.suptitle(overall_title, va='baseline')
    
# two plot: plot from t=0 to t=8 and another from t=0 to t=100 
ax_a = fig.add_subplot(2,1,1)                  

start, stop = start_stop_indices(t_pts, 0., 10.)
ax_a.semilogy(t_pts[start : stop], Delta_U[start : stop], 
            color='blue', label=None)    
ax_a.set_xlabel(r'$t$')
ax_a.set_ylabel(r'$|\Delta\mathbf{U}|$')

ax_b = fig.add_subplot(2,1,2)                  

start, stop = start_stop_indices(t_pts, 0., 100.)
plot_y_vs_x(t_pts[start : stop], Delta_U[start : stop], 
            color='blue', label=None, semilogy=True)    
ax_b.set_xlabel(r'$t$')
ax_b.set_ylabel(r'$|\Delta\mathbf{U}|$')
                              
fig.tight_layout()
plt.show()

### **Increasing Perturbation Amplitude and Sensitivity to Initial Conditions**

To explore the transition from small oscillations to strongly nonlinear dynamics, the initial condition is constructed by perturbing the aligned relative equilibrium along the direction of a linear normal mode,

$$
\mathbf{U}(0;A)=\mathbf{U}_{\mathrm{eq}}+A\,\Delta\mathbf{U},
$$

where $A$ controls the perturbation amplitude. Increasing $A$ therefore generates a one-parameter family of ICs extending progressively farther from the relative equilibrium. For each amplitude, the nonlinear equations of motion are integrated, and the coordinate evolution $\theta_1(t)$ and projected state-space trajectory $(\theta_1,\dot{\theta}_1)$ are examined.

The coordinate evolution is plotted over $0\leq t\leq30$ to illustrate the short-time behavior. The state-space trajectories are examined over two additional intervals, $30\leq t\leq60$ and $10\leq t\leq1000$. These intervals are chosen to exclude the initial transient associated with the imposed perturbation and to reveal the subsequent dynamical structure over intermediate and long timescales. In particular, the intermediate-time projection illustrates the local organization of the motion, while the long-time projection reveals how the trajectory explores the accessible region of the projected state space. Since the DSP is a conservative system, excluding the initial transient does not imply that the system relaxes toward an attractor.

At $A=0.01$, the system remains within the small-oscillation regime. The coordinate evolution is nearly sinusoidal, while the projected state-space trajectory closely follows a closed elliptical curve characteristic of the corresponding linear normal mode. As the perturbation amplitude is increased, nonlinear effects become progressively more apparent. For $A=0.1$, $0.25$, and $0.5$, the overall oscillatory structure remains highly regular, although small distortions and modulations begin to appear in both the coordinate and state-space trajectories.

By $A=1$, the departure from the nearly elliptical small-amplitude motion becomes more noticeable. The state-space trajectory no longer retraces a single closed curve, but instead begins to form a narrow collection of neighboring loops. At $A=2$ and $A=4$, the modulation becomes considerably stronger. The coordinate evolution develops increasingly irregular variations in amplitude and waveform, while the long-time state-space projections broaden into dense bands composed of many overlapping loops. These trajectories are clearly far outside the linear regime, although the complicated appearance of a two-dimensional projection alone is not sufficient to identify the underlying motion as chaotic.

A more pronounced qualitative change occurs for $A=8$ and $A=16$. The coordinate evolution becomes highly irregular, with little of the simple repetitive structure observed at smaller amplitudes. The intermediate-time state-space projections display strongly varying loops that explore a much larger region of the $(\theta_1,\dot{\theta}_1)$ plane, while the long-time projections become densely populated and increasingly intricate. The progression from a nearly elliptical orbit at small $A$ to these broad, irregular projections demonstrates the increasing importance of nonlinear coupling as the system is driven farther from the relative equilibrium.

To further distinguish complicated nonlinear motion from possible chaotic behavior, sensitivity to ICs is examined by evolving two trajectories whose initial states differ by a small perturbation. Their separation is measured using the full state-space distance

$$
\delta(t)
=
\left\|
\mathbf{U}^{(2)}(t)-\mathbf{U}^{(1)}(t)
\right\|.
$$

At the smaller and intermediate amplitudes, the initially nearby trajectories separate relatively gradually and remain close over the early stages of the evolution. Increasing the amplitude changes this behavior substantially. In particular, for $A=8$ and $A=16$, the separation eventually grows by several orders of magnitude, indicating that a very small difference between the initial states can develop into macroscopically different trajectories. The rapid and irregular growth observed at these larger amplitudes is qualitatively distinct from the behavior at lower amplitudes and provides evidence of increasing sensitivity to ICs.

Taken together, the coordinate evolution, projected state-space trajectories, and sensitivity analysis reveal a clear qualitative progression as the perturbation amplitude is increased: nearly periodic small oscillations give way to strongly modulated nonlinear motion, followed at sufficiently large amplitudes by irregular trajectories exhibiting strong sensitivity to their ICs. These observations suggest the emergence of chaotic dynamics at large perturbation amplitudes, but they do not by themselves provide a complete characterization of the motion. In particular, because the full double spherical pendulum evolves in an eight-dimensional state space, the two-dimensional state-space plots shown here are only projections of the full dynamics. Poincaré sections are therefore examined next to more clearly distinguish organized nonlinear motion from chaotic behavior.

In [ ]:
# Labels for individual plot axes
theta1_vs_time_labels = (r'$t$', r'$\theta_1(t)$')
theta1_dot_vs_time_labels = (r'$t$', r'$d\theta_1/dt(t)$')
theta1_state_space_labels = (r'$\theta_1$', r'$d\theta_1/dt$')

# Common plotting time (generate the full time then use slices)
t_start = 0.
t_end = 200.
delta_t = 0.01

t_pts = np.arange(t_start, t_end + delta_t, delta_t)  

# Pendulum parameters
m1_param = 1.
m2_param = 1.
R1_param = 1.
R2_param = 1.
g_param = 9.8

# Instantiate the pendulums
p4 = Pendulum(m1=m1_param, m2=m2_param, R1=R1_param, R2=R2_param, g=g_param)
p5 = Pendulum(m1=m1_param, m2=m2_param, R1=R1_param, R2=R2_param, g=g_param)
p6 = Pendulum(m1=m1_param, m2=m2_param, R1=R1_param, R2=R2_param, g=g_param)
p7 = Pendulum(m1=m1_param, m2=m2_param, R1=R1_param, R2=R2_param, g=g_param)
p8 = Pendulum(m1=m1_param, m2=m2_param, R1=R1_param, R2=R2_param, g=g_param)
p9 = Pendulum(m1=m1_param, m2=m2_param, R1=R1_param, R2=R2_param, g=g_param)

# solve each of the pendulum odes
U0_4 = mode_initial_conditions(0.01)
theta1_4, theta1_dot_4, theta2_4, theta2_dot_4, phi1_4, phi1_dot_4, phi2_4, phi2_dot_4 = p4.solve_ode(t_pts, *U0_4)

U0_5 = mode_initial_conditions(1.)
theta1_5, theta1_dot_5, theta2_5, theta2_dot_5, phi1_5, phi1_dot_5, phi2_5, phi2_dot_5 = p5.solve_ode(t_pts, *U0_5)

U0_6 = mode_initial_conditions(2.)
theta1_6, theta1_dot_6, theta2_6, theta2_dot_6, phi1_6, phi1_dot_6, phi2_6, phi2_dot_6 = p6.solve_ode(t_pts, *U0_6)

U0_7 = mode_initial_conditions(4.)
theta1_7, theta1_dot_7, theta2_7, theta2_dot_7, phi1_7, phi1_dot_7, phi2_7, phi2_dot_7 = p7.solve_ode(t_pts, *U0_7)

U0_8 = mode_initial_conditions(8.)
theta1_8, theta1_dot_8, theta2_8, theta2_dot_8, phi1_8, phi1_dot_8, phi2_8, phi2_dot_8 = p8.solve_ode(t_pts, *U0_8)

U0_9 = mode_initial_conditions(16.)
theta1_9, theta1_dot_9, theta2_9, theta2_dot_9, phi1_9, phi1_dot_9, phi2_9, phi2_dot_9 = p9.solve_ode(t_pts, *U0_9)

In [ ]:
# Change the common font size
font_size = 14
plt.rcParams.update({'font.size': font_size})
box_props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)

# start the plot!
fig = plt.figure(figsize=(12,14.44))
overall_title = 'Coordinate Evolution with Increasing Perturbation Amplitude' # + \
                # rf' $\omega = {omega_ext:.2f},$' + \
                # rf' $\omega_0 = {omega_0:.2f},$' + \
                # rf' $\beta = {beta:.2f},$' + \
                # rf'  $\phi_0 = {phi_0:.2f},$' + \
                # rf' $\dot\phi_0 = {phi_dot_0:.2f}$' + \
                # '\n'     # \n means a new line (adds some space here)
fig.suptitle(overall_title, va='baseline')
    
# plot 1a: plot from t=0 to t=10   
ax_1a = fig.add_subplot(6,2,1)                  
start, stop = start_stop_indices(t_pts, 0., 10.)    
plot_y_vs_x(t_pts[start : stop], theta1_4[start : stop], 
            axis_labels=theta1_vs_time_labels, 
            color='blue',
            label=None, 
            ax=ax_1a) 
# ax_1a.set_ylim(-3, 3.5)
ax_1a.axhline(y=0., color='black', alpha=0.5)
# textstr = r'$\gamma = 1.060$' + '\n' + r' Period 1'
# ax_1a.text(-5.8, 0., textstr, bbox=box_props)
    
# plot 1b: plot from t=28 to t=40 blown up   
ax_1b = fig.add_subplot(6,2,2)                  
start, stop = start_stop_indices(t_pts, 28., 40.)    
plot_y_vs_x(t_pts[start : stop], theta1_4[start : stop], 
            axis_labels=theta1_vs_time_labels, 
            color='blue',
            label=None, 
            ax=ax_1b) 
# ax_1b.set_ylim(-2.5, -1.6)
    
# plot 2a: plot from t=0 to t=10   
ax_2a = fig.add_subplot(6,2,3)                  
start, stop = start_stop_indices(t_pts, 0., 10.)    
plot_y_vs_x(t_pts[start : stop], theta1_5[start : stop], 
            axis_labels=theta1_vs_time_labels, 
            color='blue',
            label=None, 
            ax=ax_2a) 
# ax_2a.set_ylim(-3, 3.5)
ax_2a.axhline(y=0., color='black', alpha=0.5)
# textstr = r'$\gamma = 1.078$' + '\n' + r' Period 2'
# ax_2a.text(-5.8, 0., textstr, bbox=box_props)
    
# plot 2b: plot from t=28 to t=40 blown up   
ax_2b = fig.add_subplot(6,2,4)                  
start, stop = start_stop_indices(t_pts, 28., 40.)    
plot_y_vs_x(t_pts[start : stop], theta1_5[start : stop], 
            axis_labels=theta1_vs_time_labels, 
            color='blue',
            label=None, 
            ax=ax_2b) 
# ax_2b.set_ylim(-2.5, -1.6)
    
# plot 3a: plot from t=0 to t=10   
ax_3a = fig.add_subplot(6,2,5)                  
start, stop = start_stop_indices(t_pts, 0., 10.)    
plot_y_vs_x(t_pts[start : stop], theta1_6[start : stop], 
            axis_labels=theta1_vs_time_labels, 
            color='blue',
            label=None, 
            ax=ax_3a) 
# ax_3a.set_ylim(-3, 3.5)
ax_3a.axhline(y=0., color='black', alpha=0.5)
# textstr = r'$\gamma = 1.081$' + '\n' + r' Period 4'
# ax_3a.text(-5.8, 0., textstr, bbox=box_props)
    
# plot 3b: plot from t=28 to t=40 blown up   
ax_3b = fig.add_subplot(6,2,6)                  
start, stop = start_stop_indices(t_pts, 28., 40.)    
plot_y_vs_x(t_pts[start : stop], theta1_6[start : stop], 
            axis_labels=theta1_vs_time_labels, 
            color='blue',
            label=None, 
            ax=ax_3b) 
# ax_3b.set_ylim(-2.5, -1.6)
    
# plot 4a: plot from t=0 to t=10   
ax_4a = fig.add_subplot(6,2,7)                  
start, stop = start_stop_indices(t_pts, 0., 10.)    
plot_y_vs_x(t_pts[start : stop], theta1_7[start : stop], 
            axis_labels=theta1_vs_time_labels, 
            color='blue',
            label=None, 
            ax=ax_4a) 
# ax_4a.set_ylim(-3, 3.5)
ax_4a.axhline(y=0., color='black', alpha=0.5)
# textstr = r'$\gamma = 1.0826$' + '\n' + r' Period 8'
# ax_4a.text(-5.8, 0., textstr, bbox=box_props)
    
# plot 4b: plot from t=28 to t=40 blown up   
ax_4b = fig.add_subplot(6,2,8)                  
start, stop = start_stop_indices(t_pts, 28., 40.)    
plot_y_vs_x(t_pts[start : stop], theta1_7[start : stop], 
            axis_labels=theta1_vs_time_labels, 
            color='blue',
            label=None, 
            ax=ax_4b) 
# ax_4b.set_ylim(-2.5, -1.6)
    
# plot 5a: plot from t=0 to t=10   
ax_5a = fig.add_subplot(6,2,9)                  
start, stop = start_stop_indices(t_pts, 0., 10.)    
plot_y_vs_x(t_pts[start : stop], theta1_8[start : stop], 
            axis_labels=theta1_vs_time_labels, 
            color='blue',
            label=None, 
            ax=ax_5a) 
# ax_5a.set_ylim(-3, 3.5)
ax_5a.axhline(y=0., color='black', alpha=0.5)
# textstr = r'$\gamma = 1.08294$' + '\n' + r' Period 16?'
# ax_5a.text(-5.8, 0., textstr, bbox=box_props)
    
# plot 5b: plot from t=28 to t=40 blown up   
ax_5b = fig.add_subplot(6,2,10)                  
start, stop = start_stop_indices(t_pts, 28., 40.)    
plot_y_vs_x(t_pts[start : stop], theta1_8[start : stop], 
            axis_labels=theta1_vs_time_labels, 
            color='blue',
            label=None, 
            ax=ax_5b) 
# ax_5b.set_ylim(-2.5, -1.6)

# plot 6a: plot from t=0 to t=10   
ax_6a = fig.add_subplot(6,2,11)                  
start, stop = start_stop_indices(t_pts, 0., 10.)    
plot_y_vs_x(t_pts[start : stop], theta1_9[start : stop], 
            axis_labels=theta1_vs_time_labels, 
            color='blue',
            label=None, 
            ax=ax_6a) 
# ax_6a.set_ylim(-3, 3.5)
ax_6a.axhline(y=0., color='black', alpha=0.5)
# textstr = r'$\gamma = 1.08294$' + '\n' + r' Period 16?'
# ax_6a.text(-5.8, 0., textstr, bbox=box_props)
    
# plot 6b: plot from t=28 to t=40 blown up   
ax_6b = fig.add_subplot(6,2,12)                  
start, stop = start_stop_indices(t_pts, 28., 40.)    
plot_y_vs_x(t_pts[start : stop], theta1_9[start : stop], 
            axis_labels=theta1_vs_time_labels, 
            color='blue',
            label=None, 
            ax=ax_6b) 
# ax_6b.set_ylim(-2.5, -1.6)

fig.tight_layout()
fig.savefig('Figure_12.8.png', bbox_inches='tight')  # always bbox_inches='tight'

In [ ]:
def poincare_section(t_pts, theta1, theta1_dot, theta2, theta2_dot, theta2_section):
    """Indices where theta2 crosses theta2_section with theta2_dot > 0."""
    t_poincare = []
    theta1_poincare = []
    theta1_dot_poincare = []
    for i in range(len(theta2) - 1):
        # Positive crossing of theta2 = theta2_section)
        if (theta2[i] < theta2_section and theta2[i + 1] >= theta2_section and theta2_dot[i] > 0.):
            # Fraction of the timestep at which the crossing occurs
            alpha = ((theta2_section - theta2[i]) / (theta2[i + 1] - theta2[i]))
            # Interpolate to the actual crossing
            t_cross = (t_pts[i] + alpha * (t_pts[i + 1] - t_pts[i]))
            theta1_cross = (theta1[i] + alpha * (theta1[i + 1] - theta1[i]))
            theta1_dot_cross = (theta1_dot[i] + alpha * (theta1_dot[i + 1] - theta1_dot[i]))
            t_poincare.append(t_cross)
            theta1_poincare.append(theta1_cross)
            theta1_dot_poincare.append(theta1_dot_cross)

    return (np.array(t_poincare), np.array(theta1_poincare), np.array(theta1_dot_poincare))

theta2_section = theta2_o_aligned

t_p4, theta1_p4, theta1_dot_p4 = poincare_section(t_pts, theta1_4, theta1_dot_4, theta2_4, theta2_dot_4, theta2_section)
t_p5, theta1_p5, theta1_dot_p5 = poincare_section(t_pts, theta1_5, theta1_dot_5, theta2_5, theta2_dot_5, theta2_section)
t_p6, theta1_p6, theta1_dot_p6 = poincare_section(t_pts, theta1_6, theta1_dot_6, theta2_6, theta2_dot_6, theta2_section)
t_p7, theta1_p7, theta1_dot_p7 = poincare_section(t_pts, theta1_7, theta1_dot_7, theta2_7, theta2_dot_7, theta2_section)
t_p8, theta1_p8, theta1_dot_p8 = poincare_section(t_pts, theta1_8, theta1_dot_8, theta2_8, theta2_dot_8, theta2_section)
t_p9, theta1_p9, theta1_dot_p9 = poincare_section(t_pts, theta1_9, theta1_dot_9, theta2_9, theta2_dot_9, theta2_section)

In [ ]:
# Change the common font size
font_size = 14
plt.rcParams.update({'font.size': font_size})
box_props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)

y_min = -5.
y_max = 5.
x_min = -0.1
x_max = 1.6
y_min_ps = -5.
y_max_ps = 5.
x_min_ps = -0.1
x_max_ps = 1.6

# start the plot!
fig_ss = plt.figure(figsize=(12,14.44))
overall_title = 'Projected State space and Poincare sections:' # + \
                # rf' $\omega = {omega_ext:.2f},$' + \
                # rf' $\omega_0 = {omega_0:.2f},$' + \
                # rf' $\beta = {beta:.2f},$' + \
                # rf'  $\phi_0 = {phi_0:.2f},$' + \
                # rf' $\dot\phi_0 = {phi_dot_0:.2f}$'
                # #'\n'     # \n means a new line (adds some space here)
fig_ss.suptitle(overall_title)
    
# plot 1a: state space plot from t=40. to t=100.   
ax_ss_1a = fig_ss.add_subplot(6,2,1)                
start, stop = start_stop_indices(t_pts, 40., 100.)
mask_4_ss = (t_p4 >= 40.) & (t_p4 < 100.)      
plot_y_vs_x(theta1_4[start : stop], theta1_dot_4[start : stop], 
            axis_labels=theta1_state_space_labels, 
            color='blue',
            label=None, 
            ax=ax_ss_1a) 
plot_y_vs_x(theta1_p4[mask_4_ss], 
            theta1_dot_p4[mask_4_ss],
            axis_labels=theta1_state_space_labels, 
            color='red',
            label=None,
            points=True,
            ax=ax_ss_1a) 
ax_ss_1a.set_ylim(y_min, y_max)
ax_ss_1a.set_xlim(x_min, x_max)
textstr = r'$A = 0.01$'
ax_ss_1a.text(-1, 0.5, textstr, bbox=box_props)
    
# plot 1b: poincare plot from t=100 to t=120.  
ax_ss_1b = fig_ss.add_subplot(6,2,2)                  
start, stop = start_stop_indices(t_pts, 100., 120.) 
mask_4_ps = (t_p4 >= 100.) & (t_p4 < 120.)
plot_y_vs_x(theta1_p4[mask_4_ps], 
            theta1_dot_p4[mask_4_ps],
            axis_labels=theta1_state_space_labels, 
            color='red',
            label=None,
            points=True,
            ax=ax_ss_1b) 
ax_ss_1b.set_ylim(y_min_ps, y_max_ps)
ax_ss_1b.set_xlim(x_min_ps, x_max_ps)
    
# plot 2a: state space plot from t=40. to t=100.  
ax_ss_2a = fig_ss.add_subplot(6,2,3)                  
start, stop = start_stop_indices(t_pts, 40., 100.)    
mask_5_ss = (t_p5 >= 40.) & (t_p5 < 100.)
plot_y_vs_x(theta1_5[start : stop], theta1_dot_5[start : stop], 
            axis_labels=theta1_state_space_labels, 
            color='blue',
            label=None, 
            ax=ax_ss_2a) 
plot_y_vs_x(theta1_p5[mask_5_ss], 
            theta1_dot_p5[mask_5_ss],
            axis_labels=theta1_state_space_labels, 
            color='red',
            points=True,
            label=None, 
            ax=ax_ss_2a) 
ax_ss_2a.set_ylim(y_min, y_max)
ax_ss_2a.set_xlim(x_min, x_max)
textstr = r'$A = 1$'
ax_ss_2a.text(-1, 0.5, textstr, bbox=box_props)
    
# plot 2b: poincare plot from t=100 to t=120.  
ax_ss_2b = fig_ss.add_subplot(6,2,4)                  
start, stop = start_stop_indices(t_pts, 100., 120.)    
mask_5_ps = (t_p5 >= 100.) & (t_p5 < 120.)
plot_y_vs_x(theta1_p5[mask_5_ps], 
            theta1_dot_p5[mask_5_ps],
            axis_labels=theta1_state_space_labels, 
            color='red',
            points=True,
            label=None, 
            ax=ax_ss_2b) 
ax_ss_2b.set_ylim(y_min_ps, y_max_ps)
ax_ss_2b.set_xlim(x_min_ps, x_max_ps)
    
# plot 3a: state space plot from t=40. to t=100.  
ax_ss_3a = fig_ss.add_subplot(6,2,5)                  
start, stop = start_stop_indices(t_pts, 40., 100.)   
mask_6_ss = (t_p6 >= 40.) & (t_p6 < 100.)
plot_y_vs_x(theta1_6[start : stop], theta1_dot_6[start : stop], 
            axis_labels=theta1_state_space_labels, 
            color='blue',
            label=None, 
            ax=ax_ss_3a) 
plot_y_vs_x(theta1_p6[mask_6_ss], 
            theta1_dot_p6[mask_6_ss],
            axis_labels=theta1_state_space_labels, 
            color='red',
            points=True,
            label=None, 
            ax=ax_ss_3a) 
ax_ss_3a.set_ylim(y_min, y_max)
ax_ss_3a.set_xlim(x_min, x_max)
textstr = r'$A = 2$'
ax_ss_3a.text(-1, 0.5, textstr, bbox=box_props)
    
# plot 3b: poincare plot from t=100 to t=120.  
ax_ss_3b = fig_ss.add_subplot(6,2,6)                  
start, stop = start_stop_indices(t_pts, 100., 120.)    
mask_6_ps = (t_p6 >= 100.) & (t_p6 < 120.)
plot_y_vs_x(theta1_p6[mask_6_ps], 
            theta1_dot_p6[mask_6_ps],
            axis_labels=theta1_state_space_labels, 
            color='red',
            points=True,
            label=None, 
            ax=ax_ss_3b) 
ax_ss_3b.set_ylim(y_min_ps, y_max_ps)
ax_ss_3b.set_xlim(x_min_ps, x_max_ps)
    
# plot 4a: state space plot from t=40. to t=100.  
ax_ss_4a = fig_ss.add_subplot(6,2,7)                  
start, stop = start_stop_indices(t_pts, 40., 100.)    
mask_7_ss = (t_p7 >= 40.) & (t_p7 < 100.)
plot_y_vs_x(theta1_7[start : stop], theta1_dot_7[start : stop], 
            axis_labels=theta1_state_space_labels, 
            color='blue',
            label=None, 
            ax=ax_ss_4a) 
plot_y_vs_x(theta1_p7[mask_7_ss], 
            theta1_dot_p7[mask_7_ss],
            axis_labels=theta1_state_space_labels, 
            color='red',
            points=True,
            label=None, 
            ax=ax_ss_4a) 
ax_ss_4a.set_ylim(y_min, y_max)
ax_ss_4a.set_xlim(x_min, x_max)
textstr = r'$A = 4$'
ax_ss_4a.text(-1, 0.5, textstr, bbox=box_props)
    
# plot 4b: poincare plot from t=100 to t=120.  
ax_ss_4b = fig_ss.add_subplot(6,2,8)                  
start, stop = start_stop_indices(t_pts, 100., 120.)   
mask_7_ps = (t_p7 >= 100.) & (t_p7 < 120.)
plot_y_vs_x(theta1_p7[mask_7_ps], 
            theta1_dot_p7[mask_7_ps],
            axis_labels=theta1_state_space_labels, 
            color='red',
            points=True,
            label=None, 
            ax=ax_ss_4b) 
ax_ss_4b.set_ylim(y_min_ps, y_max_ps)
ax_ss_4b.set_xlim(x_min_ps, x_max_ps)
    
# plot 5a: state space plot from t=40. to t=100.  
ax_ss_5a = fig_ss.add_subplot(6,2,9)                  
start, stop = start_stop_indices(t_pts, 40., 100.)    
mask_8_ss = (t_p8 >= 40.) & (t_p8 < 100.)
plot_y_vs_x(theta1_8[start : stop], theta1_dot_8[start : stop], 
            axis_labels=theta1_state_space_labels, 
            color='blue',
            label=None, 
            ax=ax_ss_5a) 
plot_y_vs_x(theta1_p8[mask_8_ss], 
            theta1_dot_p8[mask_8_ss],
            axis_labels=theta1_state_space_labels, 
            color='red',
            points=True,
            label=None, 
            ax=ax_ss_5a) 
ax_ss_5a.set_ylim(y_min, y_max)
ax_ss_5a.set_xlim(x_min, x_max)
textstr = r'$A = 8$'
ax_ss_5a.text(-1, 0.5, textstr, bbox=box_props)
    
# plot 5b: poincare plot from t=100 to t=120.  
ax_ss_5b = fig_ss.add_subplot(6,2,10)                  
start, stop = start_stop_indices(t_pts, 100., 120.)    
mask_8_ps = (t_p8 >= 100.) & (t_p8 < 120.)
plot_y_vs_x(theta1_p8[mask_8_ps], 
            theta1_dot_p8[mask_8_ps],
            axis_labels=theta1_state_space_labels, 
            color='red',
            points=True,
            label=None, 
            ax=ax_ss_5b) 
ax_ss_5b.set_ylim(y_min_ps, y_max_ps)
ax_ss_5b.set_xlim(x_min_ps, x_max_ps)

# plot 6a: state space plot from t=40. to t=100.  
ax_ss_6a = fig_ss.add_subplot(6,2,11)                  
start, stop = start_stop_indices(t_pts, 40., 100.)    
mask_9_ss = (t_p9 >= 40.) & (t_p9 < 100.)
plot_y_vs_x(theta1_9[start : stop], theta1_dot_9[start : stop], 
            axis_labels=theta1_state_space_labels, 
            color='blue',
            label=None, 
            ax=ax_ss_6a) 
plot_y_vs_x(theta1_p9[mask_9_ss], 
            theta1_dot_p9[mask_9_ss],
            axis_labels=theta1_state_space_labels, 
            color='red',
            points=True,
            label=None, 
            ax=ax_ss_6a) 
ax_ss_6a.set_ylim(y_min, y_max)
ax_ss_6a.set_xlim(x_min, x_max)
textstr = r'$A = 16$'
ax_ss_6a.text(-1, 0.5, textstr, bbox=box_props)
    
# plot 6b: poincare plot from t=100 to t=120.  
ax_ss_6b = fig_ss.add_subplot(6,2,12)                  
start, stop = start_stop_indices(t_pts, 100., 120.)    
mask_9_ps = (t_p9 >= 100.) & (t_p9 < 120.)
plot_y_vs_x(theta1_p9[mask_9_ps], 
            theta1_dot_p9[mask_9_ps],
            axis_labels=theta1_state_space_labels, 
            color='red',
            points=True,
            label=None, 
            ax=ax_ss_6b) 
ax_ss_6b.set_ylim(y_min_ps, y_max_ps)
ax_ss_6b.set_xlim(x_min_ps, x_max_ps)

fig.tight_layout()
# fig.savefig('Figure_12.8_state_space_and_Poincare.png', bbox_inches='tight')  
# always bbox_inches='tight'

### **Poincaré Sections**

The projected state-space trajectories above become increasingly complicated as the perturbation amplitude $A$ is increased. However, because the trajectory evolves continuously through a high-dimensional state space, its projection onto the $(\theta_1, \dot{\theta}_1)$ plane can become dense and difficult to interpret. A Poincaré section provides a simpler way of examining the underlying structure of the motion by recording the state of the system only when the trajectory crosses a specified surface in phase space.

For the DSP, we define the Poincaré section by 

$$ 
P = \left\{ \mathbf{U}: \theta_2 = \theta_{2, 0}, \qquad \dot{\theta}_2 > 0\right\}, 
$$

where $\theta_{2, 0}$ is the equilibrium value of $\theta_2$. The condition $\dot{\theta}_2 > 0$ restricts the section to crossings in a single direction, preventing the upward and downward intersections with the same surface from being mixed together. At each crossing of $P$, the coordinates $(\theta_1, \dot{\theta}_1)$ are recorded, producing a two-dimensional projection of the Poincaré section. 

Because the numerical solution is evaluated at discrete time intervals, the trajectory will generally cross $\theta_2 = \theta_{2, 0}$ between two consecutive numerical samples rather than exactly at one of them. The location of each crossing is therefore estimated using linear interpolation. If the crossing occurs between $t_i$ and $t_{i + 1}$, the interpolation parameter is 

$$
\alpha = \frac{\theta_{2, 0} - \theta_2(t_i)}{\theta_2(t_{i + 1}) - \theta_2(t_i)}, 
$$

and the corresponding Poincaré coordinates are approximated by 

$$
\theta_{1, P} = \theta_1(t_i) + \alpha\left[\theta_1(t_{i + 1}) - \theta_1(t_i)\right], \\
\dot{\theta}_{1, P} = \dot{\theta}_1(t_i) + \alpha\left[\dot{\theta}_1(t_{i + 1}) - \dot{\theta}_1(t_i)\right]. 
$$

The figure above compares the projected state-space trajectory in $(\theta_1, \dot{\theta}_1)$ with its corresponding projected Poincaré section for increasing perturbation amplitude. The continuous state-space trajectories are shown in blue, while intersections with $P$ are shown in red. As in the previous state-space analysis, the initial portion of the trajectory is excluded so that the comparison emphasizes the intermediate- and longer-time behavior rather than the initial transient response. Common axis limits are used throughout the figure so that the physical extent of the trajectories and Poincaré sections can be compared directly between different values of $A$.

For $A = 0.01$, the state-space trajectory is an extremely small closed orbit about the relative equilibrium, and the Poincaré intersections are localized to essentially a single point at the scale of the figure. This is consistent with the nearly perriodic motion expected in the linear regime. At $A = 1$, the state-space orbit remains narrow and regular, while the Poincaré intersections remain strongly localized. 

At $A = 2$, the projected trajectory begins to broaden and the Poincaré section separates into a small structured set of intersections. By $A = 4$, the state-space trajectory forms a substantially broader band, while the Poincaré intersections occupy an extended but still relatively confined and organized region. These structures indicate that the nonlinear motion has become considerably more complicated than the small-amplitude oscillations, although the intersections remain strongly organized rather than broadly dispersed throughout the projected section. 

A more pronounced qualitative change occurs for $A = 8$ and $A = 16$. The projected state-space trajectories now cover broad regions of the $(\theta_1, \dot{\theta}_1)$ plane and no longer resemble the narrow closed or banded trajectories observed at smaller amplitudes. Correspondingly, the Poincaré intersections become widely dispersed over the projected section rather than remaining concentrated within a small structured region. This transition agrees with the increasingly irregular coordinate trajectories and enhanced sensitivity to nearby ICs observed previously. 

The Poincaré sections therefore provide an additional view of the transition from near-linear, highly organized motion at small $A$ toward increasingly irregular dynamics at large $A$. In particular, the qualitative distinction between the confined sections for $A \leq 4$ and the broadly dispersed sections for $A = 8$ and $A = 16$ is consistent with the other indicators of chaotic behavior examined above. However, because $(\theta_1, \dot{\theta}_1)$ represents only a two-dimensional projection of the higher dimensional Poincaré section, the appearance of a broadly scattered set of points along is not sufficient to establish chaos. Instead, the Poincaré sections are interpreted together with the coordinate evolution, projected state-space trajectories, and sensitivity to ICs as complementary evidence for the onset of strongly irregular, potentially chaotic dynamics at sufficiently large perturbation amplitudes. 

# 4 Utilization of Advanced Numerical Approaches

## 4.1 Investigation of Multiple Numerical Integration Techniques

The EOMs derived for the DSP form a coupled system of nonlinear, second-order ODEs. Because obtaining a practical closed-form solution is generally infeasible, numerical integration is required to approximate the system's evolution from specified ICs.

To apply standard ODE solvers, the second-order EOMs are rewritten as a first-order initial value problem (IVP). Given generalized coordinates $\mathbf{q}$ and accelerations determined by

$$
\ddot{\mathbf{q}}=\mathbf{a}(\mathbf{q},\dot{\mathbf{q}},t),
$$

the state vector is defined as

$$
\mathbf{y}=
\begin{bmatrix}
\mathbf{q}\\
\dot{\mathbf{q}}
\end{bmatrix},
$$

yielding

$$
\dot{\mathbf{y}}=\mathbf{f}(t,\mathbf{y}),
\qquad
\mathbf{y}(t_0)=\mathbf{y}_0.
$$

Numerical integration approximates the solution by advancing this state through discrete timesteps. The original DSP simulation uses SciPy's `solve_ivp`, which provides several integration algorithms with different accuracy, stability, and computational characteristics.

This investigation compares six methods (RK23, RK45, DOP853, Radau, BDF, and LSODA) under common ICs and numerical tolerances. Their performance is evaluated in terms of computational effort and numerical accuracy to identify suitable approaches for simulating the DSP.

### Numerical Integration and Adaptive Error Control

For a timestep $h$, a numerical integrator advances an approximation $\mathbf{y}_n$ at time $t_n$ to a new state at $t_{n+1}=t_n+h$.

The simplest example is the forward Euler method,

$$
\mathbf{y}_{n+1}
=
\mathbf{y}_n+h\mathbf{f}(t_n,\mathbf{y}_n),
$$

which estimates the next state using the derivative at the beginning of the timestep. Although computationally inexpensive, Euler's first-order accuracy can require very small timesteps to resolve nonlinear dynamics reliably.

Higher-order Runge-Kutta methods improve accuracy by evaluating derivatives at multiple intermediate states. Many modern solvers additionally employ **adaptive timestep selection**, automatically adjusting $h$ according to estimates of the local numerical error.

Embedded Runge-Kutta methods estimate this error by comparing approximations of different orders:

$$
\mathbf{e}_{n+1}
\approx
\mathbf{y}_{n+1}^{(p)}
-
\mathbf{y}_{n+1}^{(p-1)}.
$$

The estimated error is assessed against a componentwise tolerance scale of approximately

$$
\mathrm{atol}_i+\mathrm{rtol}|y_i|,
$$

where `atol` controls the absolute error scale and `rtol` controls the allowable error relative to the magnitude of the state.

If the estimated error exceeds the permitted tolerance, the step is rejected and retried with a smaller timestep. Otherwise, it is accepted, and the solver may increase the timestep for subsequent steps.

This introduces a fundamental tradeoff: tighter tolerances generally improve local accuracy but increase the number of derivative evaluations and computational cost. Moreover, local error control does not guarantee a correspondingly small global trajectory error, particularly for nonlinear systems in which small discrepancies may grow over time.

### Integration Methods Available Through `solve_ivp`

The six methods considered can be classified into three categories.

**Explicit Runge-Kutta methods**

Explicit methods calculate intermediate derivatives directly from previously evaluated stages, making them particularly suitable for nonstiff problems.

- **RK23:** An embedded Runge-Kutta method of orders 3 and 2. Its relatively low computational cost per step may be offset by the greater number of steps required for high accuracy.
- **RK45:** An embedded Dormand–Prince method of orders 5 and 4. It provides a balance between accuracy and computational cost and is SciPy's default `solve_ivp` method.
- **DOP853:** A high-order explicit Runge-Kutta method of order 8. Although more expensive per step, it can achieve high accuracy with comparatively fewer steps, particularly under strict tolerances.

These methods illustrate the tradeoff between the computational cost of individual steps and the number of steps required to achieve a desired accuracy.

**Implicit methods and stiff differential equations**

Implicit methods are designed primarily for stiff systems, where numerical stability may force explicit solvers to use extremely small timesteps. They generally require solving additional algebraic systems during integration, increasing computational cost per step but improving stability.

- **Radau:** A fifth-order implicit Runge-Kutta method with favorable stability properties for stiff systems.
- **BDF:** A variable-order implicit multistep method using backward differentiation formulas of orders 1 through 5.

For predominantly nonstiff DSP dynamics, the additional computational cost of implicit methods may outweigh their stability advantages. Nevertheless, their inclusion provides a useful comparison between algorithms designed for different numerical conditions.

**Automatic stiffness detection**

**LSODA** automatically switches between an Adams method for nonstiff behavior and a BDF method when stiffness is detected. This flexibility makes it useful when the stiffness characteristics of a system are unknown or vary during integration.

However, automatic method switching does not necessarily guarantee greater efficiency than a solver specifically suited to the problem.

### Comparison Objective

The investigation examines how solver choice and error tolerances affect the accuracy and computational cost of DSP simulations. Because nonlinear dynamics can amplify small numerical discrepancies, performance is assessed using more than execution time alone.

By comparing computational effort, agreement with a high-accuracy reference solution, and energy conservation, the analysis aims to identify integration methods that provide a suitable balance between numerical accuracy and efficiency.

Note: The following cell benchmarks six numerical integration methods across five relative error tolerances, requiring 30 independent integrations of the DSP EOMs. Execution may take several minutes or longer, depending on the hardware and solver performance, particularly for the most stringent tolerances. Runtime measurements are hardware-dependent and should be interpreted as comparative results rather than universal performance benchmarks.

In [ ]:
import time 
import pandas as pd

methods = ["RK23", "RK45", "DOP853", "Radau", "BDF", "LSODA"]
relerrs = [1e-3, 1e-5, 1e-7, 1e-9, 1e-11]

t_start = 0.
t_end = 50.01
delta_t = 0.01
t_pts = np.arange(t_start, t_end, delta_t)

theta1_0 = np.pi / 3
u1_0 = 3.0
theta2_0 = np.pi / 3
u2_0 = 4.0
phi1_0 = 0.  # -np.pi / 2.
v1_0 = -3.0
phi2_0 = -np.pi
v2_0 = -4.0

solutions = {}
records = []
for method in methods:
    for relerr in relerrs:
        print(
            f"Running {method}, rtol={relerr:.0e}...",
            flush=True
        )
        abserr = 1e-2 * relerr
        start = time.perf_counter()
        sol = p1.solve_ode_benchmark(t_pts, theta1_0, u1_0, theta2_0, u2_0, phi1_0, v1_0, phi2_0, v2_0, method=method, abserr=abserr, relerr=relerr)
        runtime = time.perf_counter() - start
        solutions[(method, relerr)] = sol
        records.append({
            "method": method,
            "rtol": relerr,
            "atol": abserr,
            "runtime": runtime, 
            "nfev": sol.nfev,
            "njev": sol.njev,
            "nlu": sol.nlu,
            "success": sol.success
        })
benchmark = pd.DataFrame(records)
benchmark

All six integration methods successfully completed the simulation across the full range of tested tolerances. In general, computational cost increased as the error tolerances were tightened, although the magnitude of this increase varied considerably between methods. RK23 exhibited the strongest increase in computational effort, rising from 12,611 function evaluations and approximately 0.57 s at $\mathrm{rtol} = 10^{-3}$ to over 4.4 million evaluations and 208 s at $\mathrm{rtol} = 10^{-11}$. RK45 scaled more favorably, while DOP853 showed particularly moderate growth in computational cost as the tolerance was tightened. 

The implicit methods Radau and BDF generally required greater computational effort than the explicit higher-order Runge-Kutta methods, with additional Jacobian evaluations and LU decompositions associated with their implicit solution procedures. LSODA remained comparatively efficient throughout the tolerance sweep, with its runtime increasing from approximately 0.79 s to 3.56 s. Overall, DOP853 and LSODA displayed the smallest growth in runtime at increasingly strict tolerlances, while RK23 and Radau became particularly expensive at the tightest tolerances.

These results provides an initial comparison of computational cost, but they do not yet establish which method is most efficient for a given level of numerical accuracy. Since identical values of $\mathrm{rtol}$ and $\mathrm{atol}$ do not necessarily produce identical global errors across different integration methods, a reference solution is required before a direct work-precision comparison can be made. 

In [ ]:
ref_12 = p1.solve_ode_benchmark(
    t_pts, 
    theta1_0, u1_0,
    theta2_0, u2_0, 
    phi1_0, v1_0, 
    phi2_0, v2_0,
    method="DOP853",
    relerr=1e-12,
    abserr=1e-14
)

ref_13 = p1.solve_ode_benchmark(
    t_pts,
    theta1_0, u1_0, 
    theta2_0, u2_0,
    phi1_0, v1_0, 
    phi2_0, v2_0,
    method="DOP853",
    relerr=1e-13,
    abserr=1e-15
)

print(ref_12.success, ref_12.message)
print(ref_13.success, ref_13.message)
print(ref_12.nfev, ref_13.nfev)

ref_diff = ref_13.y - ref_12.y
ref_diff_norm =  np.linalg.norm(ref_diff, axis=0)

plt.figure(figsize=(8, 5))
plt.semilogy(t_pts, ref_diff_norm)
plt.xlabel("Time (s)")
plt.ylabel(r"$\|\mathbf{y}_{13} - \mathbf{y}_{12}\|_2$")
plt.title("Convergence of Reference Trajectory")
plt.grid(True)
plt.show()

In [ ]:
thresholds = [1e-8, 1e-6, 1e-4, 1e-2, 1]
for threshold in thresholds:
    indices = np.where(ref_diff_norm >= threshold)[0]

    if len(indices) > 0:
        i = indices[0]
        print(f"{threshold:.0e}: first exceeded at t = {t_pts[i]:.2f} s")
    else:
        print(f"{threshold:.0e}: never exceeded")

In [ ]:
# Reference-comparison window
t_error_max = 10.0
mask = ref_13.t <= t_error_max

# Separation between the two reference candidates over [0, 10] s
ref_diff_10 = ref_diff_norm[mask]

max_ref_diff = np.max(ref_diff_10)
rms_ref_diff = np.sqrt(np.mean(ref_diff_10**2))

print(f"Maximum reference separation: {max_ref_diff:.3e}")
print(f"RMS reference separation: {rms_ref_diff:.3e}")

Since the specified solver tolerances control local error estimates rather than the global trajectory error, a high-accuracy reference solution was required to evaluate the numerial accuracy of each integration method. DOP853 was selected to construct this reference due to its high order and favorable computational performance at strict tolerances in the preliminary benchmark. Two reference candidates were generated using $\mathrm{rtol} = 10^{-12}$ and $10^{-13}$, with corresponding absolute tolerances of $10^{-14}$ and $10^{-15}$, respectively. Comparing these trajectories provides an empirical convergence check rather than assuming that a single tightly toleranced solution represents the exact solution.

Due to the sensitive dynamics of the DSP, small numerical differences between the reference candidates were progressively amplified at longer times. The trajectory-error analysis was therefore restricted to $0 \leq t \leq 10$ s. Within this interval, the two candidates exhibited and RMS state-space separation of $4.780 \times 10^{-8}$ and a maximum separation of $9.984 \times 10^{-7}$. The tighter DOP853 solution was consequently adopted as a practical numerical reference over this interval, rather than being interpreted as an exact solution. 

In [ ]:
# Practical reference over the comparison interval
y_ref = ref_13.y[:, mask]

rms_errors = []
max_errors = []
for method in methods:
    for relerr in relerrs:
        sol = solutions[(method, relerr)]
        sol_mask = sol.t <= t_error_max
        # State-space difference from reference over [0, 10] s
        diff = sol.y[:, sol_mask] - y_ref
        # Euclidean state-space error at each output time
        error_t = np.linalg.norm(diff, axis=0)
        # Aggregate error measures
        rms_error = np.sqrt(np.mean(error_t**2))
        max_error = np.max(error_t)
        rms_errors.append(rms_error)
        max_errors.append(max_error)

benchmark["rms_error"] = rms_errors
benchmark["max_error"] = max_errors

In [ ]:
plt.figure(figsize=(9, 6))

for method in methods:
    data = benchmark[benchmark["method"] == method]

    plt.loglog(
        data["rms_error"],
        data["runtime"],
        marker="o",
        label=method
    )

plt.xlabel("RMS Trajectory Error")
plt.ylabel("runtime (s)")
plt.title("Work-Precision Comparison of Numerical Integrators")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

for method in methods:
    data = benchmark[benchmark["method"] == method]

    plt.loglog(
        data["rms_error"], 
        data["nfev"],
        marker="o",
        label=method
    )

plt.xlabel("RMS Trajectory Error")
plt.ylabel("Function Evaluations")
plt.title("Work-Precision Comparison by Function Evaluations")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()

The work-precision comparison demonstrates that equal solver tolerances do not correspond to equal global trajectory accuracy. Across the tested accuracy range, DOP853, LSODA, and RK45 generally provided the most favorable relationship between computational cost and RMS trajectory error. LSODA and RK45 were particularly competitive at moderate accuracy, while the eighth-order DOP853 method maintained strong efficiency as the required accuracy increased. The comparatively shallow slope of DOP853's work-precision curve indicates that substantial reductions in trajectory error were achieved with relatively modest increases in computational runtime. This behavior is consistent with the advantages of its high-order integration scheme. DOP853 therefore provided the most favorable overall work-precision behavior among the tested methods.

Radau achieved the smallest measured trajectory error, but only at substantially greater computational cost due to the additional work associated with its implicit formulation. BDF was similarly less competitive than the leading explicit and automatically switching methods. RK23 exhibited the poorest scaling with increasing accuracy requirements, requiring approximately $4.4 \times 10^6$ function evaluations at the strictest tolerance. The comparatively strong performance of the explicit Runge–Kutta methods, together with the absence of an efficiency advantage for the implicit methods, provides no indication that stiffness is a dominant numerical limitation for the trajectory and parameter regime investigated here.

In [ ]:
energy_rms_errors = []
energy_max_errors = []

for method in methods:
    for relerr in relerrs:
        sol = solutions[(method, relerr)]
        H_vals = H_num(*sol.y, *parameters)
        relative_energy_error = ((H_vals - H_vals[0]) / np.abs(H_vals[0]))
        energy_rms_errors.append(np.sqrt(np.mean(relative_energy_error**2)))
        energy_max_errors.append(np.max(np.abs(relative_energy_error)))

benchmark["energy_rms_error"] = energy_rms_errors
benchmark["energy_max_error"] = energy_max_errors

In [ ]:
plt.figure(figsize=(9, 6))

for method in methods:
    data = benchmark[benchmark["method"] == method]
    plt.loglog(
        data["energy_rms_error"],
        data["runtime"],
        marker="o",
        label=method
    )

plt.xlabel("RMS Relative Energy Error")
plt.ylabel("Runtime (s)")
plt.title("Energy Work-Precision Comparison")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()

Energy conservation provided an additional accuracy measure independent of pointwise agreement with the reference trajectory. 
For every integration method, tightening the error tolerances resulted in progressively improved conservation of the Hamiltonian. 
The energy work-precision comparison again favored DOP853, which achieved an RMS relative energy error of $1.37 \times 10^{-10}$ at the strictest tested tolerance with a runtime of only $3.84$ s. 
Radau achieved the smallest energy error overall, reaching $2.05 \times 10^{-12}$, but required substantially greater computational effort. 
This result complements the reference-trajectory comparison, since conservation of the Hamiltonian remains a meaningful numerical diagnostic even after sensitive dependence causes nearby trajectories to diverge. 

## 4.2 Using ML-Based Numerical Algorithms

The preceding investigations examined the DSP through analytical EOMs and conventional numerical integration methods. An alternative approach is to approximate the system's dynamics using machine learning (ML), allowing a neural network (NN) to learn a dynamical representation from simulated trajectories.

Of particular interest are physics-informed architectures, which incorporate known mechanical principles into the learning process rather than treating the system as an unrestricted state-to-acceleration mapping.

### Lagrangian Neural Networks (LNN)

The initial investigation employed a **Lagrangian Neural Network (LNN)**, which parameterizes a scalar Lagrangian $L_\theta(\mathbf{q},\dot{\mathbf{q}})$ using a neural network. The corresponding equations of motion are obtained through the E–L equations,

$$
\frac{d}{dt}
\left(
\frac{\partial L_\theta}{\partial\dot{\mathbf{q}}}
\right)
-
\frac{\partial L_\theta}{\partial\mathbf{q}}
=0.
$$

Automatic differentiation provides the derivatives required to determine the generalized accelerations.

For the DSP, the implementation additionally employed sine and cosine representations of angular coordinates and regularization to address numerical difficulties associated with a potentially singular learned mass matrix.

Although this formulation introduces a physical inductive bias by requiring the dynamics to arise from a learned Lagrangian, the resulting model was unable to reproduce the analytical DSP trajectories with sufficient accuracy.

The implementation, training procedure, and numerical investigation are documented separately in `dsp_lnn.ipynb` and are therefore not repeated here.

### Transition to Constrained Lagrangian Neural Networks (CLNN)

The limitations of the initial LNN motivated an investigation of **Constrained Lagrangian Neural Networks (CLNNs)**, which incorporate known holonomic constraints directly into the learned dynamics.

Unlike the generalized-coordinate LNN formulation, the CLNN represents the DSP using Cartesian coordinates while explicitly enforcing the fixed lengths of its two rods. Although six Cartesian positional coordinates are used, the constraints reduce the system to four independent configurational degrees of freedom, consistent with the original angular formulation.

This introduces a stronger physical inductive bias: the network learns the unknown dynamical representation while the prescribed geometric constraints are incorporated into the EOMs.

The initial CLNN experiment (**Exp1**) was trained using randomly sampled trajectory segments. While the model reproduced many analytical trajectories accurately, a small subset exhibited substantially larger prediction errors. These problematic trajectories motivated further investigation into whether the training data adequately represented the configurations encountered during evaluation.

A second experiment (**Exp2**) was subsequently introduced to examine this possibility. Rather than increasing the dataset size or training duration, Exp2 employed geometrically stratified sampling to increase the representation of configurations involving larger first-bob polar angles. This allowed the effect of training-data distribution to be investigated while holding the model architecture and overall training effort fixed.

Following this comparison, a third experiment (**Exp3**) was conducted to investigate whether greater training effort could improve performance without explicitly modifying the sampling strategy. Exp3 retained random sampling but increased both the number of training chunks and optimization updates by a factor of four.

Thus, the three CLNN experiments emerged sequentially from the results of the preceding investigations. Exp1 established the initial model performance, Exp2 investigated whether geometric training-data coverage could address the observed prediction difficulties, and Exp3 examined whether increasing training-data quantity and optimization effort could provide a comparable improvement.

The following subsections develop the constrained Cartesian formulation, describe the training and evaluation procedures, and analyze the resulting differences in predictive performance.

The CLNN investigation follows the framework introduced by Finzi, Wang, and Wilson in *Simplifying Hamiltonian and Lagrangian Neural Networks via Explicit Constraints*. Their approach incorporates known holonomic constraints into a learned mechanical system, allowing dynamics to be represented in Cartesian coordinates while maintaining the geometric relationships between its components.

Rather than reproducing the general derivation of the CLNN framework, this investigation focuses on its adaptation to the DSP, the associated training procedure, and the accuracy of the resulting learned dynamics.

#### Cartesian Representation of the DSP

Consider two point masses with Cartesian positions

$$
\mathbf{r}_1=(x_1,y_1,z_1),
\qquad
\mathbf{r}_2=(x_2,y_2,z_2),
$$

connected by rigid, massless rods of lengths $R_1$ and $R_2$. The first rod connects the origin to the first mass, while the second connects the two masses.

The six Cartesian positional coordinates are subject to two holonomic constraints,

$$
\Phi_1(\mathbf{r}_1,\mathbf{r}_2)
=
\mathbf{r}_1\cdot\mathbf{r}_1-R_1^2=0,
$$

$$
\Phi_2(\mathbf{r}_1,\mathbf{r}_2)
=
(\mathbf{r}_2-\mathbf{r}_1)\cdot(\mathbf{r}_2-\mathbf{r}_1)-R_2^2=0.
$$

These constraints enforce the fixed rod lengths and reduce the number of independent configurational degrees of freedom from six to four.

Differentiating the constraints with respect to time gives the corresponding velocity constraints,

$$
\mathbf{r}_1\cdot\dot{\mathbf{r}}_1=0,
$$

$$
(\mathbf{r}_2-\mathbf{r}_1)\cdot
(\dot{\mathbf{r}}_2-\dot{\mathbf{r}}_1)=0.
$$

The velocity constraints require the motion to remain tangent to the constraint manifold. Differentiating once more yields acceleration-level conditions, which are incorporated into the constrained equations of motion.

For the Cartesian state

$$
\mathbf{x}=
\begin{bmatrix}
\mathbf{r}_1\\
\mathbf{r}_2
\end{bmatrix},
\qquad
\mathbf{z}=
\begin{bmatrix}
\mathbf{x}\\
\dot{\mathbf{x}}
\end{bmatrix},
$$

the mechanical Lagrangian takes the form

$$
L(\mathbf{x},\dot{\mathbf{x}})
=
T(\dot{\mathbf{x}})-V(\mathbf{x}),
$$

where the kinetic energy is

$$
T=
\frac{1}{2}m_1\|\dot{\mathbf{r}}_1\|^2+
\frac{1}{2}m_2\|\dot{\mathbf{r}}_2\|^2.
$$

The constrained EOMs can be expressed using Lagrange multipliers,

$$
M\ddot{\mathbf{x}}
+
\nabla_{\mathbf{x}}V
=
J(\mathbf{x})^\mathsf{T}\boldsymbol{\lambda},
$$

where $M$ is the Cartesian mass matrix, $J(\mathbf{x})$ is the Jacobian of the holonomic constraints, and $\boldsymbol{\lambda}$ represents the constraint forces. The multipliers are determined so that the resulting accelerations satisfy the differentiated constraints.

#### Adaptation to the CLNN Framework

The DSP was implemented using the Cartesian rigid-body representation provided by the CLNN framework. The two pendulum masses and their connections are encoded through the system's body graph, while the constraint structure specifies the geometric relationships that must be maintained during integration.

In the implemented architecture, the NN parameterizes the potential energy as a function of Cartesian positions,

$$
V_\theta=V_\theta(\mathbf{x}),
$$

while the constrained dynamics framework supplies the mechanical structure required to compute accelerations consistent with the prescribed constraints.

Consequently, the model does not need to learn the rod lengths or the associated constraint forces as arbitrary functions of the state. Instead, these relationships are incorporated into the EOMs, and the learned potential determines the remaining state-dependent forces.

Before evaluating the NN models, an independent Cartesian constrained formulation was constructed and compared against the analytical DSP equations. This provided a numerical consistency check between the angular-coordinate reference dynamics and the Cartesian representation used for the CLNN investigation.

#### Experimental Design

Three trained CLNN models were evaluated to investigate the effects of training-data distribution and training effort.

| Experiment | Sampling strategy | Training chunks | Epochs | Optimization updates |
|:---:|---|---:|---:|---:|
| **Exp1** | Random | 200 | 500 | 2,000 |
| **Exp2** | Geometrically stratified | 200 | 500 | 2,000 |
| **Exp3** | Random | 800 | 2,000 | 8,000 |

All three models use the same NN architecture, consisting of three hidden layers with 256 units per layer and 133,635 trainable parameters.

The comparisons serve two distinct purposes:

1. **Exp1 versus Exp2:** Examines the effect of training-data distribution while holding the number of training chunks and optimization updates fixed.
2. **Exp1 versus Exp3:** Examines the combined effect of increasing the training-data quantity and optimization duration.

The second comparison does not isolate these two factors independently, since both are modified simultaneously.

Model performance is evaluated against 50 analytical reference trajectories, each containing 100 time points separated by $\Delta t=0.03\,\mathrm{s}$, corresponding to a prediction horizon of $2.97\,\mathrm{s}$.

The principal diagnostics are training-data geometric coverage, trajectory prediction accuracy, and preservation of the DSP's mechanical constraints. Particular attention is given to the distinction between typical prediction accuracy and occasional large trajectory errors, as well as the influence of nonlinear dynamical sensitivity on the evolution of those errors.


In [ ]:
def cartesian_position(q, R1=1., R2=1.):
    theta1, theta2, phi1, phi2 = q
    r1 = R1 * np.array([np.sin(theta1) * np.cos(phi1),
                        np.sin(theta1) * np.sin(phi1),
                        np.cos(theta1)])
    r2 = r1 + R2 * np.array([np.sin(theta2) * np.cos(phi2),
                             np.sin(theta2) * np.sin(phi2),
                             np.cos(theta2)])

    return np.concatenate([r1, r2])

def cartesian_velocity(q, qdot, R1=1., R2=1.):
    theta1, theta2, phi1, phi2 = q
    theta1_dot, theta2_dot, phi1_dot, phi2_dot = qdot

    v1 = R1 * np.array([theta1_dot * np.cos(theta1) * np.cos(phi1) - phi1_dot * np.sin(theta1) * np.sin(phi1),
                        theta1_dot * np.cos(theta1) * np.sin(phi1) + phi1_dot * np.sin(theta1) * np.cos(phi1),
                        -theta1_dot * np.sin(theta1)])
    v2 = v1 + R2 * np.array([theta2_dot * np.cos(theta2) * np.cos(phi2) - phi2_dot * np.sin(theta2) * np.sin(phi2),
                             theta2_dot * np.cos(theta2) * np.sin(phi2) + phi2_dot * np.sin(theta2) * np.cos(phi2),
                             -theta2_dot * np.sin(theta2)])
    return np.concatenate([v1, v2])

def cartesian_state(q, qdot, R1=1., R2=1.):
    x = cartesian_position(q, R1=R1, R2=R2)
    v = cartesian_velocity(q, qdot, R1=R1, R2=R2)

    return np.concatenate([x, v])

def gradient_potential(x, m1=1., m2=1., g=9.8):
    return np.array([0.0, 0.0, -m1 * g, 0.0, 0.0, -m2 * g])

def Minv(y, m1=1., m2=1.):
    y1 = y[:3] / m1
    y2 = y[3:] / m2

    return np.concatenate([y1, y2])

def constraints(x, R1=1., R2=1.):
    r1 = x[:3]
    r2 = x[3:]

    return np.array([
        np.dot(r1, r1) - R1**2,
        np.dot(r2 - r1, r2 - r1) - R2**2
    ])

def gradient_constraint(x):
    r1 = x[:3]
    r2 = x[3:]

    G = np.zeros((6, 2))

    # Phi_1 = r1.r1 - R1^2
    G[:3, 0] = 2.0 * r1

    # Phi_2 = (r2-r1).(r2-r1) - R2^2
    G[:3, 1] = -2.0 * (r2 - r1)
    G[3:, 1] = 2.0 * (r2 - r1)

    return G

def GdotTv(v):
    v1 = v[:3]
    v2 = v[3:]

    return np.array([2.0 * np.dot(v1, v1), 2.0 * np.dot(v2 - v1, v2 - v1)])

def unconstrained_acceleration(x, m1=1., m2=1., g=9.8):
    dV = gradient_potential(x, m1=m1, m2=m2, g=g)

    return -Minv(dV, m1=m1, m2=m2)

def constrained_acceleration(x, v, m1=1., m2=1., g=9.8):
    f = unconstrained_acceleration(x, m1=m1, m2=m2, g=g)
    G = gradient_constraint(x)
    MinvG = np.column_stack([Minv(G[:, i], m1=m1, m2=m2) for i in range(G.shape[1])])
    GTMinvG = G.T @ MinvG
    rhs = (G.T @ f + GdotTv(v))
    lam = np.linalg.solve(GTMinvG, rhs)
    correction = MinvG @ lam
    
    return f - correction

def cartesian_dynamics(t, state, m1=1., m2=1., g=9.8):
    x = state[:6]
    v = state[6:]
    a = constrained_acceleration(x, v, m1=m1, m2=m2, g=g)

    return np.concatenate([v, a])

In [ ]:
q0 = np.array([3*np.pi/7, 3*np.pi/4, 3*np.pi/6, 3*np.pi/5])
qdot0 = np.array([0.2, -0.1, 0.3, 0.4])

state0 = cartesian_state(q0, qdot0)

x0 = state0[:6]
v0 = state0[6:]

print("position constraint:")
print(constraints(x0))

G0 = gradient_constraint(x0)

print("velocity constraint:")
print(G0.T @ v0)

a0 = constrained_acceleration(x0, v0)

print("acceleration constraint:")
print(
    G0.T @ a0
    + GdotTv(v0)
)

In [ ]:
t_eval = np.linspace(0.0, 10.0, 1001)

theta1_traj, u1_traj, theta2_traj, u2_traj, phi1_traj, v1_traj, phi2_traj, v2_traj = p1.solve_ode(
    t_eval,
    q0[0], qdot0[0],
    q0[1], qdot0[1],
    q0[2], qdot0[2],
    q0[3], qdot0[3],
    abserr=1e-9,
    relerr=1e-9
)

q_traj = np.column_stack([
    theta1_traj, theta2_traj, phi1_traj, phi2_traj
])

qdot_traj = np.column_stack([
    u1_traj, u2_traj, v1_traj, v2_traj
])

In [ ]:
x_analytical = np.array([cartesian_position(q) for q in q_traj])
v_analytical = np.array([cartesian_velocity(q, qdot) for q, qdot in zip(q_traj, qdot_traj)])

sol_cart = solve_ivp(cartesian_dynamics, (t_eval[0], t_eval[-1]), state0, t_eval=t_eval, rtol=1e-9, atol=1e-9)

x_cart = sol_cart.y[:6].T
v_cart = sol_cart.y[6:].T

In [ ]:
position_error = np.linalg.norm(x_cart - x_analytical, axis=1)

plt.figure(figsize=(8, 5))
plt.plot(t_eval, position_error)
plt.xlabel("Time")
plt.ylabel(r"$||x_{\mathrm{cartesian}} - x_{\mathrm{analytical}}||$")
plt.title("Cartesian Constrained vs Analytical Trajectory")
plt.grid()
plt.show()

print("Max position error:", np.max(position_error))
print("Mean position error:", np.mean(position_error))

### Training Procedure and Model Evaluation

The CLNN models were trained using trajectory data generated from the analytical DSP dynamics. Each training sample consists of a short segment of Cartesian position and velocity states, allowing the model to learn the system's evolution while incorporating its prescribed geometric constraints.

The three experiments were developed sequentially in response to the observed prediction performance. Although they share the same CLNN architecture, each subsequent experiment investigates a different potential means of improving accuracy: geometric training-data coverage in Exp2 and increased training data quantity and optimization effort in Exp3.

#### Training-Data Selection

Training trajectories were sampled in short temporal segments, referred to as *training chunks*. Two sampling strategies were investigated:

- **Random sampling (Exp1 and Exp3):** Training chunks were selected without explicitly balancing their distribution across the DSP's configuration space.
- **Geometrically stratified sampling (Exp2):** Training chunks were selected to provide greater representation of configurations involving larger first-bob polar angles.

The distinction is important because the DSP does not visit every region of its configuration space equally frequently under a given distribution of initial conditions. Consequently, randomly sampled training data may provide extensive coverage of commonly encountered configurations while underrepresenting less frequently visited regions.

Stratified sampling attempts to reduce this imbalance without necessarily increasing the total number of training samples.

The first comparison, Exp1 versus Exp2, isolates the effect of the sampling strategy under equal training-data quantity and optimization duration. The second comparison, Exp1 versus Exp3, investigates the combined effect of using four times as many training chunks and four times as many optimization updates.

#### Model Training

All three CLNN models employ the same three-hidden-layer neural network with 256 units per layer and 133,635 trainable parameters.

The learned potential-energy representation is incorporated into the constrained Lagrangian EOMs. During training, the network parameters are optimized using simulated trajectory data, while the known mechanical constraints remain part of the dynamical formulation.

Exp1 and Exp2 were trained for 500 epochs and 2,000 optimization updates, whereas Exp3 was trained for 2,000 epochs and 8,000 updates. The resulting model parameters were saved as checkpoints and subsequently loaded for evaluation.

Because the architecture is held fixed, differences in performance can be examined in relation to the experimental changes in sampling strategy and training effort. However, the Exp1–Exp3 comparison cannot independently distinguish the effects of dataset size and optimization duration.

#### Evaluation Procedure

The trained models were evaluated against a shared set of 50 analytical reference trajectories. Each trajectory contains 100 states sampled at intervals of

$$
\Delta t = 0.03\,\mathrm{s},
$$

corresponding to a prediction horizon of

$$
T = 2.97\,\mathrm{s}.
$$

For each experiment, the CLNN was initialized from the same analytical initial states and numerically integrated using the adaptive Dormand–Prince method (`dopri5`) with relative and absolute tolerances of $10^{-7}$.

The resulting predictions were compared against the analytical trajectories using two principal error measures.

**Trajectory mean absolute error (MAE)** measures the average absolute discrepancy across all test trajectories, time points, and Cartesian state components:

$$
\mathrm{MAE}
=
\frac{1}{NTD}
\sum_{i=1}^{N}
\sum_{j=1}^{T}
\sum_{k=1}^{D}
\left|
\hat{z}_{ijk}-z_{ijk}
\right|,
$$

where $N=50$ is the number of trajectories, $T=100$ is the number of sampled time points, and $D=12$ is the number of Cartesian state components.

To examine how prediction accuracy evolves over time, instantaneous MAE is also calculated for each trajectory:

$$
\mathrm{MAE}_i(t)
=
\frac{1}{D}
\sum_{k=1}^{D}
\left|
\hat{z}_{ik}(t)-z_{ik}(t)
\right|.
$$

**Relative state-space error** measures the discrepancy between predicted and analytical states, normalized by the sum of their Euclidean norms:

$$
E_i^{\mathrm{rel}}(t)
=
\frac{
\left\|
\hat{\mathbf{z}}_i(t)-\mathbf{z}_i(t)
\right\|_2
}{
\left\|
\hat{\mathbf{z}}_i(t)
\right\|_2
+
\left\|
\mathbf{z}_i(t)
\right\|_2
}.
$$

Here, $\hat{\mathbf{z}}_i(t)$ and $\mathbf{z}_i(t)$ represent the predicted and analytical 12-dimensional Cartesian state vectors for trajectory $i$ at time $t$, respectively.

This symmetric normalization provides a dimensionless measure of prediction error, facilitating comparisons across trajectories with different state magnitudes. Unlike absolute error, relative state-space error accounts for the overall magnitude of the states being compared.

For both instantaneous MAE and relative state-space error, the mean across all 50 test trajectories is plotted as a function of time. Shaded regions represent 95% bootstrap confidence intervals for the estimated mean, following the plotting methodology used by Finzi et al. Both quantities are displayed on logarithmic vertical axes to illustrate error growth over multiple orders of magnitude.

These metrics capture complementary aspects of prediction accuracy. MAE measures the absolute discrepancy between predicted and analytical states, while relative state-space error measures this discrepancy in relation to the magnitudes of the states themselves. Examining their time evolution reveals how prediction errors accumulate throughout the evaluation horizon and how the three experiments differ in their average predictive performance.

In [ ]:
import wandb
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from pytorch_lightning import Trainer
from pl_trainer import DynamicsModel, SaveTestLogCallback
import os

In [ ]:
import os
import torch
from pathlib import Path

# Resolve the repository root without hard-coding a local path. In order:
#   1. the DSP_REPO_ROOT environment variable, if set;
#   2. `project_root` from the setup cell at the top of this notebook;
#   3. the nearest enclosing directory holding both `biases/` and `experiments/`.
def find_repo_root():
    env_root = os.environ.get("DSP_REPO_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in [globals().get("project_root"), cwd, *cwd.parents]:
        if candidate is None:
            continue
        candidate = Path(candidate).resolve()
        if (candidate / "biases").is_dir() and (candidate / "experiments").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root. Run this notebook from the repo's "
        "notebooks/ directory, or set the DSP_REPO_ROOT environment variable."
    )

REPO_ROOT = find_repo_root()
WANDB_ROOT = REPO_ROOT / "experiments" / "DoubleSphericalPendulum" / "CLNN" / "wandb"

# W&B run ids for the three CLNN experiments. The run directories are committed
# under experiments/, so these resolve on any checkout of the repository.
RUN_IDS = {
    "Exp1": "kj5fk851",
    "Exp2": "v1vp8r4x",
    "Exp3": "ii31lbgd",
}

def run_files_dir(run_id):
    matches = sorted(WANDB_ROOT.glob(f"*run-*-{run_id}/files"))
    if not matches:
        raise FileNotFoundError(
            f"No W&B run directory for id '{run_id}' under "
            f"{WANDB_ROOT.relative_to(REPO_ROOT)}. Train the model first, or "
            f"check out the committed experiments/ tree."
        )
    return matches[-1]

run_dirs = {name: run_files_dir(run_id) for name, run_id in RUN_IDS.items()}

# Load the saved evaluation results.
test_logs = {}

for name, run_dir in run_dirs.items():
    log_path = run_dir / "test_log.pt"
    test_logs[name] = torch.load(
        log_path,
        map_location="cpu",
        weights_only=False
    )

In [ ]:
import torch

labels = []
series = []

for network_class, test_log in test_logs.items():

    pred_zts = test_log["pred_zts"]
    true_zts = test_log["true_zts"]
    pert_zts = test_log["pert_zts"]

    body = "DSP"

    rel_err_pred_true = (pred_zts - true_zts).pow(2).sum((2, 3, 4)).sqrt() / (pred_zts.pow(2).sum((2,3,4)).sqrt() + true_zts.pow(2).sum((2,3,4)).sqrt())
    rel_err_pert_true = (pert_zts - true_zts).pow(2).sum((2, 3, 4)).sqrt() / (pert_zts.pow(2).sum((2,3,4)).sqrt() + true_zts.pow(2).sum((2,3,4)).sqrt())

    #rel_err_pred_true = ((pred_zts - true_zts).pow(2).sum((2, 3, 4)) / (true_zts).pow(2).sum((2,3,4))).sqrt()
    #rel_err_pert_true = ((pert_zts - true_zts).pow(2).sum((2, 3, 4)) / (true_zts).pow(2).sum((2,3,4))).sqrt()

    #avg_rel_err_pred = rel_err_pred_true.mean(0).numpy()
    #avg_rel_err_pert = rel_err_pert_true.mean(0).numpy()
    avg_rel_err_pred = rel_err_pred_true.cpu().numpy()
    avg_rel_err_pert = rel_err_pert_true.cpu().numpy()

    # turn into list of series from batch of series
    series += avg_rel_err_pred.tolist()
    labels += [(body, network_class, "avg_rel_err_pred", batch) for batch in range(len(avg_rel_err_pred))]

    series += avg_rel_err_pert.tolist()
    labels += [(body, network_class, "avg_rel_err_pert", batch) for batch in range(len(avg_rel_err_pert))]

    # Instantaneous mean absolute error (MAE)
    mae_pred_true = (pred_zts - true_zts).abs().mean(dim=(2, 3, 4))
    mae_pert_true = (pert_zts - true_zts).abs().mean(dim=(2, 3, 4))

    avg_mae_pred = mae_pred_true.cpu().numpy()
    avg_mae_pert = mae_pert_true.cpu().numpy()

    series += avg_mae_pred.tolist()
    labels += [(body, network_class, "avg_mae_pred", batch) for batch in range(len(avg_mae_pred))]

    series += avg_mae_pert.tolist()
    labels += [(body, network_class, "avg_mae_pert", batch) for batch in range(len(avg_mae_pert))]

In [ ]:
#index = pd.MultiIndex.from_tuples(labels, names =('Body', 'Network', "series"))
levels = ['Body', 'Network', "Series", "Batch"]
index = pd.MultiIndex.from_tuples(labels, names=levels)
df = pd.DataFrame(series, index=index).reset_index()

In [ ]:
df.head()

In [ ]:
dt = 0.03
# chunk_len = run.config["chunk_len"] * dt
n_timesteps = test_logs["Exp1"]["true_zts"].shape[1]
integration_time = (n_timesteps - 1) * dt


In [ ]:
df = df.rename(columns={i: dt * i for i in range(n_timesteps)})

In [ ]:
df = pd.melt(df, id_vars=levels, var_name="time")

In [ ]:
df = df.sort_values(by=levels)

In [ ]:
sns.set(font_scale=1.75)
sns.set_style('whitegrid')
#colors = [(147/255, 205/255, 221/255), "#6A0078", (96/255,74/255,123/255), "#8E6100"]
colors = ["#00abdf", "#00058A", "#6A0078", (96/255,74/255,123/255), "#8E6100"]
sns.set_palette(sns.color_palette(colors))

from matplotlib import rc
rc('text', usetex=False)
# rc('text.latex', preamble=[r'\usepackage{sansmath}', r'\sansmath']) #r'\usepackage{DejaVuSans}'AnimationPendulumPlot
rc('font',**{'family':'sans-serif','sans-serif':['DejaVu Sans']})
rc('xtick.major', pad=12)
rc('ytick.major', pad=12)
rc('grid', linewidth=1.3)
#import matplotlib as mpl
#mpl.rcParams['figure.dpi'] = 1000

In [ ]:
df["Body"].unique()

In [ ]:
n_plots = len(df["Body"].unique())
fig, axes = plt.subplots(ncols=n_plots, figsize=(4 * n_plots, 5), squeeze=False)
axes = axes.ravel()

i = 0
for value, ax in zip(df["Body"].unique(), axes):
    data = df.loc[(df["Series"] == "avg_rel_err_pred") & (df["Body"] == value)]
    # ax.axvline(chunk_len, label=r"$T_{minibatch}$", color="k", linestyle="--", linewidth=1.0)
    sns.lineplot(data=data, x="time", y="value", hue="Network", ax=ax, errorbar=('ci', 95))
    
    ax.set(yscale="log", xlabel="Time (seconds)", title=f"{value}", ylim=(1e-5, 1))
    ax.set(ylabel="")
    
    if i > 0: # hack
        ax.set(yticklabels=[])
    
    ax.legend().remove()
    i += 1

axes[0].set(ylabel="Relative error")
handles, labels = axes[-1].get_legend_handles_labels()
# handles = handles[0:1] + handles[2:]
# labels = labels[0:1] + labels[2:] # Get rid of legend title
fig.subplots_adjust(bottom=0.35, wspace=0.05, hspace=0.25)
#fig.subplots_adjust(bottom=0.4, wspace=0.05, hspace=0.25)
fig.legend(handles=handles, labels=labels, bbox_to_anchor=(0.5, 0.02), loc="lower center", ncol=len(labels), frameon=False)

In [ ]:
n_plots = len(df["Body"].unique())
fig, axes = plt.subplots(ncols=n_plots, figsize=(4 * n_plots, 5), squeeze=False)
axes = axes.ravel()

i = 0
for value, ax in zip(df["Body"].unique(), axes):
    data = df.loc[(df["Series"] == "avg_mae_pred") & (df["Body"] == value)]
    # ax.axvline(chunk_len, label=r"$T_{minibatch}$", color="k", linestyle="--", linewidth=1.0)
    sns.lineplot(data=data, x="time", y="value", hue="Network", ax=ax, errorbar=('ci', 95))
    
    ax.set(yscale="log", xlabel="Time (seconds)", title=f"{value}", ylim=(1e-5, 1))
    ax.set(ylabel="")
    
    if i > 0: # hack
        ax.set(yticklabels=[])
    
    ax.legend().remove()
    i += 1

axes[0].set(ylabel="Mean absolute error")
handles, labels = axes[-1].get_legend_handles_labels()
# handles = handles[0:1] + handles[2:]
# labels = labels[0:1] + labels[2:] # Get rid of legend title
fig.subplots_adjust(bottom=0.35, wspace=0.05, hspace=0.25)
#fig.subplots_adjust(bottom=0.4, wspace=0.05, hspace=0.25)
fig.legend(handles=handles, labels=labels, bbox_to_anchor=(0.5, 0.02), loc="lower center", ncol=len(labels), frameon=False)

### Experimental Results and Analysis

The predictive performance of the three CLNN experiments was evaluated using relative state-space error and instantaneous MAE. Both metrics were calculated over 50 analytical reference trajectories spanning a prediction horizon of 2.97 seconds. The resulting comparisons provide insight into the effects of training-data selection, dataset size, and optimization duration on the accuracy of the learned DSP dynamics.

#### Trajectory Prediction Accuracy

The relative state-space error and instantaneous MAE plots reveal substantial differences between the three experiments.

Exp1 exhibits the fastest growth in prediction error, with both metrics increasing rapidly during the early stages of integration. Its mean error remains considerably higher than those of Exp2 and Exp3 throughout most of the evaluation horizon. The corresponding 95% bootstrap confidence intervals also become noticeably wider as time progresses, indicating greater uncertainty in the estimated mean error across the test trajectories.

In contrast, Exp2 and Exp3 maintain substantially lower prediction errors. Their mean relative-error and MAE curves exhibit comparatively gradual growth, remaining within similar ranges over the evaluation interval. Although their curves occasionally intersect, their confidence intervals overlap considerably, preventing a clear conclusion that either experiment consistently outperforms the other.

The agreement between the two error metrics is significant. Relative state-space error normalizes the prediction discrepancy by the magnitudes of the compared states, whereas MAE measures the absolute componentwise discrepancy. The similar trends observed in both plots indicate that the differences in model performance are not simply consequences of the normalization procedure.

Overall, the results suggest that the training modifications introduced in Exp2 and Exp3 substantially improve average finite-horizon prediction accuracy relative to Exp1.

#### Influence of Training-Data Coverage

One possible explanation for the observed performance differences is the geometric distribution of the training data.

For Exp1, approximately 95% of the training chunks remain below a maximum first-bob polar angle of $90^\circ$. Exp3 exhibits a similar distribution, with approximately 94.6% of its training chunks concentrated below this threshold. Although Exp3 contains four times as many training chunks, its relative distribution across the configuration space remains strongly concentrated in the same region.

Exp2 differs substantially. Through geometrically stratified sampling, approximately 50% of its training chunks reach or exceed $90^\circ$, providing greater representation of configurations that are comparatively uncommon in the randomly sampled datasets.

This distinction is important because a learned dynamical model must generalize beyond the configurations most frequently encountered during training. When particular regions of configuration space are underrepresented, the model may learn an accurate approximation of the dynamics in commonly sampled regions while exhibiting larger errors elsewhere.

The improved prediction accuracy of Exp2 relative to Exp1 is therefore consistent with the hypothesis that broader geometric coverage improves the learned dynamics. Because these experiments use the same architecture, training-data quantity, and optimization duration, their comparison provides evidence that the distribution of training samples can be an important determinant of predictive performance.

#### Training-Data Quantity and Optimization Duration

The comparison between Exp1 and Exp3 provides a complementary perspective.

Unlike Exp2, which modifies the sampling strategy, Exp3 retains random sampling while increasing both the number of training chunks and the number of optimization updates by a factor of four.

Exp3 achieves substantially lower mean relative-error and MAE values than Exp1, indicating that increased training data and optimization effort can improve prediction accuracy even without explicitly balancing geometric coverage.

Nevertheless, the training-data coverage distributions of Exp1 and Exp3 remain similar. This suggests that Exp3's improved performance does not necessarily result from a major redistribution of training samples across configuration space. Instead, the larger dataset may provide denser sampling of frequently encountered configurations, while the additional optimization updates allow the network to fit the sampled dynamics more effectively.

Because dataset size and optimization duration were increased simultaneously, their individual contributions cannot be isolated from this comparison.

The similar predictive performance of Exp2 and Exp3 is particularly noteworthy. Exp2 achieves competitive results through geometrically stratified sampling without the fourfold increase in training-data quantity and optimization updates used in Exp3. This suggests that improving the distribution of training data may offer an efficient alternative to increasing training effort.

However, a more systematic investigation would be required to establish the relative computational efficiency of these approaches.

#### Interpretation and Limitations

The experimental results highlight the importance of both training-data coverage and training effort in learning constrained mechanical dynamics.

Exp1 demonstrates that maintaining the prescribed mechanical constraints does not, by itself, guarantee accurate trajectory predictions. Although the CLNN formulation incorporates the DSP's geometric constraints, the learned dynamical representation must still approximate the system's evolution accurately.

The increasing prediction errors also illustrate a general challenge associated with numerical integration of learned nonlinear dynamical systems. Small discrepancies in the learned dynamics can accumulate over time, producing progressively larger differences between predicted and analytical trajectories. The extent of this accumulation depends on both the accuracy of the learned model and the sensitivity of the underlying dynamics.

The confidence intervals provide additional information about the reliability of the estimated mean prediction errors. In particular, the wider intervals observed for Exp1 suggest greater variability in prediction accuracy across the evaluation trajectories. However, these intervals represent uncertainty in the mean rather than the full distribution of trajectory errors, and should not be interpreted as bounds on individual predictions.

Furthermore, the experiments evaluate prediction accuracy over a finite 2.97-second horizon. Their results therefore do not establish long-term stability or accuracy over substantially longer integrations.

#### Summary

The three CLNN experiments demonstrate that predictive accuracy depends strongly on the training procedure, even when the model architecture and constrained dynamical formulation are held fixed.

Exp1 exhibits the largest average prediction errors, while Exp2 and Exp3 achieve substantially better agreement with the analytical DSP trajectories. The Exp1–Exp2 comparison suggests that geometrically stratified sampling can improve generalization by increasing representation of less frequently encountered configurations. Meanwhile, the Exp1–Exp3 comparison demonstrates that increasing training-data quantity and optimization effort can also improve predictive accuracy.

Taken together, these findings suggest that the distribution of training samples is an important consideration in physics-informed machine learning. For the DSP, strategically improving geometric coverage appears capable of producing prediction accuracy comparable to that obtained through substantially greater training effort.

The results support the use of constrained neural architectures as an alternative representation of the DSP dynamics, while emphasizing that explicit enforcement of mechanical constraints and accurate learning of the underlying dynamics are distinct requirements.

# 5 Hamiltonian vs. Lagrangian

Classical mechanics admits several mathematically equivalent formulations, each offering a different perspective on the dynamics of physical systems. Thus far, we have studied the DSP primarily through **Lagrangian mechanics**, deriving its equations of motion from the principle of stationary action and investigating its behavior through numerical integration.

An alternative, equally fundamental formulation is **Hamiltonian mechanics**, which recasts the equations of motion as a system of first-order differential equations in terms of generalized coordinates and their conjugate momenta.

Although both formulations describe the same underlying physics, the Hamiltonian approach provides a particularly powerful framework for studying the geometric structure of dynamical systems. In particular, it introduces the concept of **phase space**, where the evolution of a system can be interpreted as a continuous flow through a space of possible states. This perspective is central to the study of nonlinear dynamics, chaos, statistical mechanics, and, more broadly, the development of quantum mechanics.

For the DSP, Hamiltonian mechanics offers an opportunity not only to verify the consistency of our previous Lagrangian results but also to explore deeper properties of its chaotic dynamics, including the preservation of phase-space volume.

## 5.1 Exploring the Hamiltonian approach

In Lagrangian mechanics, the dynamics of a system are described by the Lagrangian

$$
\mathcal{L}(\mathbf{q},\dot{\mathbf{q}},t) = T - U,
$$

where $T$ and $U$ denote the kinetic and potential energies, respectively. The generalized coordinates $\mathbf{q}$ and generalized velocities $\dot{\mathbf{q}}$ satisfy the Euler–Lagrange equations:

$$
\boxed{
\frac{d}{dt}\left(\frac{\partial\mathcal{L}}{\partial\dot q_i}\right)
-\frac{\partial\mathcal{L}}{\partial q_i}=0.
}
$$

For the double spherical pendulum, the four generalized coordinates are

$$
\mathbf{q}=
\begin{pmatrix}
\theta_1\\
\theta_2\\
\phi_1\\
\phi_2
\end{pmatrix}.
$$

Hamiltonian mechanics introduces a new set of variables known as the **canonical momenta**, defined by

$$
\boxed{
p_i=\frac{\partial\mathcal{L}}{\partial\dot q_i}.
}
$$

These momenta are conjugate to their corresponding generalized coordinates and are not, in general, equivalent to the generalized velocities or simply the products of mass and velocity.

Provided the relationship between momenta and velocities is invertible, the Hamiltonian is constructed through the **Legendre transformation**:

$$
\boxed{
\mathcal{H} = \sum_{i=1}^{n} p_i \dot{q}_i - \mathcal{L},
}
$$

where the generalized velocities are expressed in terms of the coordinates and momenta.

The resulting equations of motion are Hamilton's canonical equations:

$$
\boxed{
\dot q_i=\frac{\partial\mathcal{H}}{\partial p_i},
\qquad
\dot p_i=-\frac{\partial\mathcal{H}}{\partial q_i}.
}
$$

Unlike the second-order E–L equations, Hamilton's equations form a coupled system of first-order differential equations. For the DSP, the four generalized coordinates and four canonical momenta define an **eight-dimensional canonical phase space**:

$$
\mathbf{z}=
\begin{pmatrix}
\mathbf{q}\\
\mathbf{p}
\end{pmatrix}
=
\begin{pmatrix}
\theta_1\\
\theta_2\\
\phi_1\\
\phi_2\\
p_{\theta_1}\\
p_{\theta_2}\\
p_{\phi_1}\\
p_{\phi_2}
\end{pmatrix}.
$$

Each point in this phase space represents a complete instantaneous state of the system, and Hamilton's equations determine its subsequent evolution.

For the autonomous, conservative DSP considered here, the Hamiltonian coincides with the total energy, $\mathcal{H} = T + V$, and is conserved along exact trajectories.

### Objectives

In this subsection, we will reformulate the DSP using Hamiltonian mechanics and demonstrate its equivalence to the previously derived Lagrangian formulation.

Specifically, we will:

1. **Derive the canonical momenta** from the DSP Lagrangian using $p_i = \partial\mathcal{L}/\partial\dot q_i$.
2. **Invert the momentum–velocity relations** using matrix algebra to express the generalized velocities in terms of generalized coordinates and canonical momenta.
3. **Construct the Hamiltonian** through the Legendre transformation and examine its relationship to the total energy. 
4. **Derive Hamilton's canonical equations** and implement them numerically as an eight-dimensional system of first-order differential equations.
5. **Demonstrate numerical equivalence** by integrating both formulations from identical physical initial conditions and comparing the resulting Cartesian trajectories. 

Having established the equivalence of the two formulations, we will then turn to the geometric interpretation of Hamiltonian dynamics in Section 5.2, with particular emphasis on Liouville's theorem, phase-space volume preservation, and their implications for chaotic motion.

In [ ]:
import sympy as sp

# Defining symbols
t, g = sp.symbols('t g')
m1, m2 = sp.symbols('m1 m2')
R1, R2 = sp.symbols('R1 R2')

# Defining functions
theta1, theta2, phi1, phi2 = sp.symbols(r'\theta_1, \theta_2, \phi_1, \phi_2', cls=sp.Function)

theta1 = theta1(t)
theta2 = theta2(t)
phi1 = phi1(t)
phi2 = phi2(t)

theta1_dot = sp.diff(theta1, t)
theta2_dot = sp.diff(theta2, t)
phi1_dot = sp.diff(phi1, t)
phi2_dot = sp.diff(phi2, t)
theta1_ddot = sp.diff(theta1_dot, t)
theta2_ddot = sp.diff(theta2_dot, t)
phi1_ddot = sp.diff(phi1_dot, t)
phi2_ddot = sp.diff(phi2_dot, t)

x1 = R1 * sp.sin(theta1) * sp.cos(phi1)
y1 = R1 * sp.sin(theta1) * sp.sin(phi1)
z1 = R1 * sp.cos(theta1)
x2 = x1 + R2 * sp.sin(theta2) * sp.cos(phi2)
y2 = y1 + R2 * sp.sin(theta2) * sp.sin(phi2)
z2 = z1 + R2 * sp.cos(theta2)

x1_num = sp.lambdify((theta1, theta2, phi1, phi2, R1, R2), x1)
y1_num = sp.lambdify((theta1, theta2, phi1, phi2, R1, R2), y1)
z1_num = sp.lambdify((theta1, theta2, phi1, phi2, R1, R2), z1)
x2_num = sp.lambdify((theta1, theta2, phi1, phi2, R1, R2), x2)
y2_num = sp.lambdify((theta1, theta2, phi1, phi2, R1, R2), y2)
z2_num = sp.lambdify((theta1, theta2, phi1, phi2, R1, R2), z2)

T1 = sp.Rational(1, 2) * m1 * (sp.diff(x1, t)**2 + sp.diff(y1, t)**2 + sp.diff(z1, t)**2)
T2 = sp.Rational(1, 2) * m2 * (sp.diff(x2, t)**2 + sp.diff(y2, t)**2 + sp.diff(z2, t)**2)
T = T1 + T2

U1 = -m1 * g * z1
U2 = -m2 * g * z2
U = U1 + U2

L = T - U

### Generalized Momenta

We begin by deriving the generalized momenta associated with the DSP's four generalized coordinates. 
By definition,

$$
p_i = \frac{\partial\mathcal{L}(\mathbf{q}, \dot{\mathbf{q}}, t)}{\partial\dot{q}_i}, \qquad [i = 1,\cdots, n]
$$

Unlike ordinary linear momentum, generalized momentum depends on the choice of generalized coordinates and the structure of the Lagrangian. 
For the DSP, the coupling bewteen the two pendulums means that each generalized momentum may depend on multiple generalized velocities. 

These momentum-velocity relationships provide the starting point for the Legendre transformation from the Lagrangian to the Hamiltonian formulation. 

In [ ]:
p_theta1 = sp.diff(L, theta1_dot).simplify()
p_theta2 = sp.diff(L, theta2_dot).simplify()
p_phi1 = sp.diff(L, phi1_dot).simplify()
p_phi2 = sp.diff(L, phi2_dot).simplify()

In [ ]:
momentum_args = (
    theta1, theta2, phi1, phi2,
    theta1_dot, theta2_dot, phi1_dot, phi2_dot,
    m1, m2, R1, R2
)

p_theta1_num = sp.lambdify(momentum_args, p_theta1, cse=True)
p_theta2_num = sp.lambdify(momentum_args, p_theta2, cse=True)
p_phi1_num = sp.lambdify(momentum_args, p_phi1, cse=True)
p_phi2_num = sp.lambdify(momentum_args, p_phi2, cse=True)

### Inverting the Momentum-Velocity Relations

To express the Hamiltonian entirely in terms of the canonical coordinates and momenta, we must first solve the momentum-velocity relations for the generalized velocities 

$$
\dot{\mathbf{q}} = \dot{\mathbf{q}}(\mathbf{q}, \mathbf{p}, t).
$$

This inversion is essential for constructing the Hamiltonian and evaluating the Hamilton equations numerically. 

In [ ]:
q1, q2, q3, q4 = sp.symbols(r'q1 q2 q3 q4')
p1, p2, p3, p4 = sp.symbols(r'p1 p2 p3 p4')

M, P = sp.linear_eq_to_matrix([p_theta1 - p1, p_theta2 - p2, p_phi1 - p3, p_phi2 - p4], [theta1_dot, theta2_dot, phi1_dot, phi2_dot])
vel = M.LUsolve(P)

pq_subs = {
    theta1_dot: vel[0],
    theta2_dot: vel[1], 
    phi1_dot: vel[2],
    phi2_dot: vel[3],
}

### Constructing the Hamiltonian

Having expressed the generalized velocities in terms of the canonical momenta, we construct the Hamiltonian through the Legendre transformation:

$$
\mathcal{H} = \sum_{i = 1}^4 p_i \dot{q}_i - \mathcal{L}.
$$

Substituting the inverted momentum-velocity relationships yields a Hamiltonian expressed entirely in terms of the canonical variables $(\mathbf{q}, \mathbf{p})$. 

For the conservative DSP, the resulting Hamiltonian cooresponds to the total energy:

$$
\mathcal{H} = T + V.
$$

This provides both an alternative description of the system's dynamics and a conserved quantity against which numerical solutions can be evaluated. 

In [ ]:
H = p1 * theta1_dot + p2 * theta2_dot + p3 * phi1_dot + p4 * phi2_dot - L
H = H.subs(pq_subs, simultaneous=True)
H = H.subs({
    theta1: q1, 
    theta2: q2, 
    phi1: q3,
    phi2: q4
}, simultaneous=True)

### Hamilton's Equations

The Hamiltonian determines the evolution of the DSP through Hamilton's equations:

$$
\dot{q}_i = \frac{\partial\mathcal{H}}{\partial p_i}, \qquad \dot{p}_i = -\frac{\partial\mathcal{H}}{\partial q_i}.
$$

For our four degree of freedom system, these produce eight coupled first-order differential equations: four governing the generalized coordinates and four governing their conjugate momenta.

We obtain these equations symbolically by differentiating the Hamiltonian, then convert the resulting expressions into numerical functions using `sympy.lambdify`.

The numerical equations are subsequently intergrated using `scipy.integrate.solve_ivp`, allowing us to simulate the DSP directly in phase space. 

In [ ]:
qdot1 = sp.diff(H, p1)
qdot2 = sp.diff(H, p2)
qdot3 = sp.diff(H, p3)
qdot4 = sp.diff(H, p4)
pdot1 = -sp.diff(H, q1)
pdot2 = -sp.diff(H, q2)
pdot3 = -sp.diff(H, q3)
pdot4 = -sp.diff(H, q4)

In [ ]:
vel_args = (t, g, m1, m2, R1, R2, q1, q2, q3, q4, p1, p2, p3, p4)

q1_dot_num = sp.lambdify(vel_args, qdot1, cse=True)
p1_dot_num = sp.lambdify(vel_args, pdot1, cse=True)
q2_dot_num = sp.lambdify(vel_args, qdot2, cse=True)
p2_dot_num = sp.lambdify(vel_args, pdot2, cse=True)

q3_dot_num = sp.lambdify(vel_args, qdot3, cse=True)
p3_dot_num = sp.lambdify(vel_args, pdot3, cse=True)
q4_dot_num = sp.lambdify(vel_args, qdot4, cse=True)
p4_dot_num = sp.lambdify(vel_args, pdot4, cse=True)

In [ ]:
class HamiltonianPendulum(Pendulum):

    def dy_dt(self, t, y):
        q1, q2, q3, q4, p1, p2, p3, p4 = y
        h_args = (t, self.g, self.m1, self.m2, self.R1, self.R2,
                            q1, q2, q3, q4, p1, p2, p3, p4)

        return np.array([
            q1_dot_num(*h_args), q2_dot_num(*h_args), q3_dot_num(*h_args), q4_dot_num(*h_args),
            p1_dot_num(*h_args), p2_dot_num(*h_args), p3_dot_num(*h_args), p4_dot_num(*h_args)
        ])

    def solve_ode(self, t_pts, q1_0, q2_0, q3_0, q4_0, p1_0, p2_0, p3_0, p4_0,
                  abserr=1e-8, relerr=1e-6, method='RK45'):
        y = [q1_0, q2_0, q3_0, q4_0, p1_0, p2_0, p3_0, p4_0]
        q1, q2, q3, q4, p1, p2, p3, p4 = solve_ivp(self.dy_dt, (t_pts[0], t_pts[-1]), y, t_eval=t_pts,
                                                   atol=abserr, rtol=relerr,
                                                   method=method).y

        return q1, q2, q3, q4, p1, p2, p3, p4

In [ ]:
class HamiltonianAnimationPendulumPlot(AnimationPendulumPlot):

    def add_pendulum(self, pend, q1_0=0., q2_0=0., q3_0=0., q4_0=0., p1_0=0., p2_0=0., p3_0=0., p4_0=0.):
        self.pend = pend
        q1, q2, q3, q4, p1, p2, p3, p4 = pend.solve_ode(self.t_pts, q1_0, q2_0, q3_0, q4_0,
                                                        p1_0, p2_0, p3_0, p4_0)
        
        self.theta1_list.append(q1)
        self.theta1_dot_list.append(p1)
        self.theta2_list.append(q2)
        self.theta2_dot_list.append(p2)
        self.phi1_list.append(q3)
        self.phi1_dot_list.append(p3)
        self.phi2_list.append(q4)
        self.phi2_dot_list.append(p4)

    def plot_setup(self, plot_start, plot_end):
        super().plot_setup(plot_start, plot_end)
        self.ax_3.set_ylabel(r'$p_i$')

In [ ]:
# Create a pendulum animation instance. 
pendulum_anim = HamiltonianAnimationPendulumPlot(phi_vs_t=True, 
                                                 state_space=True,
                                                physical_pend=True)

# Common plotting time (generate the full time here then use slices below)
t_start = 0.
t_end = 200.
delta_t = 0.01
pendulum_anim.t_pts_init(t_start, t_end, delta_t)

# Pendulum parameters
m1_param = 1.
m2_param = 1.
R1_param = 1.
R2_param = 1.
g_param = 9.8

# Instantiate a pendulum 
p10 = HamiltonianPendulum(m1=m1_param, m2=m2_param, R1=R1_param, R2=R2_param, g=g_param)

# Initial conditions specified
theta1_0 = np.pi / 3
u1_0 = 3.0
theta2_0 = np.pi / 3
u2_0 = 4.0
phi1_0 = 0.  # -np.pi / 2.
v1_0 = -3.0
phi2_0 = -np.pi
v2_0 = -4.0

ic_args = (
    theta1_0, theta2_0, phi1_0, phi2_0,
    u1_0, u2_0, v1_0, v2_0,
    m1_param, m2_param, R1_param, R2_param
)
p_theta1_0 = p_theta1_num(*ic_args)
p_theta2_0 = p_theta2_num(*ic_args)
p_phi1_0 = p_phi1_num(*ic_args)
p_phi2_0 = p_phi2_num(*ic_args)

# Add a pendulum to the animation plots; this solves the differential
#  equation for the full t_pts array, generating phi and phi_dot internally. 
pendulum_anim.add_pendulum(p10, q1_0=theta1_0, q2_0=theta2_0, q3_0=phi1_0, q4_0=phi2_0, p1_0=p_theta1_0, p2_0=p_theta2_0, p3_0=p_phi1_0, p4_0=p_phi2_0)

In [ ]:
plot_start = 0.   # time to begin plotting
plot_end = 30.    # time to end plotting
pendulum_anim.plot_setup(plot_start, plot_end)

# Start the animation (adjust skip and interval for a smooth plot at a 
#  useful speed)
skip = 2        # skip between time points (in units of delta_t) 
interval = 15.  # time between frames in milliseconds
# pendulum_anim.start_animation(skip, interval)
display(pendulum_anim.start_animation(skip, interval))

### Numerical Equivalence: Lagrangian vs. Hamiltonian Dynamics

Having implemented both formulations, we now investigate their numerical equivalence by comparing trajectories generated from identical physical ICs.

Because the Lagrangian and Hamiltonian formulations use different state variables, we first convert the initial generalized velocities into their corresponding canonical momenta using the momentum-velocity relationship above.

Rather than directly comparing generalized coordinates, we compare the Cartesian positions of the two pendulum masses. This avoids misleading discrepancies associated with spherical-coordinate singularities.

For each mass, we define the Cartesian-position difference as

$$
|\Delta\mathbf{r}_i(t)| = \sqrt{(x_i^{(L)}-x_i^{(H)})^2+(y_i^{(L)}-y_i^{(H)})^2+(z_i^{(L)}-z_i^{(H)})^2},
\qquad i = 1, 2.
$$

where the superscripts $L$ and $H$ denote the Lagrangian and Hamiltonian formulations, respectively.

This quantity measures the physical separation between the cooresponding bobs predicted by the two formulations. 

We numerically integrate both ystems over the same time interval, using identical physical parameters and equivalent ICs, and plot $|\Delta\mathbf{r}_1(t)|$ and $|\Delta\mathbf{r}_2(t)|$ separately. 

If the two implementations are consistent, both positional discrepancies should remain small within numerical accuracy. Because the DSP exhibits nonlinear and potentially chaotic dynamics, small numerical errors may nevertheless amplify over time.

In [ ]:
# Compare Lagrangian and Hamiltonian trajectories

# Common time grid
t_pts = np.arange(0., 10., 0.01)

# Physical parameters
L_p1 = Pendulum(m1=1., m2=1., R1=1., R2=1., g=9.8)
H_p1 = HamiltonianPendulum(m1=1., m2=1., R1=1., R2=1., g=9.8)

# Initial conditions
theta1_0, theta2_0 = np.pi/3, np.pi/3
phi1_0, phi2_0 = 0., -np.pi
u1_0, u2_0 = 3., 4.,
v1_0, v2_0 = -3., -4.

# convert initial velocities to canonical momenta
ic_args = (
    theta1_0, theta2_0, phi1_0, phi2_0,
    u1_0, u2_0, v1_0, v2_0,
    1., 1., 1., 1.
)

p_theta1_0 = p_theta1_num(*ic_args)
p_theta2_0 = p_theta2_num(*ic_args)
p_phi1_0 = p_phi1_num(*ic_args)
p_phi2_0 = p_phi2_num(*ic_args)

# Solve Lagrangian equations
theta1_L, u1_L, theta2_L, u2_L, phi1_L, v1_L, phi2_L, v2_L = L_p1.solve_ode(t_pts, theta1_0, u1_0, theta2_0, u2_0,
                                                                              phi1_0, v1_0, phi2_0, v2_0)

# Solve Hamiltonian equations
q1_H, q2_H, q3_H, q4_H, p1_H, p2_H, p3_H, p4_H = H_p1.solve_ode(t_pts, theta1_0, theta2_0, phi1_0, phi2_0, 
                                                                p_theta1_0, p_theta2_0, p_phi1_0, p_phi2_0)

# Cartesian positions: Lagrangian
r1_L = np.array([
    x1_num(theta1_L, theta2_L, phi1_L, phi2_L, R1_param, R2_param),
    y1_num(theta1_L, theta2_L, phi1_L, phi2_L, R1_param, R2_param),
    z1_num(theta1_L, theta2_L, phi1_L, phi2_L, R1_param, R2_param)
])
r2_L = np.array([
    x2_num(theta1_L, theta2_L, phi1_L, phi2_L, R1_param, R2_param),
    y2_num(theta1_L, theta2_L, phi1_L, phi2_L, R1_param, R2_param),
    z2_num(theta1_L, theta2_L, phi1_L, phi2_L, R1_param, R2_param)
])

# Cartesian positions: Hamiltonian
r1_H = np.array([
    x1_num(q1_H, q2_H, q3_H, q4_H, R1_param, R2_param),
    y1_num(q1_H, q2_H, q3_H, q4_H, R1_param, R2_param),
    z1_num(q1_H, q2_H, q3_H, q4_H, R1_param, R2_param)
])
r2_H = np.array([
    x2_num(q1_H, q2_H, q3_H, q4_H, R1_param, R2_param),
    y2_num(q1_H, q2_H, q3_H, q4_H, R1_param, R2_param),
    z2_num(q1_H, q2_H, q3_H, q4_H, R1_param, R2_param)
])

# Calculate Cartesian-position differences
delta_r1 = np.linalg.norm(r1_L - r1_H, axis=0)
delta_r2 = np.linalg.norm(r2_L - r2_H, axis=0)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(t_pts, delta_r1, '--', label=r'$|\Delta\vec{r}_1(t)|$')
ax.plot(t_pts, delta_r2, '--', label=r'$|\Delta\vec{r}_2(t)|$')

ax.set_xlabel('Time (s)')
ax.set_ylabel('Position difference (m)')
ax.set_title('Lagrangian vs. Hamiltonian: Cartesian-Position Differences')
ax.legend()
ax.grid(True)

plt.show()

### Results and Discussion

The Cartesian-position comparison provides a numerical assessment of the equivalence between the Lagrangian and Hamiltonian formulations of the DSP.

Both formulations begin from identical physical initial conditions, with the Hamiltonian initial momenta obtained through the momentum-velocity relationship. The resulting trajectories are compared by calculating the Euclidean distance between the corresponding Cartesian positions of each pendulum bob.

The positional discrepancies, $|\Delta\mathbf{r}_1(t)|$ and $|\Delta\mathbf{r}_2(t)|$, remain small during the initial stages of the simulation. As time progresses, the discrepancies may grow and oscillate, reflecting the sensitivity of the DSP's nonlinear dynamics to numerical perturbations.

Importantly, comparing Cartesian positions avoids artificial discrepancies associated with the spherical-coordinate representation, particularly when either pendulum approaches a polar-coordinate singularity.

Although the two formulations are analytically equivalent, their independently integrated numerical solutions are not expected to remain identical indefinitely. Differences in numerical integration error can accumulate and become amplified by the system's dynamics.

The observed initial agreement is consistent with the equivalence of the two formulations for the tested initial conditions. A more rigorous numerical verification can be obtained by repeating the comparison with progressively tighter integration tolerances and examining whether both positional discrepancies decrease.

Together with the analytical Legendre transformation, this numerical comparison supports the consistency of the Lagrangian and Hamiltonian descriptions of the DSP.

Having established this connection, we now turn to the geometric interpretation of Hamiltonian mechanics, beginning with the DSP's eight-dimensional canonical phase space and the phase-space volume preservation described by Liouville's theorem.

## 5.2 Comments on the benefit/physics intuition from the methods

Having established the numerical equivalence of the Lagrangian and Hamiltonian formulations, we now explore the deeper physical interpretation of Hamiltonian dynamics.

One of the principal advantages of Hamiltonian mechanics is its geometric description of dynamical systems in **phase space**. Rather than treating the equations of motion solely as differential equations to be solved, we can interpret them as defining a continuous flow through the space of all possible states.

This perspective reveals fundamental properties of conservative dynamical systems, particularly the preservation of phase-space volume described by **Liouville's theorem**.

We will investigate how this conservation law relates to the chaotic behavior of the double spherical pendulum and conclude by comparing the physical insights offered by the Lagrangian and Hamiltonian formulations.

### Phase Space

In Hamiltonian mechanics, the state of a system is specified by its generalized coordinates $\mathbf q$ and their conjugate momenta $\mathbf p$.

For the double spherical pendulum, we have four generalized coordinates:

$$
\mathbf q=(\theta_1,\theta_2,\phi_1,\phi_2),
$$

and four corresponding canonical momenta:

$$
\mathbf p=(p_{\theta_1},p_{\theta_2},p_{\phi_1},p_{\phi_2}).
$$

Together, these define the eight-dimensional canonical phase space:

$$
\boxed{
\mathbf{z}=(\theta_1,\theta_2,\phi_1,\phi_2,
p_{\theta_1},p_{\theta_2},p_{\phi_1},p_{\phi_2}).
}
$$

Each point in this space represents a complete instantaneous state of the DSP. As the system evolves, its state traces a trajectory through phase space according to Hamilton's canonical equations:

$$
\dot q_i=\frac{\partial\mathcal H}{\partial p_i},
\qquad
\dot p_i=-\frac{\partial\mathcal H}{\partial q_i}.
$$

The Hamiltonian therefore defines a vector field in phase space, specifying the direction of evolution at each state.

For the conservative DSP, the Hamiltonian is independent of time and is conserved along exact trajectories:

$$
\frac{d\mathcal H}{dt}
=
\sum_{i=1}^{4}
\left(
\frac{\partial\mathcal H}{\partial q_i}\dot q_i+
\frac{\partial\mathcal H}{\partial p_i}\dot p_i
\right)
=0.
$$

Consequently, trajectories with total energy $E$ remain confined to the energy hypersurface

$$
\mathcal H(\mathbf q,\mathbf p)=E.
$$

Although the full canonical phase space is eight-dimensional, a regular constant-energy hypersurface is seven-dimensional.

This geometric interpretation provides the foundation for understanding Liouville's theorem and its implications for chaotic Hamiltonian dynamics.

### Liouville's Theorem

Liouville's theorem states that the volume occupied by an ensemble of states in phase space remains constant under Hamiltonian evolution. Although a region may stretch, contract along individual directions, or change shape, its total phase-space volume is preserved. A formal proof can be found in John R. Taylor's *Classical Mechanics*.

For the DSP, the phase space is eight-dimensional, consisting of four generalized coordinates and their conjugate momenta. Liouville's theorem therefore applies to the evolution of volumes in this full eight-dimensional space. Importantly, the theorem does not imply that nearby trajectories remain close together: chaotic sensitivity to ICs can coexist with phase-space volume preservation.

To visualize this principle, we return to the small-oscillation analysis about the aligned double-conical relative equilibrium developed in Section 2.2. Each oscillatory normal mode of the linearized system satisfies

$$
\ddot{\xi}_i+\omega_i^2\xi_i=0.
$$

Introducing an effective modal mass $\mu$ and conjugate momentum $p_{\xi_i}=\mu\dot{\xi}_i$, we represent a single mode using the reduced Hamiltonian

$$
\mathcal H_i(\xi_i,p_{\xi_i})
=\frac{p_{\xi_i}^2}{2\mu}
+\frac12\mu\omega_i^2\xi_i^2.
$$

For the numerical visualization, we choose $\mu=1$ as a normalization of the reduced oscillator model. Hamilton's equations then become

$$
\dot{\xi}_i=\frac{p_{\xi_i}}{\mu},
\qquad
\dot p_{\xi_i}=-\mu\omega_i^2\xi_i.
$$

We generate an ensemble of initial conditions in the $(\xi_i,p_{\xi_i})$ phase plane and evolve each point independently. The resulting snapshots illustrate how an initially compact region changes shape while preserving its area.

This two-dimensional demonstration applies to an oscillatory normal mode of the **linearized** DSP. It illustrates Liouville's theorem but does not constitute a numerical verification of volume preservation in the full nonlinear eight-dimensional system.

In [ ]:
import numpy as np
from numpy.random import uniform, rand

from scipy.integrate import solve_ivp
from scipy.spatial import ConvexHull

import matplotlib.pyplot as plt

In [ ]:
# The dpi (dots-per-inch) setting will affect the resolution and how large
# the plots appear on screen and printed. So you may want/need to adjust
# the figsize when creating the figure.
plt.rcParams['figure.dpi'] = 100. # this is the default for notebook

# Change the common font size (smaller when higher dpi)
font_size = 10
plt.rcParams.update({'font.size': font_size})

In [ ]:
class NormalModeHamiltonianPendulum:

    def __init__(self, omega, mu=1.0):
        self.omega = omega
        self.mu = mu

    def dy_dt(self, t, y):
        xi, p_xi = y
        return [
            p_xi / self.mu,
            -self.mu * self.omega**2 * xi
        ]

    def solve_ode(self, t_pts, xi_0, p_xi_0, 
                  abserr=1e-10, relerr=1e-10):
        y = [xi_0, p_xi_0]
        solution = solve_ivp(self.dy_dt, (t_pts[0], t_pts[-1]), 
                        y, t_eval=t_pts,
                        atol=abserr, rtol=relerr)
        xi, p_xi = solution.y

        return xi, p_xi

In [ ]:
# Labels for individual plot axes
xi_vs_time_labels = (r'$t$', r'$\xi(t)$')
p_xi_vs_time_labels = (r'$t$', r'$p_\xi(t)$')
phase_space_labels = (r'$\xi$', r'$p_\xi$')

# R1, R2 = 1., 1.,
# g = 9.8
# m1, m2 = 1., 1.

# Instantiate a pendulum 
p11 = NormalModeHamiltonianPendulum(omega=omega1_aligned_pert)

In [ ]:
# Generate initial conditions
num_samples = 300
xi_and_p_low = [1.5, 0.0]
xi_and_p_high = [2.0, 0.5]
out = uniform(low=xi_and_p_low, high=xi_and_p_high, size=(num_samples,2))

fig = plt.figure()
ax = fig.add_subplot(1,1,1)
ax.plot(out[:, 0], out[:, 1], '.')
ax.set_aspect(1)
ax.set_xlabel(r'$\xi$')
ax.set_ylabel(r'$p_\xi$')
ax.set_xlim(-3., 3.)
ax.set_ylim(-2., 2.)

fig.tight_layout()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(1,1,1)

for xi, p_xi in out:
    ax.plot(xi, p_xi, '.')

ax.set_xlabel(r'$\xi$')
ax.set_ylabel(r'$p_\xi$')
ax.set_xlim(-3., 3.)
ax.set_ylim(-2., 2.)
ax.set_aspect('auto')

fig.tight_layout()

In [ ]:
# Common plotting time (generate the full time then use slices)
delta_t = 0.5
t_start = 0.
num_t = 15
t_end = t_start + (num_t - 1) * delta_t

t_pts = np.arange(t_start, t_end+delta_t, delta_t)

# start the plot! 
fig = plt.figure(figsize=(15,5.*num_t/3.))
overall_title = 'Liouville theorem visualization with linearized pendulum'

plt_num = 0
all_orbits = []
for xi_0, p_xi_0 in out:
    xi, p_xi = p11.solve_ode(t_pts, xi_0, p_xi_0)
    all_orbits.append([xi, p_xi])

for i in np.arange(num_t):
    ax_c = fig.add_subplot(int(num_t/3), 3, i+1)
    ax_c.set_title(rf'$t = {t_pts[i]:.1f}$')
    for j in np.arange(num_samples):
        xi_now, p_xi_now = all_orbits[j]
        ax_c.plot(xi_now[i], p_xi_now[i], '.')
        ax_c.set_xlim(-9., 9.)
        ax_c.set_ylim(-6., 6.)

fig.tight_layout()
#fig.suptitle(overall_title, va='top')

fig.savefig('Liouville_test.png', bbox_inches='tight')

### Interpretation of the Phase-Space Visualization

The evolving ensemble illustrates the geometric character of Hamiltonian dynamics. Although the initial distribution changes orientation and becomes stretched or compressed along different directions, the phase-space flow preserves its enclosed area.

For the reduced harmonic-oscillator model, the evolution is linear. Consequently, an initial rectangular region transforms into a parallelogram rather than developing the curved distortions characteristic of nonlinear dynamics.

This distinction is important when interpreting the visualization. The reduced normal-mode system captures the local oscillatory behavior near a conical relative equilibrium, whereas the full nonlinear double spherical pendulum can exhibit considerably more complicated motion.

Nevertheless, both systems are governed by Hamiltonian mechanics, and Liouville's theorem expresses a fundamental geometric constraint on their evolution.

### Lagrangian and Hamiltonian Mechanics: A Final Comparison

Throughout this investigation, the DSP has been described using both Lagrangian and Hamiltonian mechanics. Although these formulations arise from different mathematical perspectives, they provide equivalent descriptions of the system's dynamics.

The Lagrangian formulation begins with the generalized coordinates $\mathbf q$ and velocities $\dot{\mathbf q}$, describing the system through the Lagrangian $\mathcal L=T-V$. The EOMs follow from the E–L equations,

$$
\frac{d}{dt}\left(\frac{\partial\mathcal L}{\partial\dot q_i}\right)
-\frac{\partial\mathcal L}{\partial q_i}=0.
$$

This approach is particularly convenient for systems involving geometric constraints and generalized coordinates, as demonstrated by our initial derivation of the DSP EOMs.

The Hamiltonian formulation instead describes the dynamics using generalized coordinates $\mathbf q$ and their conjugate momenta $\mathbf p$. Through the Legendre transformation,

$$
\mathcal H=\sum_i p_i\dot q_i-\mathcal L,
$$

the EOMs can be expressed as Hamilton's canonical equations,

$$
\dot q_i=\frac{\partial\mathcal H}{\partial p_i},
\qquad
\dot p_i=-\frac{\partial\mathcal H}{\partial q_i}.
$$

While this formulation requires additional work to obtain the conjugate momenta and Hamiltonian, it reveals the canonical phase-space structure of the system. This structure provides a natural framework for studying phase-space trajectories, symplectic geometry, and Liouville's theorem.

The principal distinctions between the two formulations are summarized below.

| Feature | Lagrangian Mechanics | Hamiltonian Mechanics |
|---|---|---|
| Fundamental function | $\mathcal L(\mathbf q,\dot{\mathbf q},t)$ | $\mathcal H(\mathbf q,\mathbf p,t)$ |
| State variables | Generalized coordinates and velocities | Generalized coordinates and conjugate momenta |
| Equations of motion | Second-order E–L equations | First-order canonical equations |
| Primary emphasis | Variational principles, constraints, and generalized coordinates | Canonical phase space and symplectic structure |
| Advantages for the DSP | Convenient derivation of the EOMs | Natural description of phase-space geometry and Liouville's theorem |

For the DSP, the numerical comparisons in Section 5.1 illustrate the agreement between these two descriptions when equivalent initial conditions are used and numerical errors are sufficiently controlled.

The Liouville visualization provides a final example of the insight gained from the Hamiltonian perspective. Rather than examining only individual trajectories, we consider how an entire ensemble of ICs evolves in phase space. Even as the distribution changes shape, its area remains conserved in the reduced canonical normal-mode system.

Ultimately, the Lagrangian and Hamiltonian formulations should not be regarded as competing approaches, but as complementary mathematical frameworks. The Lagrangian formulation provides a convenient route from the physical construction of the system to its EOMs, while the Hamiltonian formulation exposes the geometric structure underlying those equations.

Together, they demonstrate how the same mechanical system can be understood from different, yet fundamentally connected, perspectives.

## Limitations and Future Directions

Although the present investigation explores several aspects of the DSP, a number of limitations and opportunities for further study remain.

**Numerical integration.** The numerical solutions depend on finite integration tolerances and the properties of the chosen algorithms. Agreement between formulations provides evidence of numerical consistency but does not eliminate accumulated integration error. Structure-preserving methods, particularly symplectic integrators, offer a natural direction for further investigation of long-term Hamiltonian dynamics.

**Chaotic dynamics.** Sensitivity to initial conditions and Poincaré sections provide insight into the system's nonlinear behavior. A more quantitative investigation could estimate Lyapunov exponents and examine how chaotic behavior varies with energy and initial conditions.

**Machine-learning models.** The constrained Lagrangian neural network experiments demonstrate the potential of incorporating physical constraints into learned dynamics. Their conclusions remain dependent on the selected training distributions, model configurations, and evaluation horizons. Additional experiments could investigate generalization across different physical parameters and longer prediction intervals.

**Phase-space volume preservation.** The numerical illustration of Liouville's theorem uses a two-dimensional normal-mode Hamiltonian obtained from the linearized dynamics near an aligned conical relative equilibrium. A more direct numerical investigation of the full nonlinear DSP could compute the Jacobian of its eight-dimensional Hamiltonian flow and examine phase-space volume preservation.

**Effective Modal Mass and Canonical Reduction.** The Liouville visualization uses a reduced harmonic-oscillator model associated with an oscillatory normal mode of the aligned conical solution, with the modal mass normalized to $\mu=1$. A more rigorous treatment would derive the effective modal mass and canonical momentum directly from the DSP's quadratic Lagrangian about the rotating equilibrium, accounting for gyroscopic coupling and eigenvector normalization. This would establish a more explicit connection between the linearized DSP dynamics and the reduced Hamiltonian phase space.

These extensions are not necessary to establish the results presented here. Rather, they illustrate how the analytical and computational methods developed throughout this project can support increasingly detailed investigations of nonlinear dynamical systems.

# Conclusion 

The double spherical pendulum provides a compelling example of how a mechanical system built from simple components can exhibit remarkably complex dynamics. By extending the familiar spherical pendulum to two coupled bodies, we encountered a system whose behavior encompasses periodic motion, relative equilibria, small oscillations, nonlinear sensitivity, and chaos.

Throughout this investigation, the Lagrangian and Hamiltonian formulations provided complementary descriptions of the same underlying physics. Analytical methods revealed the system's symmetries, conserved quantities, and special solutions, while numerical methods made it possible to investigate trajectories and behaviors beyond the reach of simple closed-form solutions. Comparisons of numerical integration techniques and experiments with constrained Lagrangian neural networks further illustrated how computational methods can both illuminate and approximate physical dynamics.

The final investigation of Liouville's theorem offers a different perspective: even when individual trajectories exhibit complicated behavior, Hamiltonian evolution obeys a precise geometric structure in phase space.

The broader significance of this project extends beyond the DSP. Many of the methods explored here—deriving equations of motion, identifying symmetries, analyzing equilibria, studying perturbations, and examining phase-space structure—can be applied to other dynamical systems.

A mechanical model need not be complicated in its construction to be rich in its behavior. Sometimes, adding just one degree of freedom, one coupling, or one constraint is enough to reveal an entirely new layer of physics.

The DSP is one such example, but it is far from the only one.

## References

1. Taylor, J. R. (2005). *Classical Mechanics* (1st ed.). University Science Books. Chapter 13, §13.7, “Liouville's Theorem.”

2. Cranmer, M., Greydanus, S., Hoyer, S., Battaglia, P., Spergel, D., & Ho, S. (2020). *Lagrangian Neural Networks*. ICLR 2020 Workshops: DeepDiffEq. https://arxiv.org/abs/2003.04630

3. Finzi, M., Wang, K. A., & Wilson, A. G. (2020). *Simplifying Hamiltonian and Lagrangian Neural Networks via Explicit Constraints*. Advances in Neural Information Processing Systems (NeurIPS), 33.

4. Furnstahl, R. J. (2019). *Liouville Theorem Visualization*. Computational physics teaching notebook, revised March 16, 2019. Adapted for the reduced normal-mode Hamiltonian of the double spherical pendulum.